In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — STEP E1
# LOCATE DATASET
# ================================================================

from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model"

print("=" * 70)
print("ENVIRONMENTAL MODEL — STEP E1")
print("DATASET LOCATION")
print("=" * 70)

print("\nEnvironmental Model directory:")
print(BASE_DIR)

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(
        f"\nEnvironmental_Model folder not found:\n{BASE_DIR}"
    )

print("\nFolders found:")
for item in sorted(os.listdir(BASE_DIR)):
    print(" -", item)

RAW_DIR = os.path.join(BASE_DIR, "01_raw_data")

print("\nRaw data directory:")
print(RAW_DIR)

if not os.path.exists(RAW_DIR):
    raise FileNotFoundError(
        f"\n01_raw_data folder not found:\n{RAW_DIR}"
    )

print("\nFiles inside 01_raw_data:")

files = []

for root, dirs, filenames in os.walk(RAW_DIR):
    for filename in filenames:
        full_path = os.path.join(root, filename)
        files.append(full_path)

if len(files) == 0:
    raise FileNotFoundError(
        "\nNo files found inside 01_raw_data."
    )

for i, file_path in enumerate(files, 1):
    print(f"{i}. {file_path}")

print("\n" + "=" * 70)
print("DATASET LOCATION CHECK COMPLETED")
print("=" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ENVIRONMENTAL MODEL — STEP E1
DATASET LOCATION

Environmental Model directory:
/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model

Folders found:
 - 01_raw_data

Raw data directory:
/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model/01_raw_data

Files inside 01_raw_data:
1. /content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model/01_raw_data/environmental_dataset.csv

DATASET LOCATION CHECK COMPLETED


In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — E1.1
# FULL DATASET INSPECTION
# ================================================================

import pandas as pd
import numpy as np
import os

csv_files = [
    f for f in files
    if f.lower().endswith(".csv")
]

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV dataset found.")

if len(csv_files) > 1:
    print("Multiple CSV files found:")
    for i, f in enumerate(csv_files, 1):
        print(i, f)
    raise ValueError(
        "\nMultiple CSV files found. Send me the output so we can select the correct dataset."
    )

INPUT_FILE = csv_files[0]

print("=" * 70)
print("ENVIRONMENTAL MODEL — DATASET INSPECTION")
print("=" * 70)

print("\nInput file:")
print(INPUT_FILE)

df = pd.read_csv(INPUT_FILE)

print("\nDataset loaded successfully.")

# ------------------------------------------------
# BASIC STRUCTURE
# ------------------------------------------------

print("\n" + "=" * 70)
print("DATASET STRUCTURE")
print("=" * 70)

print("\nShape:")
print(df.shape)

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:03d}. {col}")

# ------------------------------------------------
# DATA TYPES
# ------------------------------------------------

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(df.dtypes)

# ------------------------------------------------
# FIRST ROWS
# ------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 10 ROWS")
print("=" * 70)

print(df.head(10).to_string())

# ------------------------------------------------
# MISSING VALUES
# ------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing = df.isnull().sum()

missing_table = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percent": (missing / len(df)) * 100
})

missing_table = missing_table[
    missing_table["Missing_Count"] > 0
].sort_values(
    "Missing_Count",
    ascending=False
)

if len(missing_table) == 0:
    print("No missing values detected.")
else:
    print(missing_table.to_string())

# ------------------------------------------------
# DUPLICATES
# ------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

# ------------------------------------------------
# UNIQUE VALUES
# ------------------------------------------------

print("\n" + "=" * 70)
print("UNIQUE VALUE SUMMARY")
print("=" * 70)

for col in df.columns:

    unique_count = df[col].nunique(dropna=False)

    print(
        f"{col:<30} "
        f"Unique values: {unique_count}"
    )

# ------------------------------------------------
# NUMERIC / CATEGORICAL
# ------------------------------------------------

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_cols = df.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\n" + "=" * 70)
print("FEATURE TYPE SUMMARY")
print("=" * 70)

print("\nNumeric columns:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical columns:", len(categorical_cols))
print(categorical_cols)

# ------------------------------------------------
# DETAILED CATEGORICAL VALUES
# ------------------------------------------------

print("\n" + "=" * 70)
print("CATEGORICAL VALUE SUMMARY")
print("=" * 70)

for col in categorical_cols:

    print("\n" + "-" * 60)
    print("Column:", col)
    print("Unique values:", df[col].nunique(dropna=False))

    print(
        df[col]
        .value_counts(dropna=False)
        .head(30)
        .to_string()
    )

# ------------------------------------------------
# NUMERIC SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("NUMERIC FEATURE SUMMARY")
print("=" * 70)

if len(numeric_cols) > 0:
    print(
        df[numeric_cols]
        .describe()
        .T
        .to_string()
    )

# ------------------------------------------------
# POTENTIAL ASD / TARGET COLUMNS
# ------------------------------------------------

print("\n" + "=" * 70)
print("POTENTIAL TARGET / ASD-RELATED COLUMNS")
print("=" * 70)

keywords = [
    "asd",
    "autism",
    "diagnos",
    "target",
    "class",
    "outcome",
    "label",
    "condition",
    "k2q35a"
]

potential_cols = []

for col in df.columns:

    col_lower = str(col).lower()

    if any(keyword in col_lower for keyword in keywords):
        potential_cols.append(col)

if potential_cols:
    print("Potential target-related columns:")
    for col in potential_cols:
        print(" -", col)

        print(
            df[col]
            .value_counts(dropna=False)
            .head(30)
            .to_string()
        )
else:
    print("No obvious ASD/target column found automatically.")

# ------------------------------------------------
# CORRELATION WITH NUMERIC TARGET CANDIDATES
# ------------------------------------------------

print("\n" + "=" * 70)
print("ENVIRONMENTAL DATASET INSPECTION COMPLETED")
print("=" * 70)

print("\nSamples :", len(df))
print("Columns :", len(df.columns))
print("Numeric :", len(numeric_cols))
print("Categorical :", len(categorical_cols))
print("Duplicate rows :", duplicate_count)

print("\nIMPORTANT:")
print("No preprocessing or model training has been performed yet.")
print("Target/leakage decisions will be made after reviewing this output.")

ENVIRONMENTAL MODEL — DATASET INSPECTION

Input file:
/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model/01_raw_data/environmental_dataset.csv

Dataset loaded successfully.

DATASET STRUCTURE

Shape:
(55162, 32)

Columns:
001. fpl_i1
002. higrade
003. family_r
004. house_gen
005. metro_yn
006. mpc_yn
007. currins
008. insgap
009. foodsit
010. k11q61
011. k10q40_r
012. k10q30
013. k10q23
014. k10q22
015. k10q20
016. k10q11
017. k10q12
018. k10q13
019. k10q14
020. ace1
021. ace3
022. ace4
023. ace5
024. ace6
025. ace7
026. ace8
027. ace9
028. ace10
029. hoursleep
030. physactiv
031. screentime
032. k2q35a

DATA TYPES
fpl_i1          int64
higrade         int64
family_r      float64
house_gen     float64
metro_yn      float64
mpc_yn        float64
currins       float64
insgap        float64
foodsit       float64
k11q61        float64
k10q40_r      float64
k10q30        float64
k10q23        float64
k10q22        float64
k10q20        float64
k10q11        float64
k10q12    

In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — STEP E2
# LEAKAGE-SAFE DATA PREPARATION
# ================================================================

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ================================================================
# CONFIGURATION
# ================================================================

BASE_DIR = "/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model"

RAW_DIR = os.path.join(BASE_DIR, "01_raw_data")

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "02_preprocessing"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("ENVIRONMENTAL MODEL — STEP E2")
print("LEAKAGE-SAFE DATA PREPARATION")
print("=" * 70)

# ================================================================
# LOCATE CSV
# ================================================================

csv_files = []

for root, dirs, filenames in os.walk(RAW_DIR):
    for filename in filenames:
        if filename.lower().endswith(".csv"):
            csv_files.append(
                os.path.join(root, filename)
            )

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file found inside 01_raw_data."
    )

if len(csv_files) > 1:
    print("\nMultiple CSV files found:")
    for i, path in enumerate(csv_files, 1):
        print(i, path)

    raise ValueError(
        "Multiple CSV files found. Keep only the intended environmental dataset."
    )

INPUT_FILE = csv_files[0]

print("\nInput file:")
print(INPUT_FILE)

# ================================================================
# LOAD DATA
# ================================================================

df = pd.read_csv(INPUT_FILE)

print("\nOriginal dataset shape:")
print(df.shape)

# ================================================================
# TARGET
# ================================================================

TARGET = "k2q35a"

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' not found."
    )

print("\nTarget column:")
print(TARGET)

print("\nOriginal target distribution:")
print(
    df[TARGET]
    .value_counts(dropna=False)
    .sort_index()
)

# ================================================================
# DUPLICATE REMOVAL
# ================================================================

print("\n" + "=" * 70)
print("DUPLICATE REMOVAL")
print("=" * 70)

duplicate_count = df.duplicated().sum()

print("Duplicate rows detected:", duplicate_count)

df_clean = df.drop_duplicates(
    keep="first"
).reset_index(drop=True)

print(
    "Shape after duplicate removal:",
    df_clean.shape
)

# ================================================================
# TARGET MISSINGNESS
# ================================================================

print("\n" + "=" * 70)
print("TARGET VALIDATION")
print("=" * 70)

missing_target = df_clean[TARGET].isna().sum()

print(
    "Missing target values:",
    missing_target
)

if missing_target > 0:

    print(
        "\nRemoving rows with missing target..."
    )

    df_clean = df_clean[
        df_clean[TARGET].notna()
    ].reset_index(drop=True)

print(
    "Shape after removing missing targets:",
    df_clean.shape
)

# ================================================================
# VALIDATE TARGET VALUES
# ================================================================

unique_targets = sorted(
    df_clean[TARGET].unique().tolist()
)

print("\nTarget values after cleaning:")
print(unique_targets)

allowed_targets = {1, 2}

if not set(unique_targets).issubset(
    allowed_targets
):
    raise ValueError(
        f"Unexpected target values detected: {unique_targets}"
    )

# ================================================================
# TARGET ENCODING
# ================================================================
#
# IMPORTANT:
#
# Original:
# 1 = ASD candidate
# 2 = Non-ASD candidate
#
# Internal model representation:
# 1 = ASD
# 0 = Non-ASD
#
# ================================================================

df_clean["Target"] = (
    df_clean[TARGET]
    .map({
        1: 1,
        2: 0
    })
    .astype(int)
)

# Remove original target
df_clean = df_clean.drop(
    columns=[TARGET]
)

print("\nEncoded target distribution:")
print(
    df_clean["Target"]
    .value_counts()
    .sort_index()
)

# ================================================================
# INPUT FEATURES
# ================================================================

feature_columns = [
    col
    for col in df_clean.columns
    if col != "Target"
]

X = df_clean[
    feature_columns
].copy()

y = df_clean[
    "Target"
].copy()

print("\nNumber of input features:")
print(len(feature_columns))

print("\nInput features:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:02d}. {col}")

# ================================================================
# NUMERIC / CATEGORICAL CHECK
# ================================================================

numeric_features = X.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\nNumeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

# ================================================================
# STRATIFIED TRAIN / TEST SPLIT
# ================================================================

print("\n" + "=" * 70)
print("STRATIFIED TRAIN / TEST SPLIT")
print("=" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Test samples    :", len(X_test))

print("\nTraining distribution:")
print(
    y_train.value_counts()
    .sort_index()
)

print("\nTest distribution:")
print(
    y_test.value_counts()
    .sort_index()
)

# ================================================================
# TRAINING-ONLY PREPROCESSING
# ================================================================

print("\n" + "=" * 70)
print("BUILDING PREPROCESSING PIPELINE")
print("=" * 70)

print(
    "\nIMPORTANT:"
    "\nAll preprocessing statistics are fitted ONLY on training data."
)

# All current features are numeric.
#
# Missing values:
# median imputation
#
# Scaling:
# StandardScaler
#
# Both are fitted only on training data.

preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("\nFitting preprocessing on training data...")

X_train_processed = preprocessor.fit_transform(
    X_train
)

print("Transforming unseen test data...")

X_test_processed = preprocessor.transform(
    X_test
)

# ================================================================
# VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("PREPROCESSING VALIDATION")
print("=" * 70)

print(
    "\nTraining processed shape:",
    X_train_processed.shape
)

print(
    "Test processed shape    :",
    X_test_processed.shape
)

print(
    "\nTraining NaN:",
    np.isnan(X_train_processed).sum()
)

print(
    "Test NaN    :",
    np.isnan(X_test_processed).sum()
)

print(
    "Training Inf:",
    np.isinf(X_train_processed).sum()
)

print(
    "Test Inf    :",
    np.isinf(X_test_processed).sum()
)

# ================================================================
# SAVE PROCESSED DATASETS
# ================================================================

train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_columns
)

train_processed_df["Target"] = y_train.values

test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_columns
)

test_processed_df["Target"] = y_test.values

train_file = os.path.join(
    OUTPUT_DIR,
    "environmental_train_processed.csv"
)

test_file = os.path.join(
    OUTPUT_DIR,
    "environmental_test_processed_unseen.csv"
)

preprocessor_file = os.path.join(
    OUTPUT_DIR,
    "environmental_preprocessor.joblib"
)

feature_names_file = os.path.join(
    OUTPUT_DIR,
    "environmental_processed_feature_names.txt"
)

cleaned_file = os.path.join(
    OUTPUT_DIR,
    "environmental_cleaned_no_leakage.csv"
)

report_file = os.path.join(
    OUTPUT_DIR,
    "environmental_preprocessing_report.txt"
)

train_processed_df.to_csv(
    train_file,
    index=False
)

test_processed_df.to_csv(
    test_file,
    index=False
)

joblib.dump(
    preprocessor,
    preprocessor_file
)

with open(
    feature_names_file,
    "w"
) as f:

    for col in feature_columns:
        f.write(col + "\n")

# Save cleaned unprocessed dataset too
df_clean.to_csv(
    cleaned_file,
    index=False
)

# ================================================================
# CLASS WEIGHTS
# ================================================================

class_counts = y_train.value_counts()

n_total = len(y_train)
n_classes = 2

class_weights = {}

for cls in sorted(class_counts.index):

    class_weights[int(cls)] = (
        n_total /
        (
            n_classes *
            class_counts[cls]
        )
    )

# ================================================================
# REPORT
# ================================================================

report_lines = []

report_lines.append(
    "ENVIRONMENTAL MODEL — STEP E2\n"
)

report_lines.append(
    "=" * 70 + "\n"
)

report_lines.append(
    f"Original samples: {len(df)}\n"
)

report_lines.append(
    f"Duplicates removed: {duplicate_count}\n"
)

report_lines.append(
    f"Missing target rows removed: {missing_target}\n"
)

report_lines.append(
    f"Final samples: {len(df_clean)}\n"
)

report_lines.append(
    f"Target: {TARGET}\n"
)

report_lines.append(
    "Target encoding: 1=ASD, 0=Non-ASD\n"
)

report_lines.append(
    f"Input features: {len(feature_columns)}\n"
)

report_lines.append(
    f"Training samples: {len(X_train)}\n"
)

report_lines.append(
    f"Test samples: {len(X_test)}\n"
)

report_lines.append(
    f"Processed dimensions: {X_train_processed.shape[1]}\n"
)

report_lines.append(
    f"Class weights: {class_weights}\n"
)

with open(
    report_file,
    "w"
) as f:

    f.writelines(
        report_lines
    )

# ================================================================
# FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("STEP E2 — FINAL SUMMARY")
print("=" * 70)

print(
    "\nOriginal samples       :",
    len(df)
)

print(
    "Duplicates removed     :",
    duplicate_count
)

print(
    "Missing target removed :",
    missing_target
)

print(
    "Final samples          :",
    len(df_clean)
)

print(
    "\nTarget:",
    TARGET
)

print(
    "Target encoding:",
    "1 = ASD, 0 = Non-ASD"
)

print(
    "\nTraining samples:",
    len(X_train)
)

print(
    "Unseen test samples:",
    len(X_test)
)

print(
    "\nProcessed dimensions:",
    X_train_processed.shape[1]
)

print(
    "\nClass weights:"
)

for cls, weight in class_weights.items():
    print(
        f"Class {cls}: {weight:.4f}"
    )

print(
    "\nTraining NaN:",
    np.isnan(X_train_processed).sum()
)

print(
    "Test NaN:",
    np.isnan(X_test_processed).sum()
)

print(
    "\nFiles saved:"
)

print(train_file)
print(test_file)
print(preprocessor_file)
print(feature_names_file)
print(cleaned_file)
print(report_file)

print("\n" + "=" * 70)
print("STEP E2 COMPLETED")
print("=" * 70)

ENVIRONMENTAL MODEL — STEP E2
LEAKAGE-SAFE DATA PREPARATION

Input file:
/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model/01_raw_data/environmental_dataset.csv

Original dataset shape:
(55162, 32)

Target column:
k2q35a

Original target distribution:
k2q35a
1.0     2130
2.0    52736
NaN      296
Name: count, dtype: int64

DUPLICATE REMOVAL
Duplicate rows detected: 7098
Shape after duplicate removal: (48064, 32)

TARGET VALIDATION
Missing target values: 296

Removing rows with missing target...
Shape after removing missing targets: (47768, 32)

Target values after cleaning:
[1.0, 2.0]

Encoded target distribution:
Target
0    45650
1     2118
Name: count, dtype: int64

Number of input features:
31

Input features:
01. fpl_i1
02. higrade
03. family_r
04. house_gen
05. metro_yn
06. mpc_yn
07. currins
08. insgap
09. foodsit
10. k11q61
11. k10q40_r
12. k10q30
13. k10q23
14. k10q22
15. k10q20
16. k10q11
17. k10q12
18. k10q13
19. k10q14
20. ace1
21. ace3
22. ace4
23. ace5
24.

In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — STEP E3
# 31-D → 128-D REPRESENTATION LEARNING
#
# Leakage-safe training:
#   TRAIN → model learning
#   VALIDATION → early stopping/model selection
#   TEST → final unseen evaluation only
#
# Final multimodal compatibility:
#   Environmental 128-D → later 32-D alignment
# ================================================================

import os
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------
# CONFIGURATION
# ------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/"
    "ASD_Multimodal_Project/"
    "Environmental_Model"
)

PREPROCESS_DIR = os.path.join(
    BASE_DIR,
    "02_preprocessing"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "03_representation_learning"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(
    PREPROCESS_DIR,
    "environmental_train_processed.csv"
)

TEST_FILE = os.path.join(
    PREPROCESS_DIR,
    "environmental_test_processed_unseen.csv"
)

print("=" * 70)
print("ENVIRONMENTAL MODEL — STEP E3")
print("31-D → 128-D REPRESENTATION LEARNING")
print("=" * 70)

# ------------------------------------------------
# DEVICE
# ------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print()
print("Using device:", device)

# ------------------------------------------------
# RANDOM SEEDS
# ------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------
# LOAD DATA
# ------------------------------------------------

print()
print("Loading training data...")
train_df = pd.read_csv(TRAIN_FILE)

print("Loading unseen test data...")
test_df = pd.read_csv(TEST_FILE)

print("Done.")

# ------------------------------------------------
# IDENTIFY TARGET
# ------------------------------------------------

TARGET = "Target"

if TARGET not in train_df.columns:
    raise ValueError(
        f"Target column '{TARGET}' not found in training data."
    )

if TARGET not in test_df.columns:
    raise ValueError(
        f"Target column '{TARGET}' not found in test data."
    )

# ------------------------------------------------
# IDENTIFY FEATURES
# ------------------------------------------------

feature_columns = [
    c for c in train_df.columns
    if c != TARGET
]

print()
print("=" * 70)
print("INPUT DATASET")
print("=" * 70)

print()
print("Training shape:", train_df.shape)
print("Test shape    :", test_df.shape)

print()
print("Input dimensions:", len(feature_columns))

print()
print("Feature columns:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:02d}. {col}")

# ------------------------------------------------
# VALIDATION
# ------------------------------------------------

if len(feature_columns) != 31:
    raise ValueError(
        f"Expected 31 environmental features, "
        f"found {len(feature_columns)}."
    )

if list(train_df[feature_columns].columns) != \
   list(test_df[feature_columns].columns):
    raise ValueError(
        "Training and test feature columns do not match."
    )

if train_df[feature_columns].isna().sum().sum() != 0:
    raise ValueError("Training data contains NaN values.")

if test_df[feature_columns].isna().sum().sum() != 0:
    raise ValueError("Test data contains NaN values.")

if np.isinf(
    train_df[feature_columns].to_numpy()
).sum() != 0:
    raise ValueError("Training data contains infinite values.")

if np.isinf(
    test_df[feature_columns].to_numpy()
).sum() != 0:
    raise ValueError("Test data contains infinite values.")

print()
print("Input validation: PASSED")

# ------------------------------------------------
# TARGET DISTRIBUTION
# ------------------------------------------------

print()
print("=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

print()
print("Training:")
print(train_df[TARGET].value_counts())

print()
print("Test:")
print(test_df[TARGET].value_counts())

# ------------------------------------------------
# CONVERT TO NUMPY
# ------------------------------------------------

X_all = train_df[feature_columns].values.astype(np.float32)
y_all = train_df[TARGET].values.astype(np.float32)

X_test = test_df[feature_columns].values.astype(np.float32)
y_test = test_df[TARGET].values.astype(np.float32)

# ------------------------------------------------
# TRAIN / VALIDATION SPLIT
#
# IMPORTANT:
# Test set remains untouched until final evaluation.
# ------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X_all,
    y_all,
    test_size=0.15,
    random_state=SEED,
    stratify=y_all
)

print()
print("=" * 70)
print("TRAIN / VALIDATION SPLIT")
print("=" * 70)

print()
print("Training samples  :", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples      :", len(X_test))

print()
print("Training distribution:")
print(pd.Series(y_train).value_counts())

print()
print("Validation distribution:")
print(pd.Series(y_val).value_counts())

print()
print("Test distribution:")
print(pd.Series(y_test).value_counts())

# ------------------------------------------------
# TENSOR CREATION
# ------------------------------------------------

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
)

# ------------------------------------------------
# DATALOADER
# ------------------------------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

# ------------------------------------------------
# CLASS WEIGHTS
# ------------------------------------------------

n_negative = np.sum(y_train == 0)
n_positive = np.sum(y_train == 1)

pos_weight_value = n_negative / n_positive

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device
)

print()
print("=" * 70)
print("CLASS WEIGHTING")
print("=" * 70)

print()
print("Negative samples:", n_negative)
print("Positive samples:", n_positive)
print("Positive class weight:", pos_weight_value)

# ------------------------------------------------
# MODEL
# ------------------------------------------------

class EnvironmentalEncoder(nn.Module):

    def __init__(self, input_dim=31):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                128
            ),

            nn.BatchNorm1d(128),

            nn.ReLU(),

            nn.Dropout(0.15),

            nn.Linear(
                128,
                128
            ),

            nn.BatchNorm1d(128),

            nn.ReLU()
        )

        self.classifier = nn.Linear(
            128,
            1
        )

    def forward(self, x):

        embedding = self.encoder(x)

        logits = self.classifier(
            embedding
        )

        return embedding, logits


model = EnvironmentalEncoder(
    input_dim=len(feature_columns)
).to(device)

print()
print("=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)

print(model)

# ------------------------------------------------
# LOSS
# ------------------------------------------------

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# ------------------------------------------------
# TRAINING
# ------------------------------------------------

EPOCHS = 100
PATIENCE = 12

best_val_auc = -np.inf
best_state = None
patience_counter = 0

print()
print("=" * 70)
print("TRAINING")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):

    model.train()

    running_loss = 0.0
    total_samples = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        embedding, logits = model(xb)

        logits = logits.squeeze(1)

        loss = criterion(
            logits,
            yb
        )

        loss.backward()

        optimizer.step()

        batch_size = xb.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        total_samples += batch_size

    train_loss = (
        running_loss /
        total_samples
    )

    # ------------------------------------------------
    # VALIDATION
    # ------------------------------------------------

    model.eval()

    with torch.no_grad():

        _, val_logits = model(
            X_val_tensor.to(device)
        )

        val_prob = torch.sigmoid(
            val_logits.squeeze(1)
        ).cpu().numpy()

    val_pred = (
        val_prob >= 0.5
    ).astype(int)

    val_auc = roc_auc_score(
        y_val,
        val_prob
    )

    val_balanced_acc = balanced_accuracy_score(
        y_val,
        val_pred
    )

    if epoch == 1 or epoch % 5 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Loss {train_loss:.5f} | "
            f"Val Balanced Acc "
            f"{val_balanced_acc:.4f} | "
            f"Val AUC {val_auc:.4f}"
        )

    # ------------------------------------------------
    # EARLY STOPPING
    # ------------------------------------------------

    if val_auc > best_val_auc:

        best_val_auc = val_auc

        best_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

    else:

        patience_counter += 1

    if patience_counter >= PATIENCE:

        print()
        print(
            f"Early stopping at epoch {epoch}"
        )

        break

# ------------------------------------------------
# RESTORE BEST MODEL
# ------------------------------------------------

if best_state is None:
    raise RuntimeError(
        "No best model state was saved."
    )

model.load_state_dict(best_state)

print()
print(
    f"Best Validation ROC-AUC: "
    f"{best_val_auc:.6f}"
)

# ------------------------------------------------
# FINAL UNSEEN TEST EVALUATION
# ------------------------------------------------

model.eval()

with torch.no_grad():

    _, test_logits = model(
        X_test_tensor.to(device)
    )

    test_prob = torch.sigmoid(
        test_logits.squeeze(1)
    ).cpu().numpy()

test_pred = (
    test_prob >= 0.5
).astype(int)

# ------------------------------------------------
# METRICS
# ------------------------------------------------

test_accuracy = accuracy_score(
    y_test,
    test_pred
)

test_balanced_accuracy = balanced_accuracy_score(
    y_test,
    test_pred
)

test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)

test_auc = roc_auc_score(
    y_test,
    test_prob
)

cm = confusion_matrix(
    y_test,
    test_pred
)

# ------------------------------------------------
# RESULTS
# ------------------------------------------------

print()
print("=" * 70)
print("FINAL ENVIRONMENTAL TEST RESULTS")
print("=" * 70)

print()
print(f"Accuracy          : {test_accuracy:.4f}")
print(
    f"Balanced Accuracy : "
    f"{test_balanced_accuracy:.4f}"
)
print(f"Precision         : {test_precision:.4f}")
print(f"Recall            : {test_recall:.4f}")
print(f"F1-score          : {test_f1:.4f}")
print(f"ROC-AUC           : {test_auc:.4f}")

print()
print("Confusion Matrix:")
print(cm)

# ------------------------------------------------
# GENERATE 128-D EMBEDDINGS
# ------------------------------------------------

print()
print("=" * 70)
print("GENERATING 128-D ENVIRONMENTAL EMBEDDINGS")
print("=" * 70)

# Combine cleaned train + test
X_combined = np.concatenate(
    [
        X_all,
        X_test
    ],
    axis=0
)

y_combined = np.concatenate(
    [
        y_all,
        y_test
    ],
    axis=0
)

X_combined_tensor = torch.tensor(
    X_combined,
    dtype=torch.float32
)

model.eval()

with torch.no_grad():

    embeddings, _ = model(
        X_combined_tensor.to(device)
    )

embeddings = embeddings.cpu().numpy()

print()
print(
    "All embedding shape:",
    embeddings.shape
)

print(
    "Embedding NaN:",
    np.isnan(embeddings).sum()
)

print(
    "Embedding Inf:",
    np.isinf(embeddings).sum()
)

# ------------------------------------------------
# SAVE EMBEDDINGS
# ------------------------------------------------

embedding_columns = [
    f"Environmental_128D_{i+1:03d}"
    for i in range(128)
]

embedding_df = pd.DataFrame(
    embeddings,
    columns=embedding_columns
)

embedding_df.insert(
    0,
    "Sample_Index",
    np.arange(
        len(embedding_df)
    )
)

embedding_df["Target"] = y_combined.astype(int)

embedding_file = os.path.join(
    OUTPUT_DIR,
    "environmental_128d_embeddings.csv"
)

embedding_df.to_csv(
    embedding_file,
    index=False
)

# ------------------------------------------------
# SAVE MODEL
# ------------------------------------------------

model_file = os.path.join(
    OUTPUT_DIR,
    "environmental_encoder_128d.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_dim": 31,
        "embedding_dim": 128,
        "feature_columns": feature_columns,
        "target_column": TARGET,
        "target_definition":
            "K2Q35A: ever told by healthcare provider that child has Autism/ASD",
        "best_validation_auc": best_val_auc,
        "test_accuracy": test_accuracy,
        "test_balanced_accuracy":
            test_balanced_accuracy,
        "test_precision": test_precision,
        "test_recall": test_recall,
        "test_f1": test_f1,
        "test_roc_auc": test_auc
    },
    model_file
)

# ------------------------------------------------
# SAVE TEST PREDICTIONS
# ------------------------------------------------

predictions_df = pd.DataFrame(
    {
        "Sample_Index":
            np.arange(len(test_df)),
        "Actual_Target":
            y_test.astype(int),
        "Predicted_Probability":
            test_prob,
        "Predicted_Target":
            test_pred
    }
)

prediction_file = os.path.join(
    OUTPUT_DIR,
    "environmental_test_predictions.csv"
)

predictions_df.to_csv(
    prediction_file,
    index=False
)

# ------------------------------------------------
# SAVE TRAINING REPORT
# ------------------------------------------------

report_file = os.path.join(
    OUTPUT_DIR,
    "environmental_128d_training_report.txt"
)

with open(
    report_file,
    "w"
) as f:

    f.write(
        "ENVIRONMENTAL MODEL — STEP E3\n"
    )

    f.write(
        "31-D → 128-D REPRESENTATION LEARNING\n\n"
    )

    f.write(
        f"Input dimensions : 31\n"
    )

    f.write(
        f"Embedding dimensions : 128\n"
    )

    f.write(
        f"Training samples : {len(X_train)}\n"
    )

    f.write(
        f"Validation samples : {len(X_val)}\n"
    )

    f.write(
        f"Unseen test samples : {len(X_test)}\n\n"
    )

    f.write(
        "Target:\n"
        "K2Q35A — ever told by a healthcare provider "
        "that the child has Autism/ASD\n\n"
    )

    f.write(
        f"Best validation ROC-AUC : "
        f"{best_val_auc:.6f}\n"
    )

    f.write(
        f"Test Accuracy : "
        f"{test_accuracy:.6f}\n"
    )

    f.write(
        f"Test Balanced Accuracy : "
        f"{test_balanced_accuracy:.6f}\n"
    )

    f.write(
        f"Test Precision : "
        f"{test_precision:.6f}\n"
    )

    f.write(
        f"Test Recall : "
        f"{test_recall:.6f}\n"
    )

    f.write(
        f"Test F1 : "
        f"{test_f1:.6f}\n"
    )

    f.write(
        f"Test ROC-AUC : "
        f"{test_auc:.6f}\n"
    )

    f.write(
        f"\nEmbedding shape : "
        f"{embeddings.shape}\n"
    )

# ------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------

print()
print("=" * 70)
print("STEP E3 — FINAL SUMMARY")
print("=" * 70)

print()
print("Environmental input : 31-D")
print("Environmental embedding : 128-D")

print()
print("Training samples  :", len(X_train))
print("Validation samples:", len(X_val))
print("Unseen test       :", len(X_test))

print()
print(
    f"Accuracy          : {test_accuracy:.4f}"
)

print(
    f"Balanced Accuracy : "
    f"{test_balanced_accuracy:.4f}"
)

print(
    f"Precision         : {test_precision:.4f}"
)

print(
    f"Recall            : {test_recall:.4f}"
)

print(
    f"F1-score          : {test_f1:.4f}"
)

print(
    f"ROC-AUC           : {test_auc:.4f}"
)

print()
print(
    "Embedding shape:",
    embeddings.shape
)

print()
print("Files saved:")
print(embedding_file)
print(model_file)
print(prediction_file)
print(report_file)

print()
print("=" * 70)
print("STEP E3 COMPLETED")
print("=" * 70)

ENVIRONMENTAL MODEL — STEP E3
31-D → 128-D REPRESENTATION LEARNING

Using device: cpu

Loading training data...
Loading unseen test data...
Done.

INPUT DATASET

Training shape: (38214, 32)
Test shape    : (9554, 32)

Input dimensions: 31

Feature columns:
01. fpl_i1
02. higrade
03. family_r
04. house_gen
05. metro_yn
06. mpc_yn
07. currins
08. insgap
09. foodsit
10. k11q61
11. k10q40_r
12. k10q30
13. k10q23
14. k10q22
15. k10q20
16. k10q11
17. k10q12
18. k10q13
19. k10q14
20. ace1
21. ace3
22. ace4
23. ace5
24. ace6
25. ace7
26. ace8
27. ace9
28. ace10
29. hoursleep
30. physactiv
31. screentime

Input validation: PASSED

TARGET DISTRIBUTION

Training:
Target
0    36520
1     1694
Name: count, dtype: int64

Test:
Target
0    9130
1     424
Name: count, dtype: int64

TRAIN / VALIDATION SPLIT

Training samples  : 32481
Validation samples: 5733
Test samples      : 9554

Training distribution:
0.0    31041
1.0     1440
Name: count, dtype: int64

Validation distribution:
0.0    5479
1.0    

In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — STEP E4
# 128-D ENVIRONMENTAL REPRESENTATION VALIDATION
# ================================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("ENVIRONMENTAL MODEL — STEP E4")
print("128-D ENVIRONMENTAL REPRESENTATION VALIDATION")
print("=" * 70)

# ------------------------------------------------
# PATHS
# ------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/"
    "ASD_Multimodal_Project/"
    "Environmental_Model"
)

EMBEDDING_FILE = os.path.join(
    BASE_DIR,
    "03_representation_learning",
    "environmental_128d_embeddings.csv"
)

SOURCE_FILE = os.path.join(
    BASE_DIR,
    "02_preprocessing",
    "environmental_cleaned_no_leakage.csv"
)

REPORT_FILE = os.path.join(
    BASE_DIR,
    "03_representation_learning",
    "environmental_128d_validation_report.txt"
)

# ------------------------------------------------
# CHECK FILES
# ------------------------------------------------

print("\nChecking required files...")

if not os.path.exists(EMBEDDING_FILE):
    raise FileNotFoundError(
        f"\nEnvironmental 128-D embedding file not found:\n"
        f"{EMBEDDING_FILE}"
    )

if not os.path.exists(SOURCE_FILE):
    raise FileNotFoundError(
        f"\nEnvironmental cleaned source file not found:\n"
        f"{SOURCE_FILE}"
    )

print("Embedding file found.")
print("Source file found.")

# ------------------------------------------------
# LOAD DATA
# ------------------------------------------------

print("\nLoading Environmental 128-D embeddings...")

df_emb = pd.read_csv(EMBEDDING_FILE)

print("Done.")

print("\nLoading cleaned Environmental dataset...")

df_source = pd.read_csv(SOURCE_FILE)

print("Done.")

# ------------------------------------------------
# BASIC INFORMATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("DATASET STRUCTURE")
print("=" * 70)

print("\nEmbedding dataset shape:", df_emb.shape)
print("Source dataset shape   :", df_source.shape)

# ------------------------------------------------
# DETECT TARGET
# ------------------------------------------------

TARGET_CANDIDATES = [
    "Target",
    "target",
    "Class/ASD",
    "class",
    "label"
]

target_col = None

for col in TARGET_CANDIDATES:
    if col in df_emb.columns:
        target_col = col
        break

if target_col is None:
    raise ValueError(
        "\nCould not identify target column in embedding dataset.\n"
        f"Available columns:\n{list(df_emb.columns)}"
    )

print("\nTarget column detected:", target_col)

# ------------------------------------------------
# DETECT SAMPLE ID
# ------------------------------------------------

ID_CANDIDATES = [
    "Sample_Index",
    "Sample_ID",
    "sample_id",
    "ID",
    "id",
    "Index"
]

id_col = None

for col in ID_CANDIDATES:
    if col in df_emb.columns:
        id_col = col
        break

if id_col is not None:
    print("Sample ID column detected:", id_col)
else:
    print("Sample ID column: NOT FOUND")
    print(
        "Warning: embedding dataset does not contain a recognized "
        "sample ID column."
    )

# ------------------------------------------------
# IDENTIFY EMBEDDING COLUMNS
# ------------------------------------------------

metadata_columns = []

if id_col is not None:
    metadata_columns.append(id_col)

metadata_columns.append(target_col)

embedding_cols = [
    col for col in df_emb.columns
    if col not in metadata_columns
]

print("\nEmbedding columns detected:", len(embedding_cols))

if len(embedding_cols) != 128:
    raise ValueError(
        f"\nERROR: Expected exactly 128 embedding columns, "
        f"but found {len(embedding_cols)}.\n"
        f"Columns:\n{embedding_cols}"
    )

# ------------------------------------------------
# DATASET SIZE VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE COUNT VALIDATION")
print("=" * 70)

expected_samples = len(df_source)
actual_samples = len(df_emb)

print("\nExpected samples:", expected_samples)
print("Actual samples  :", actual_samples)

sample_count_pass = expected_samples == actual_samples

print(
    "Sample count validation:",
    "PASSED" if sample_count_pass else "FAILED"
)

# ------------------------------------------------
# TARGET VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("TARGET VALIDATION")
print("=" * 70)

target_nan = df_emb[target_col].isna().sum()

print("\nTarget NaN values:", target_nan)

unique_targets = sorted(
    df_emb[target_col].dropna().unique().tolist()
)

print("Target values:", unique_targets)

binary_target_pass = set(unique_targets).issubset({0, 1})

print(
    "Binary target validation:",
    "PASSED" if binary_target_pass else "FAILED"
)

print("\nActual target distribution:")
print(df_emb[target_col].value_counts().sort_index())

# ------------------------------------------------
# COMPARE WITH SOURCE TARGET
# ------------------------------------------------

source_target_candidates = [
    "Target",
    "target",
    "Class/ASD",
    "class",
    "label"
]

source_target_col = None

for col in source_target_candidates:
    if col in df_source.columns:
        source_target_col = col
        break

if source_target_col is not None:

    print("\nSource target column:", source_target_col)

    print("\nSource target distribution:")
    print(
        df_source[source_target_col]
        .value_counts()
        .sort_index()
    )

    embedding_distribution = (
        df_emb[target_col]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    source_distribution = (
        df_source[source_target_col]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    target_distribution_pass = (
        embedding_distribution == source_distribution
    )

else:

    print("\nSource target column could not be identified.")

    target_distribution_pass = True

# ------------------------------------------------
# NUMERIC VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("EMBEDDING VALUE VALIDATION")
print("=" * 70)

non_numeric_cols = []

for col in embedding_cols:
    if not pd.api.types.is_numeric_dtype(df_emb[col]):
        non_numeric_cols.append(col)

print("\nNon-numeric embedding columns:", non_numeric_cols)

numeric_pass = len(non_numeric_cols) == 0

embedding_matrix = df_emb[embedding_cols].to_numpy(dtype=np.float32)

nan_count = np.isnan(embedding_matrix).sum()
inf_count = np.isinf(embedding_matrix).sum()

print("NaN values:", nan_count)
print("Inf values:", inf_count)

numeric_value_pass = (
    numeric_pass
    and nan_count == 0
    and inf_count == 0
)

# ------------------------------------------------
# DUPLICATE SAMPLE ID VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE ID VALIDATION")
print("=" * 70)

if id_col is not None:

    duplicate_ids = df_emb[id_col].duplicated().sum()
    missing_ids = df_emb[id_col].isna().sum()

    print("\nDuplicate sample IDs:", duplicate_ids)
    print("Missing sample IDs  :", missing_ids)

    id_validation_pass = (
        duplicate_ids == 0
        and missing_ids == 0
    )

else:

    duplicate_ids = None
    missing_ids = None

    print("\nSample ID validation skipped.")
    id_validation_pass = True

# ------------------------------------------------
# DUPLICATE EMBEDDING VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE EMBEDDING CHECK")
print("=" * 70)

duplicate_embedding_mask = (
    df_emb[embedding_cols]
    .duplicated(keep=False)
)

duplicate_embedding_rows = (
    duplicate_embedding_mask.sum()
)

duplicate_embedding_groups = (
    df_emb.loc[
        duplicate_embedding_mask,
        embedding_cols
    ]
    .drop_duplicates()
    .shape[0]
)

print("\nRows involved in duplicate embeddings:",
      duplicate_embedding_rows)

print("Duplicate embedding groups:",
      duplicate_embedding_groups)

# ------------------------------------------------
# CROSS-LABEL DUPLICATE CHECK
# ------------------------------------------------

cross_label_duplicate = False

if duplicate_embedding_rows > 0:

    duplicate_groups = (
        df_emb.loc[
            duplicate_embedding_mask,
            embedding_cols + [target_col]
        ]
        .groupby(embedding_cols, dropna=False)[target_col]
        .nunique()
    )

    cross_label_duplicate = (duplicate_groups > 1).any()

print(
    "\nDuplicate embeddings across different labels:",
    cross_label_duplicate
)

cross_label_duplicate_pass = not cross_label_duplicate

# ------------------------------------------------
# CONSTANT DIMENSION CHECK
# ------------------------------------------------

print("\n" + "=" * 70)
print("EMBEDDING DIMENSION CHECK")
print("=" * 70)

dimension_std = df_emb[embedding_cols].std()

constant_dimensions = (
    dimension_std[dimension_std == 0].index.tolist()
)

print(
    "\nConstant embedding dimensions:",
    len(constant_dimensions)
)

if len(constant_dimensions) > 0:
    print("Constant dimensions:")
    print(constant_dimensions)

constant_dimension_pass = (
    len(constant_dimensions) == 0
)

# ------------------------------------------------
# EMBEDDING STATISTICS
# ------------------------------------------------

print("\n" + "=" * 70)
print("EMBEDDING STATISTICS")
print("=" * 70)

mean_abs = np.mean(np.abs(embedding_matrix))
mean_value = np.mean(embedding_matrix)
std_value = np.std(embedding_matrix)
min_value = np.min(embedding_matrix)
max_value = np.max(embedding_matrix)

print("\nMean absolute value:", mean_abs)
print("Mean value         :", mean_value)
print("Standard deviation :", std_value)
print("Minimum value      :", min_value)
print("Maximum value      :", max_value)

# ------------------------------------------------
# CLASS-WISE EMBEDDING STATISTICS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CLASS-WISE EMBEDDING STATISTICS")
print("=" * 70)

for cls in sorted(df_emb[target_col].dropna().unique()):

    class_matrix = df_emb.loc[
        df_emb[target_col] == cls,
        embedding_cols
    ].to_numpy(dtype=np.float32)

    class_mean_abs = np.mean(np.abs(class_matrix))

    print(
        f"\nClass {int(cls)} Samples:",
        len(class_matrix)
    )

    print(
        "Mean absolute embedding value:",
        class_mean_abs
    )

# ------------------------------------------------
# FINAL VALIDATION RESULTS
# ------------------------------------------------

print("\n" + "=" * 70)
print("E4 VALIDATION RESULTS")
print("=" * 70)

results = {
    "Expected sample count":
        sample_count_pass,

    "Expected 128-D representation":
        len(embedding_cols) == 128,

    "Target column present":
        target_col is not None,

    "No target NaN":
        target_nan == 0,

    "Valid binary target distribution":
        binary_target_pass,

    "Target distribution matches source":
        target_distribution_pass,

    "All embedding columns numeric":
        numeric_pass,

    "No NaN in embeddings":
        nan_count == 0,

    "No Inf in embeddings":
        inf_count == 0,

    "No duplicate sample IDs":
        id_validation_pass,

    "No cross-label duplicate embeddings":
        cross_label_duplicate_pass,

    "No constant embedding dimensions":
        constant_dimension_pass
}

all_passed = True

for name, status in results.items():

    print(
        f"{name:<45}: "
        f"{'PASSED' if status else 'FAILED'}"
    )

    if not status:
        all_passed = False

# ------------------------------------------------
# FINAL STATUS
# ------------------------------------------------

print("\n" + "=" * 70)

if all_passed:

    print("VALIDATION STATUS : PASSED")
    print()
    print(
        "Environmental 128-D representation is VALIDATED "
        "and ready for the 128-D → 32-D alignment stage."
    )

else:

    print("VALIDATION STATUS : FAILED")
    print()
    print(
        "Environmental 128-D representation requires "
        "correction/investigation before alignment."
    )

# ------------------------------------------------
# SAVE REPORT
# ------------------------------------------------

report_lines = []

report_lines.append(
    "ENVIRONMENTAL MODEL — STEP E4\n"
    "128-D ENVIRONMENTAL REPRESENTATION VALIDATION\n"
)

report_lines.append(
    f"\nEmbedding shape: {df_emb.shape}"
)

report_lines.append(
    f"\nEmbedding dimensions: {len(embedding_cols)}"
)

report_lines.append(
    f"\nTarget column: {target_col}"
)

report_lines.append(
    f"\nExpected samples: {expected_samples}"
)

report_lines.append(
    f"\nActual samples: {actual_samples}"
)

report_lines.append(
    f"\nTarget NaN: {target_nan}"
)

report_lines.append(
    f"\nTarget values: {unique_targets}"
)

report_lines.append(
    f"\nNon-numeric embedding columns: {non_numeric_cols}"
)

report_lines.append(
    f"\nEmbedding NaN: {nan_count}"
)

report_lines.append(
    f"\nEmbedding Inf: {inf_count}"
)

report_lines.append(
    f"\nDuplicate embedding rows: {duplicate_embedding_rows}"
)

report_lines.append(
    f"\nDuplicate embedding groups: {duplicate_embedding_groups}"
)

report_lines.append(
    f"\nCross-label duplicate embeddings: "
    f"{cross_label_duplicate}"
)

report_lines.append(
    f"\nConstant embedding dimensions: "
    f"{len(constant_dimensions)}"
)

report_lines.append(
    "\n\nVALIDATION RESULTS\n"
)

for name, status in results.items():

    report_lines.append(
        f"{name}: "
        f"{'PASSED' if status else 'FAILED'}"
    )

report_lines.append(
    "\n\nFINAL STATUS: "
    + ("PASSED" if all_passed else "FAILED")
)

with open(REPORT_FILE, "w") as f:
    f.write("\n".join(report_lines))

print("\nReport saved:")
print(REPORT_FILE)

print("\n" + "=" * 70)
print("E4 COMPLETED")
print("=" * 70)

ENVIRONMENTAL MODEL — STEP E4
128-D ENVIRONMENTAL REPRESENTATION VALIDATION

Checking required files...
Embedding file found.
Source file found.

Loading Environmental 128-D embeddings...
Done.

Loading cleaned Environmental dataset...
Done.

DATASET STRUCTURE

Embedding dataset shape: (47768, 130)
Source dataset shape   : (47768, 32)

Target column detected: Target
Sample ID column detected: Sample_Index

Embedding columns detected: 128

SAMPLE COUNT VALIDATION

Expected samples: 47768
Actual samples  : 47768
Sample count validation: PASSED

TARGET VALIDATION

Target NaN values: 0
Target values: [0, 1]
Binary target validation: PASSED

Actual target distribution:
Target
0    45650
1     2118
Name: count, dtype: int64

Source target column: Target

Source target distribution:
Target
0    45650
1     2118
Name: count, dtype: int64

EMBEDDING VALUE VALIDATION

Non-numeric embedding columns: []
NaN values: 0
Inf values: 0

SAMPLE ID VALIDATION

Duplicate sample IDs: 0
Missing sample IDs  

In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — STEP E4.1
# CORRECTED DUPLICATE EMBEDDING INVESTIGATION
# ================================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("ENVIRONMENTAL MODEL — STEP E4.1")
print("CORRECTED DUPLICATE EMBEDDING INVESTIGATION")
print("=" * 70)

# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/ASD_Multimodal_Project/"
    "Environmental_Model"
)

EMBEDDING_FILE = os.path.join(
    BASE_DIR,
    "03_representation_learning",
    "environmental_128d_embeddings.csv"
)

ORIGINAL_FILE = os.path.join(
    BASE_DIR,
    "02_preprocessing",
    "environmental_cleaned_no_leakage.csv"
)

REPORT_FILE = os.path.join(
    BASE_DIR,
    "03_representation_learning",
    "environmental_128d_duplicate_embedding_investigation.txt"
)

# ----------------------------------------------------------------
# LOAD EMBEDDINGS
# ----------------------------------------------------------------

if not os.path.exists(EMBEDDING_FILE):
    raise FileNotFoundError(
        f"Embedding file not found:\n{EMBEDDING_FILE}"
    )

df = pd.read_csv(EMBEDDING_FILE)

print("\nEmbedding dataset loaded:")
print("Shape:", df.shape)

# ----------------------------------------------------------------
# IDENTIFY COLUMNS
# ----------------------------------------------------------------

ID_COL = "Sample_Index"
TARGET_COL = "Target"

if ID_COL not in df.columns:
    raise ValueError(
        f"{ID_COL} not found in embedding dataset."
    )

if TARGET_COL not in df.columns:
    raise ValueError(
        f"{TARGET_COL} not found in embedding dataset."
    )

embedding_cols = [
    c for c in df.columns
    if c not in [ID_COL, TARGET_COL]
]

print("Sample ID:", ID_COL)
print("Target:", TARGET_COL)
print("Embedding dimensions:", len(embedding_cols))

if len(embedding_cols) != 128:
    raise ValueError(
        f"Expected 128 embedding columns, "
        f"found {len(embedding_cols)}."
    )

# ----------------------------------------------------------------
# BASIC VALIDATION
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC VALIDATION")
print("=" * 70)

print("Rows:", len(df))
print("Embedding dimensions:", len(embedding_cols))

print("\nTarget values:")
print(df[TARGET_COL].value_counts(dropna=False))

nan_values = df[embedding_cols].isna().sum().sum()
inf_values = np.isinf(
    df[embedding_cols].to_numpy(dtype=np.float32)
).sum()

print("\nNaN values:", nan_values)
print("Inf values:", inf_values)

# ----------------------------------------------------------------
# SAMPLE ID VALIDATION
# ----------------------------------------------------------------

duplicate_ids = df[ID_COL].duplicated().sum()
missing_ids = df[ID_COL].isna().sum()

print("\nDuplicate Sample IDs:", duplicate_ids)
print("Missing Sample IDs:", missing_ids)

# ----------------------------------------------------------------
# EXACT DUPLICATE EMBEDDINGS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("EXACT DUPLICATE EMBEDDING ANALYSIS")
print("=" * 70)

# Create a grouping key from the 128-D vector.
# Rounding is NOT used here because we want exact duplicates.

embedding_tuples = list(
    map(tuple, df[embedding_cols].to_numpy())
)

embedding_group = (
    pd.Series(embedding_tuples)
    .groupby(
        pd.Series(embedding_tuples)
    )
    .ngroup()
)

df["_embedding_group"] = embedding_group.values

group_sizes = (
    df["_embedding_group"]
    .value_counts()
)

duplicate_groups = group_sizes[
    group_sizes > 1
]

print(
    "Duplicate embedding groups:",
    len(duplicate_groups)
)

rows_in_duplicates = int(
    duplicate_groups.sum()
)

print(
    "Rows involved in duplicate embeddings:",
    rows_in_duplicates
)

# ----------------------------------------------------------------
# LABEL CONSISTENCY
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("LABEL CONSISTENCY ANALYSIS")
print("=" * 70)

group_label_counts = (
    df.groupby("_embedding_group")[TARGET_COL]
    .nunique()
)

same_label_groups = group_label_counts[
    group_label_counts == 1
].index

cross_label_groups = group_label_counts[
    group_label_counts > 1
].index

print(
    "Same-label duplicate groups :",
    len(same_label_groups)
)

print(
    "Cross-label duplicate groups:",
    len(cross_label_groups)
)

same_label_rows = int(
    df[df["_embedding_group"].isin(
        same_label_groups
    )].shape[0]
)

cross_label_rows = int(
    df[df["_embedding_group"].isin(
        cross_label_groups
    )].shape[0]
)

print(
    "Rows in same-label duplicate groups :",
    same_label_rows
)

print(
    "Rows in cross-label duplicate groups:",
    cross_label_rows
)

# ----------------------------------------------------------------
# CROSS-LABEL DUPLICATE DETAILS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CROSS-LABEL DUPLICATE DETAILS")
print("=" * 70)

cross_label_df = df[
    df["_embedding_group"].isin(cross_label_groups)
].copy()

# Rename internal group for readable output
cross_label_df["Duplicate_Group"] = (
    cross_label_df["_embedding_group"] + 1
)

display_cols = [
    ID_COL,
    TARGET_COL,
    "Duplicate_Group"
]

print("\nFirst 50 cross-label duplicate rows:\n")

print(
    cross_label_df[
        display_cols
    ]
    .sort_values(
        ["Duplicate_Group", TARGET_COL, ID_COL]
    )
    .head(50)
    .to_string(index=False)
)

# ----------------------------------------------------------------
# CROSS-LABEL GROUP SUMMARY
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CROSS-LABEL DUPLICATE GROUP SUMMARY")
print("=" * 70)

group_summary = (
    cross_label_df
    .groupby("Duplicate_Group")
    .agg(
        Number_of_Rows=(ID_COL, "size"),
        Number_of_Labels=(TARGET_COL, "nunique"),
        Labels=(TARGET_COL, lambda x: sorted(
            x.dropna().unique().tolist()
        )),
        Sample_Indices=(ID_COL, lambda x: list(x))
    )
    .reset_index()
)

print(
    "\nFirst 30 cross-label duplicate groups:\n"
)

print(
    group_summary
    .head(30)
    .to_string(index=False)
)

# ----------------------------------------------------------------
# CHECK WHETHER DUPLICATES ARE CAUSED BY IDENTICAL INPUT ROWS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKING ORIGINAL INPUT ROWS")
print("=" * 70)

original_available = os.path.exists(ORIGINAL_FILE)

if original_available:

    original = pd.read_csv(ORIGINAL_FILE)

    print(
        "\nOriginal cleaned dataset loaded:"
    )
    print(
        "Shape:",
        original.shape
    )

    print(
        "Columns:",
        list(original.columns)
    )

    # The cleaned file does NOT contain Sample_Index.
    # Therefore, DO NOT attempt to join on Sample_Index.
    #
    # Instead, verify that the row count and Target ordering
    # correspond to the embedding dataset.

    if len(original) != len(df):

        print(
            "\nWARNING:"
        )
        print(
            "Original cleaned rows and embedding rows "
            "do not have the same count."
        )

        print(
            "Original rows:",
            len(original)
        )

        print(
            "Embedding rows:",
            len(df)
        )

        row_alignment_possible = False

    else:

        row_alignment_possible = True

        print(
            "\nRow count match: PASSED"
        )

        # Compare target sequence
        if TARGET_COL in original.columns:

            target_match = np.array_equal(
                original[TARGET_COL].to_numpy(),
                df[TARGET_COL].to_numpy()
            )

            print(
                "Target row-order match:",
                target_match
            )

        else:

            target_match = False

            print(
                "Target row-order match: "
                "CANNOT CHECK"
            )

else:

    row_alignment_possible = False

    print(
        "\nOriginal cleaned file not found:"
    )

    print(ORIGINAL_FILE)

# ----------------------------------------------------------------
# DUPLICATE INPUT ROW CHECK
# ----------------------------------------------------------------

if row_alignment_possible:

    feature_cols_original = [
        c for c in original.columns
        if c != TARGET_COL
    ]

    duplicate_input_rows = (
        original[feature_cols_original]
        .duplicated(keep=False)
    )

    print(
        "\nDuplicate original input rows:",
        int(duplicate_input_rows.sum())
    )

    # Check whether cross-label embedding duplicates
    # correspond to identical original input rows.

    cross_indices = (
        cross_label_df.index
        .to_numpy()
    )

    if len(cross_indices) > 0:

        duplicate_input_for_cross = (
            original
            .iloc[cross_indices]
            [feature_cols_original]
            .duplicated(keep=False)
        )

        print(
            "Cross-label duplicate embeddings "
            "with duplicate original input rows:",
            int(
                duplicate_input_for_cross.sum()
            )
        )

# ----------------------------------------------------------------
# IMPORTANT INTERPRETATION
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)

if len(cross_label_groups) == 0:

    interpretation = (
        "No cross-label duplicate embeddings were found. "
        "The 128-D representation does not require "
        "duplicate-related correction."
    )

elif row_alignment_possible:

    interpretation = (
        f"{len(cross_label_groups)} cross-label duplicate "
        "embedding groups were found. "
        "Because the environmental representation contains "
        "many repeated/low-variation survey patterns, "
        "exact duplicate embeddings can occur. "
        "These must be investigated before alignment."
    )

else:

    interpretation = (
        f"{len(cross_label_groups)} cross-label duplicate "
        "embedding groups were found. "
        "The cleaned input file does not contain Sample_Index, "
        "so the investigation was performed directly on the "
        "embedding representation. "
        "Do NOT remove these rows automatically."
    )

print("\n" + interpretation)

# ----------------------------------------------------------------
# SAVE REPORT
# ----------------------------------------------------------------

report_lines = []

report_lines.append(
    "ENVIRONMENTAL MODEL — STEP E4.1"
)
report_lines.append(
    "CORRECTED DUPLICATE EMBEDDING INVESTIGATION"
)
report_lines.append("=" * 70)

report_lines.append(
    f"Dataset shape: {df.shape}"
)

report_lines.append(
    f"Embedding dimensions: {len(embedding_cols)}"
)

report_lines.append(
    f"Duplicate embedding groups: "
    f"{len(duplicate_groups)}"
)

report_lines.append(
    f"Rows involved in duplicate embeddings: "
    f"{rows_in_duplicates}"
)

report_lines.append(
    f"Same-label duplicate groups: "
    f"{len(same_label_groups)}"
)

report_lines.append(
    f"Cross-label duplicate groups: "
    f"{len(cross_label_groups)}"
)

report_lines.append(
    f"Rows in same-label duplicate groups: "
    f"{same_label_rows}"
)

report_lines.append(
    f"Rows in cross-label duplicate groups: "
    f"{cross_label_rows}"
)

report_lines.append(
    f"Duplicate Sample IDs: {duplicate_ids}"
)

report_lines.append(
    f"Missing Sample IDs: {missing_ids}"
)

report_lines.append(
    f"Embedding NaN values: {nan_values}"
)

report_lines.append(
    f"Embedding Inf values: {inf_values}"
)

report_lines.append(
    f"Original input available: {original_available}"
)

if original_available:

    report_lines.append(
        f"Original/embedding row alignment possible: "
        f"{row_alignment_possible}"
    )

    if row_alignment_possible:
        report_lines.append(
            f"Target row-order match: "
            f"{target_match}"
        )

report_lines.append("")
report_lines.append(
    "INTERPRETATION:"
)
report_lines.append(
    interpretation
)

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(report_lines)
    )

# ----------------------------------------------------------------
# CLEAN TEMPORARY COLUMN
# ----------------------------------------------------------------

df.drop(
    columns=["_embedding_group"],
    inplace=True
)

print("\n" + "=" * 70)
print("E4.1 COMPLETED")
print("=" * 70)

print(
    "\nReport saved:"
)
print(REPORT_FILE)

print("\nIMPORTANT:")
print(
    "No environmental samples were removed."
)

print(
    "No embeddings were modified."
)

print(
    "Do NOT proceed to E5 until the cross-label "
    "duplicate finding has been interpreted."
)

ENVIRONMENTAL MODEL — STEP E4.1
CORRECTED DUPLICATE EMBEDDING INVESTIGATION

Embedding dataset loaded:
Shape: (47768, 130)
Sample ID: Sample_Index
Target: Target
Embedding dimensions: 128

BASIC VALIDATION
Rows: 47768
Embedding dimensions: 128

Target values:
Target
0    45650
1     2118
Name: count, dtype: int64

NaN values: 0
Inf values: 0

Duplicate Sample IDs: 0
Missing Sample IDs: 0

EXACT DUPLICATE EMBEDDING ANALYSIS
Duplicate embedding groups: 1496
Rows involved in duplicate embeddings: 4060

LABEL CONSISTENCY ANALYSIS
Same-label duplicate groups : 45007
Cross-label duplicate groups: 197
Rows in same-label duplicate groups : 47003
Rows in cross-label duplicate groups: 765

CROSS-LABEL DUPLICATE DETAILS

First 50 cross-label duplicate rows:

 Sample_Index  Target  Duplicate_Group
        10652       0             4576
        22993       0             4576
        28418       0             4576
        37624       1             4576
        25496       0             5596
        

In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — STEP E4.2
# CORRECTED 128-D ENVIRONMENTAL REPRESENTATION VALIDATION
# ================================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("ENVIRONMENTAL MODEL — STEP E4.2")
print("CORRECTED 128-D ENVIRONMENTAL REPRESENTATION VALIDATION")
print("=" * 70)

# ------------------------------------------------
# PATHS
# ------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model"

EMBEDDING_FILE = os.path.join(
    BASE_DIR,
    "03_representation_learning",
    "environmental_128d_embeddings.csv"
)

REPORT_FILE = os.path.join(
    BASE_DIR,
    "03_representation_learning",
    "environmental_128d_corrected_validation_report.txt"
)

# ------------------------------------------------
# LOAD EMBEDDINGS
# ------------------------------------------------

print("\nLoading environmental 128-D embeddings...")

if not os.path.exists(EMBEDDING_FILE):
    raise FileNotFoundError(
        f"\nEnvironmental embedding file not found:\n{EMBEDDING_FILE}"
    )

df = pd.read_csv(EMBEDDING_FILE)

print("Done.")
print("Shape:", df.shape)

# ------------------------------------------------
# IDENTIFY COLUMNS
# ------------------------------------------------

ID_COL = "Sample_Index"
TARGET_COL = "Target"

if ID_COL not in df.columns:
    raise ValueError(f"{ID_COL} not found.")

if TARGET_COL not in df.columns:
    raise ValueError(f"{TARGET_COL} not found.")

embedding_cols = [
    c for c in df.columns
    if c not in [ID_COL, TARGET_COL]
]

print("\n" + "=" * 70)
print("DATASET STRUCTURE")
print("=" * 70)

print("Sample ID column:", ID_COL)
print("Target column   :", TARGET_COL)
print("Embedding dimensions:", len(embedding_cols))

# ------------------------------------------------
# BASIC STRUCTURE
# ------------------------------------------------

expected_samples = 47768
expected_dimensions = 128

results = {}

results["Expected sample count"] = len(df) == expected_samples
results["Expected 128-D representation"] = (
    len(embedding_cols) == expected_dimensions
)

print("\nExpected samples :", expected_samples)
print("Actual samples   :", len(df))

print("\nExpected dimensions:", expected_dimensions)
print("Actual dimensions  :", len(embedding_cols))

# ------------------------------------------------
# SAMPLE ID VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE ID VALIDATION")
print("=" * 70)

duplicate_ids = df[ID_COL].duplicated().sum()
missing_ids = df[ID_COL].isna().sum()

results["No duplicate sample IDs"] = duplicate_ids == 0
results["No missing sample IDs"] = missing_ids == 0

print("Duplicate sample IDs:", duplicate_ids)
print("Missing sample IDs  :", missing_ids)

# ------------------------------------------------
# TARGET VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("TARGET VALIDATION")
print("=" * 70)

target_nan = df[TARGET_COL].isna().sum()

unique_targets = sorted(
    pd.to_numeric(df[TARGET_COL], errors="coerce")
    .dropna()
    .unique()
    .tolist()
)

target_counts = df[TARGET_COL].value_counts().sort_index()

print("Target NaN values:", target_nan)
print("\nTarget distribution:")
print(target_counts)

print("\nUnique target values:", unique_targets)

results["No target NaN"] = target_nan == 0
results["Valid binary target"] = set(unique_targets).issubset({0, 1})

# ------------------------------------------------
# EXPECTED SOURCE DISTRIBUTION
# ------------------------------------------------

expected_distribution = {
    0: 45650,
    1: 2118
}

actual_distribution = {
    int(k): int(v)
    for k, v in target_counts.items()
}

distribution_match = actual_distribution == expected_distribution

results["Target distribution matches source"] = distribution_match

print("\nExpected distribution:")
print(expected_distribution)

print("\nActual distribution:")
print(actual_distribution)

print("\nDistribution preserved:", distribution_match)

# ------------------------------------------------
# EMBEDDING NUMERIC VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("EMBEDDING VALUE VALIDATION")
print("=" * 70)

non_numeric_cols = []

for col in embedding_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        non_numeric_cols.append(col)

embedding_matrix = df[embedding_cols].to_numpy(dtype=np.float32)

nan_count = int(np.isnan(embedding_matrix).sum())
inf_count = int(np.isinf(embedding_matrix).sum())

results["All embedding columns numeric"] = len(non_numeric_cols) == 0
results["No NaN in embeddings"] = nan_count == 0
results["No Inf in embeddings"] = inf_count == 0

print("Non-numeric embedding columns:", non_numeric_cols)
print("NaN values:", nan_count)
print("Inf values:", inf_count)

# ------------------------------------------------
# EMBEDDING DIMENSION QUALITY
# ------------------------------------------------

print("\n" + "=" * 70)
print("EMBEDDING DIMENSION CHECK")
print("=" * 70)

constant_dimensions = []

for i, col in enumerate(embedding_cols):
    if df[col].nunique(dropna=False) <= 1:
        constant_dimensions.append(col)

results["No constant embedding dimensions"] = (
    len(constant_dimensions) == 0
)

print("Constant embedding dimensions:", len(constant_dimensions))

if constant_dimensions:
    print(constant_dimensions)

# ------------------------------------------------
# EMBEDDING STATISTICS
# ------------------------------------------------

print("\n" + "=" * 70)
print("EMBEDDING STATISTICS")
print("=" * 70)

print("Mean absolute value:",
      float(np.mean(np.abs(embedding_matrix))))

print("Mean value:",
      float(np.mean(embedding_matrix)))

print("Standard deviation:",
      float(np.std(embedding_matrix)))

print("Minimum value:",
      float(np.min(embedding_matrix)))

print("Maximum value:",
      float(np.max(embedding_matrix)))

# ------------------------------------------------
# CLASS-WISE STATISTICS
# ------------------------------------------------

print("\n" + "=" * 70)
print("CLASS-WISE EMBEDDING STATISTICS")
print("=" * 70)

for cls in [0, 1]:

    class_matrix = df.loc[
        df[TARGET_COL] == cls,
        embedding_cols
    ].to_numpy(dtype=np.float32)

    if len(class_matrix) > 0:

        print(f"\nClass {cls} Samples:", len(class_matrix))

        print(
            "Mean absolute embedding value:",
            float(np.mean(np.abs(class_matrix)))
        )

# ------------------------------------------------
# DUPLICATE EMBEDDING ANALYSIS
# ------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE EMBEDDING ANALYSIS")
print("=" * 70)

# Exact duplicate vectors
duplicate_mask = df.duplicated(
    subset=embedding_cols,
    keep=False
)

duplicate_rows = int(duplicate_mask.sum())

duplicate_groups = (
    df.loc[duplicate_mask, embedding_cols]
    .astype(str)
    .agg("|".join, axis=1)
    .nunique()
)

print("Rows involved in duplicate embeddings:",
      duplicate_rows)

print("Duplicate embedding groups:",
      duplicate_groups)

# ------------------------------------------------
# CROSS-LABEL DUPLICATE CHECK
# ------------------------------------------------

cross_label_groups = 0
same_label_groups = 0

if duplicate_rows > 0:

    temp = df.loc[
        duplicate_mask,
        [TARGET_COL] + embedding_cols
    ].copy()

    temp["_embedding_key"] = (
        temp[embedding_cols]
        .astype(str)
        .agg("|".join, axis=1)
    )

    grouped = temp.groupby("_embedding_key")[TARGET_COL].nunique()

    cross_label_groups = int((grouped > 1).sum())
    same_label_groups = int((grouped == 1).sum())

print("\nSame-label duplicate groups:",
      same_label_groups)

print("Cross-label duplicate groups:",
      cross_label_groups)

# IMPORTANT:
# Cross-label duplicates are reported as a diagnostic.
# They are NOT automatically treated as fatal failure.

results["Duplicate embeddings are documented"] = True

# ------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("E4.2 FINAL VALIDATION RESULTS")
print("=" * 70)

for name, passed in results.items():
    print(
        f"{name:<45}: "
        f"{'PASSED' if passed else 'FAILED'}"
    )

# ------------------------------------------------
# IMPORTANT DECISION
# ------------------------------------------------

critical_checks = [
    "Expected sample count",
    "Expected 128-D representation",
    "No duplicate sample IDs",
    "No missing sample IDs",
    "No target NaN",
    "Valid binary target",
    "Target distribution matches source",
    "All embedding columns numeric",
    "No NaN in embeddings",
    "No Inf in embeddings",
    "No constant embedding dimensions"
]

critical_pass = all(
    results[k] for k in critical_checks
)

print("\n" + "=" * 70)

if critical_pass:
    print("VALIDATION STATUS : PASSED")
    print()
    print(
        "Environmental 128-D representation is structurally "
        "validated and can proceed to the 128-D → 32-D alignment stage."
    )
else:
    print("VALIDATION STATUS : FAILED")
    print()
    print(
        "Environmental 128-D representation requires correction "
        "before alignment."
    )

print("=" * 70)

# ------------------------------------------------
# SAVE REPORT
# ------------------------------------------------

with open(REPORT_FILE, "w", encoding="utf-8") as f:

    f.write(
        "ENVIRONMENTAL MODEL — STEP E4.2\n"
        "CORRECTED 128-D REPRESENTATION VALIDATION\n"
        + "=" * 70
        + "\n\n"
    )

    f.write(f"Dataset shape: {df.shape}\n")
    f.write(f"Embedding dimensions: {len(embedding_cols)}\n\n")

    for name, passed in results.items():

        f.write(
            f"{name}: "
            f"{'PASSED' if passed else 'FAILED'}\n"
        )

    f.write("\n")
    f.write(
        f"Duplicate embedding rows: {duplicate_rows}\n"
    )

    f.write(
        f"Duplicate embedding groups: {duplicate_groups}\n"
    )

    f.write(
        f"Same-label duplicate groups: {same_label_groups}\n"
    )

    f.write(
        f"Cross-label duplicate groups: {cross_label_groups}\n"
    )

    f.write("\n")

    if critical_pass:
        f.write(
            "VALIDATION STATUS: PASSED\n"
        )
        f.write(
            "Environmental 128-D representation is ready "
            "for 128-D to 32-D alignment.\n"
        )
    else:
        f.write(
            "VALIDATION STATUS: FAILED\n"
        )
        f.write(
            "Environmental representation requires correction.\n"
        )

print("\nReport saved:")
print(REPORT_FILE)

print("\n" + "=" * 70)
print("E4.2 COMPLETED")
print("=" * 70)

ENVIRONMENTAL MODEL — STEP E4.2
CORRECTED 128-D ENVIRONMENTAL REPRESENTATION VALIDATION

Loading environmental 128-D embeddings...
Done.
Shape: (47768, 130)

DATASET STRUCTURE
Sample ID column: Sample_Index
Target column   : Target
Embedding dimensions: 128

Expected samples : 47768
Actual samples   : 47768

Expected dimensions: 128
Actual dimensions  : 128

SAMPLE ID VALIDATION
Duplicate sample IDs: 0
Missing sample IDs  : 0

TARGET VALIDATION
Target NaN values: 0

Target distribution:
Target
0    45650
1     2118
Name: count, dtype: int64

Unique target values: [0, 1]

Expected distribution:
{0: 45650, 1: 2118}

Actual distribution:
{0: 45650, 1: 2118}

Distribution preserved: True

EMBEDDING VALUE VALIDATION
Non-numeric embedding columns: []
NaN values: 0
Inf values: 0

EMBEDDING DIMENSION CHECK
Constant embedding dimensions: 0

EMBEDDING STATISTICS
Mean absolute value: 0.3378232419490814
Mean value: 0.3378232419490814
Standard deviation: 0.5210022926330566
Minimum value: 0.0
Maximu

In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — STEP E5
# 128-D → 32-D CROSS-MODAL ALIGNMENT
# ================================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("=" * 70)
print("ENVIRONMENTAL MODEL — STEP E5")
print("128-D → 32-D CROSS-MODAL ALIGNMENT")
print("=" * 70)

# ------------------------------------------------
# 1. REPRODUCIBILITY
# ------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\nUsing device: {DEVICE}")

# ------------------------------------------------
# 2. PATHS
# ------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model"

INPUT_FILE = os.path.join(
    BASE_DIR,
    "03_representation_learning",
    "environmental_128d_embeddings.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "04_cross_modal_alignment"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "environmental_32d_aligned_representation.csv"
)

OUTPUT_MODEL = os.path.join(
    OUTPUT_DIR,
    "environmental_128_to_32_alignment.pt"
)

OUTPUT_REPORT = os.path.join(
    OUTPUT_DIR,
    "environmental_128_to_32_alignment_report.txt"
)

# ------------------------------------------------
# 3. LOAD VALIDATED 128-D REPRESENTATION
# ------------------------------------------------

print("\nLoading validated environmental 128-D embeddings...")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"\nEnvironmental 128-D embedding file not found:\n{INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

print("Done.")

# ------------------------------------------------
# 4. IDENTIFY COLUMNS
# ------------------------------------------------

ID_COL = "Sample_Index"
TARGET_COL = "Target"

if ID_COL not in df.columns:
    raise ValueError(f"{ID_COL} column not found.")

if TARGET_COL not in df.columns:
    raise ValueError(f"{TARGET_COL} column not found.")

embedding_cols = [
    col for col in df.columns
    if col.startswith("Embedding_")
]

# Fallback in case the embedding columns use another naming convention
if len(embedding_cols) != 128:
    excluded = {ID_COL, TARGET_COL}

    numeric_candidates = [
        col for col in df.columns
        if col not in excluded and pd.api.types.is_numeric_dtype(df[col])
    ]

    if len(numeric_candidates) == 128:
        embedding_cols = numeric_candidates

print("\n" + "=" * 70)
print("INPUT DATASET")
print("=" * 70)

print(f"\nSamples              : {len(df)}")
print(f"Embedding dimensions: {len(embedding_cols)}")

if len(embedding_cols) != 128:
    raise ValueError(
        f"Expected exactly 128 embedding columns, "
        f"but found {len(embedding_cols)}."
    )

# ------------------------------------------------
# 5. INPUT VALIDATION
# ------------------------------------------------

if df[ID_COL].duplicated().any():
    raise ValueError("Duplicate Sample_Index values detected.")

if df[ID_COL].isna().any():
    raise ValueError("Missing Sample_Index values detected.")

if df[TARGET_COL].isna().any():
    raise ValueError("Missing Target values detected.")

X = df[embedding_cols].astype(np.float32).values

if np.isnan(X).any():
    raise ValueError("NaN values detected in 128-D embeddings.")

if np.isinf(X).any():
    raise ValueError("Infinite values detected in 128-D embeddings.")

print("\nInput validation: PASSED")

# ------------------------------------------------
# 6. NORMALIZATION
# ------------------------------------------------
#
# Important:
# Target labels are NOT used to train this projection.
#
# We normalize the 128-D representation before learning
# the 32-D representation.
# ------------------------------------------------

print("\n" + "=" * 70)
print("ALIGNMENT DATA")
print("=" * 70)

X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True)

X_std[X_std < 1e-8] = 1.0

X_normalized = (X - X_mean) / X_std

print("\n128-D representation normalization completed.")

print(f"Normalized mean : {X_normalized.mean():.6f}")
print(f"Normalized std  : {X_normalized.std():.6f}")

# ------------------------------------------------
# 7. PYTORCH DATASET
# ------------------------------------------------

X_tensor = torch.tensor(
    X_normalized,
    dtype=torch.float32
)

dataset = TensorDataset(X_tensor)

BATCH_SIZE = 512

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

# ------------------------------------------------
# 8. 128 → 32 PROJECTION MODEL
# ------------------------------------------------

class Environmental128To32(nn.Module):

    def __init__(self):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 32)
        )

    def forward(self, x):
        return self.projection(x)


model = Environmental128To32().to(DEVICE)

print("\n" + "=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)

print(model)

# ------------------------------------------------
# 9. RECONSTRUCTION NETWORK
# ------------------------------------------------
#
# The 32-D representation should preserve information
# from the original 128-D representation.
#
# Therefore, during alignment training we learn:
#
# 128-D → 32-D → 128-D
#
# The 32-D bottleneck becomes the final aligned
# representation.
# ------------------------------------------------

class EnvironmentalAutoencoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 32)
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 128)
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        return z, reconstruction


autoencoder = EnvironmentalAutoencoder().to(DEVICE)

criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    autoencoder.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

# ------------------------------------------------
# 10. TRAINING
# ------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING 128-D → 32-D PROJECTION")
print("=" * 70)

EPOCHS = 300

best_loss = float("inf")
best_state = None

PATIENCE = 40
patience_counter = 0

for epoch in range(1, EPOCHS + 1):

    autoencoder.train()

    running_loss = 0.0
    total_samples = 0

    for (batch_x,) in loader:

        batch_x = batch_x.to(DEVICE)

        optimizer.zero_grad()

        z, reconstruction = autoencoder(batch_x)

        loss = criterion(
            reconstruction,
            batch_x
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            autoencoder.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        batch_size = batch_x.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

    epoch_loss = running_loss / total_samples

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in autoencoder.state_dict().items()
        }

        patience_counter = 0

    else:

        patience_counter += 1

    if epoch == 1 or epoch % 25 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Reconstruction Loss {epoch_loss:.8f}"
        )

    if patience_counter >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}"
        )

        break

# ------------------------------------------------
# 11. RESTORE BEST MODEL
# ------------------------------------------------

if best_state is None:
    raise RuntimeError(
        "Training failed: no valid model state found."
    )

autoencoder.load_state_dict(best_state)
autoencoder.eval()

# ------------------------------------------------
# 12. GENERATE 32-D REPRESENTATIONS
# ------------------------------------------------

print("\n" + "=" * 70)
print("GENERATING ENVIRONMENTAL 32-D REPRESENTATIONS")
print("=" * 70)

with torch.no_grad():

    X_tensor_device = X_tensor.to(DEVICE)

    Z_32, X_reconstructed = autoencoder(
        X_tensor_device
    )

    Z_32 = Z_32.cpu().numpy()

    X_reconstructed = X_reconstructed.cpu().numpy()

# ------------------------------------------------
# 13. RECONSTRUCTION VALIDATION
# ------------------------------------------------

reconstruction_mse = np.mean(
    (X_normalized - X_reconstructed) ** 2
)

print(f"\n128-D input shape : {X.shape}")
print(f"32-D output shape : {Z_32.shape}")

print(f"\nReconstruction MSE: {reconstruction_mse:.8f}")

if np.isnan(Z_32).any():
    raise ValueError("NaN values found in 32-D representation.")

if np.isinf(Z_32).any():
    raise ValueError("Infinite values found in 32-D representation.")

print("Output validation: PASSED")

# ------------------------------------------------
# 14. SAVE 32-D REPRESENTATION
# ------------------------------------------------

embedding_32_cols = [
    f"Environmental_32D_{i+1:02d}"
    for i in range(32)
]

df_32 = pd.DataFrame(
    Z_32,
    columns=embedding_32_cols
)

# Preserve identifiers and target for validation / inference
df_output = pd.concat(
    [
        df[[ID_COL, TARGET_COL]].reset_index(drop=True),
        df_32
    ],
    axis=1
)

df_output.to_csv(
    OUTPUT_CSV,
    index=False
)

# ------------------------------------------------
# 15. SAVE MODEL
# ------------------------------------------------

torch.save(
    {
        "model_state_dict": autoencoder.state_dict(),
        "input_dimension": 128,
        "output_dimension": 32,
        "normalization_mean": X_mean.astype(np.float32),
        "normalization_std": X_std.astype(np.float32),
        "seed": SEED,
        "reconstruction_mse": float(reconstruction_mse)
    },
    OUTPUT_MODEL
)

# ------------------------------------------------
# 16. SAVE REPORT
# ------------------------------------------------

report = f"""
======================================================================
ENVIRONMENTAL MODEL — STEP E5
128-D → 32-D CROSS-MODAL ALIGNMENT
======================================================================

Input representation:
Environmental 128-D

Output representation:
Environmental 32-D

Samples:
{len(df)}

Input dimensions:
128

Output dimensions:
32

Target:
{TARGET_COL}

IMPORTANT:
Target labels were NOT used to train the 128-D → 32-D projection.

Normalization:
Training representation normalized before projection.

Projection architecture:
128 → 64 → 32

Reconstruction architecture:
32 → 64 → 128

Best reconstruction MSE:
{reconstruction_mse:.10f}

Output shape:
{Z_32.shape}

NaN values:
{int(np.isnan(Z_32).sum())}

Infinite values:
{int(np.isinf(Z_32).sum())}

======================================================================
MULTIMODAL COMPATIBILITY
======================================================================

The environmental representation has been reduced from 128-D to 32-D.

This 32-D representation is intended to be compatible with:

Genetic       → 32-D
Behavioral    → 32-D
Environmental → 32-D

The datasets do NOT require the same patients.

The three modality-specific encoders remain independently trained.

The final multimodal system can combine the three 32-D representations
for a new sample during inference.

======================================================================
FILES
======================================================================

32-D representation:
{OUTPUT_CSV}

Alignment model:
{OUTPUT_MODEL}

Report:
{OUTPUT_REPORT}

======================================================================
STEP E5 COMPLETED
======================================================================
"""

with open(
    OUTPUT_REPORT,
    "w",
    encoding="utf-8"
) as f:
    f.write(report)

# ------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP E5 — FINAL SUMMARY")
print("=" * 70)

print(f"\nEnvironmental 128-D input : {X.shape}")
print(f"Environmental 32-D output: {Z_32.shape}")

print(f"NaN                     : {np.isnan(Z_32).sum()}")
print(f"Inf                     : {np.isinf(Z_32).sum()}")

print(f"Reconstruction MSE      : {reconstruction_mse:.8f}")

print("\nFiles saved:")

print(OUTPUT_CSV)
print(OUTPUT_MODEL)
print(OUTPUT_REPORT)

print("\n" + "=" * 70)
print("STEP E5 COMPLETED")
print("=" * 70)

ENVIRONMENTAL MODEL — STEP E5
128-D → 32-D CROSS-MODAL ALIGNMENT

Using device: cpu

Loading validated environmental 128-D embeddings...
Done.

INPUT DATASET

Samples              : 47768
Embedding dimensions: 128

Input validation: PASSED

ALIGNMENT DATA

128-D representation normalization completed.
Normalized mean : 0.000000
Normalized std  : 1.000000

MODEL ARCHITECTURE
Environmental128To32(
  (projection): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
)

TRAINING 128-D → 32-D PROJECTION
Epoch 001 | Reconstruction Loss 0.66728499
Epoch 025 | Reconstruction Loss 0.16859918
Epoch 050 | Reconstruction Loss 0.15547574
Epoch 075 | Reconstruction Loss 0.15007602
Epoch 100 | Reconstruction Loss 0.14707823
Epoch 125 | Reconstruction Loss 0.14499413
Epoch 150 | Reconstruction Loss 0.14334728
Epoch 

In [ ]:
# ================================================================
# ENVIRONMENTAL MODEL — STEP E6
# 32-D ENVIRONMENTAL REPRESENTATION VALIDATION
# ================================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("ENVIRONMENTAL MODEL — STEP E6")
print("32-D ENVIRONMENTAL REPRESENTATION VALIDATION")
print("=" * 70)

# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/ASD_Multimodal_Project/Environmental_Model"

INPUT_FILE = os.path.join(
    BASE_DIR,
    "04_cross_modal_alignment",
    "environmental_32d_aligned_representation.csv"
)

SOURCE_FILE = os.path.join(
    BASE_DIR,
    "02_preprocessing",
    "environmental_cleaned_no_leakage.csv"
)

REPORT_FILE = os.path.join(
    BASE_DIR,
    "04_cross_modal_alignment",
    "environmental_32d_validation_report.txt"
)

# ----------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------

print("\nLoading environmental 32-D representation...")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"\n32-D representation not found:\n{INPUT_FILE}"
    )

if not os.path.exists(SOURCE_FILE):
    raise FileNotFoundError(
        f"\nSource cleaned dataset not found:\n{SOURCE_FILE}"
    )

df = pd.read_csv(INPUT_FILE)
source = pd.read_csv(SOURCE_FILE)

print("Done.")

# ----------------------------------------------------------------
# IDENTIFY COLUMNS
# ----------------------------------------------------------------

ID_COL = "Sample_Index"
TARGET_COL = "Target"

if ID_COL not in df.columns:
    raise ValueError(
        f"{ID_COL} column is missing from the 32-D representation."
    )

if TARGET_COL not in df.columns:
    raise ValueError(
        f"{TARGET_COL} column is missing from the 32-D representation."
    )

embedding_cols = [
    c for c in df.columns
    if c not in [ID_COL, TARGET_COL]
]

print("\n" + "=" * 70)
print("DATASET STRUCTURE")
print("=" * 70)

print(f"Sample ID column      : {ID_COL}")
print(f"Target column         : {TARGET_COL}")
print(f"Embedding dimensions  : {len(embedding_cols)}")

expected_samples = len(source)
actual_samples = len(df)

print(f"\nExpected samples      : {expected_samples}")
print(f"Actual samples        : {actual_samples}")

print(f"\nExpected dimensions   : 32")
print(f"Actual dimensions     : {len(embedding_cols)}")

# ----------------------------------------------------------------
# SAMPLE COUNT VALIDATION
# ----------------------------------------------------------------

sample_count_pass = actual_samples == expected_samples
dimension_pass = len(embedding_cols) == 32

# ----------------------------------------------------------------
# SAMPLE ID VALIDATION
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE ID VALIDATION")
print("=" * 70)

duplicate_ids = df[ID_COL].duplicated().sum()
missing_ids = df[ID_COL].isna().sum()

print(f"Duplicate sample IDs : {duplicate_ids}")
print(f"Missing sample IDs   : {missing_ids}")

id_unique_pass = duplicate_ids == 0
id_missing_pass = missing_ids == 0

# Check whether IDs preserve source order when source has IDs.
if ID_COL in source.columns:

    id_order_match = (
        df[ID_COL].reset_index(drop=True)
        .equals(
            source[ID_COL].reset_index(drop=True)
        )
    )

    print(f"Sample ID order match: {id_order_match}")

else:
    id_order_match = None
    print("Source dataset has no Sample_Index column.")
    print("Order comparison skipped.")

# ----------------------------------------------------------------
# TARGET VALIDATION
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET VALIDATION")
print("=" * 70)

target_nan = df[TARGET_COL].isna().sum()

print(f"Target NaN values: {target_nan}")

print("\nTarget distribution:")
print(df[TARGET_COL].value_counts().sort_index())

unique_targets = sorted(df[TARGET_COL].dropna().unique().tolist())

print(f"\nUnique target values: {unique_targets}")

binary_target_pass = set(unique_targets).issubset({0, 1})

print(f"Binary target valid: {binary_target_pass}")

# Compare target distribution with source
if TARGET_COL in source.columns:

    source_distribution = (
        source[TARGET_COL]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_distribution = (
        df[TARGET_COL]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    target_distribution_match = (
        source_distribution == actual_distribution
    )

    print("\nSource distribution:")
    print(source_distribution)

    print("\n32-D distribution:")
    print(actual_distribution)

    print(
        f"\nTarget distribution preserved: "
        f"{target_distribution_match}"
    )

else:
    target_distribution_match = False
    print("Source target column missing.")

# ----------------------------------------------------------------
# EMBEDDING VALUE VALIDATION
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("EMBEDDING VALUE VALIDATION")
print("=" * 70)

non_numeric = [
    col for col in embedding_cols
    if not pd.api.types.is_numeric_dtype(df[col])
]

embedding_values = df[embedding_cols].to_numpy(dtype=np.float32)

nan_values = np.isnan(embedding_values).sum()
inf_values = np.isinf(embedding_values).sum()

print(f"Non-numeric embedding columns: {non_numeric}")
print(f"NaN values                  : {nan_values}")
print(f"Inf values                  : {inf_values}")

numeric_pass = len(non_numeric) == 0
nan_pass = nan_values == 0
inf_pass = inf_values == 0

# ----------------------------------------------------------------
# EMBEDDING DIMENSION VALIDATION
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("EMBEDDING DIMENSION CHECK")
print("=" * 70)

constant_dimensions = []

for i, col in enumerate(embedding_cols):
    if df[col].nunique(dropna=False) <= 1:
        constant_dimensions.append(col)

print(
    f"Constant embedding dimensions: "
    f"{len(constant_dimensions)}"
)

if constant_dimensions:
    print("Constant dimensions:")
    print(constant_dimensions)

constant_pass = len(constant_dimensions) == 0

# ----------------------------------------------------------------
# EMBEDDING STATISTICS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("32-D EMBEDDING STATISTICS")
print("=" * 70)

mean_abs = np.mean(np.abs(embedding_values))
mean_value = np.mean(embedding_values)
std_value = np.std(embedding_values)
min_value = np.min(embedding_values)
max_value = np.max(embedding_values)

print(f"Mean absolute value : {mean_abs}")
print(f"Mean value          : {mean_value}")
print(f"Standard deviation  : {std_value}")
print(f"Minimum value       : {min_value}")
print(f"Maximum value       : {max_value}")

# ----------------------------------------------------------------
# CLASS-WISE STATISTICS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS-WISE EMBEDDING STATISTICS")
print("=" * 70)

for cls in sorted(df[TARGET_COL].unique()):

    class_embeddings = df.loc[
        df[TARGET_COL] == cls,
        embedding_cols
    ].to_numpy(dtype=np.float32)

    class_mean_abs = np.mean(np.abs(class_embeddings))

    print(
        f"\nClass {cls} Samples: {len(class_embeddings)}"
    )

    print(
        f"Mean absolute embedding value: "
        f"{class_mean_abs}"
    )

# ----------------------------------------------------------------
# DUPLICATE EMBEDDING ANALYSIS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE EMBEDDING ANALYSIS")
print("=" * 70)

embedding_hashes = pd.util.hash_pandas_object(
    df[embedding_cols],
    index=False
)

duplicate_mask = embedding_hashes.duplicated(keep=False)

rows_in_duplicate_embeddings = int(duplicate_mask.sum())

duplicate_groups = int(
    embedding_hashes[duplicate_mask].nunique()
)

print(
    f"Rows involved in duplicate embeddings: "
    f"{rows_in_duplicate_embeddings}"
)

print(
    f"Duplicate embedding groups: "
    f"{duplicate_groups}"
)

# Check whether duplicate vectors occur across different labels.
cross_label_groups = 0
same_label_groups = 0

if duplicate_groups > 0:

    temp = pd.DataFrame({
        "hash": embedding_hashes,
        "target": df[TARGET_COL].values
    })

    for _, group in temp[temp["hash"].duplicated(keep=False)].groupby("hash"):

        if group["target"].nunique() > 1:
            cross_label_groups += 1
        else:
            same_label_groups += 1

print(f"Same-label duplicate groups : {same_label_groups}")
print(f"Cross-label duplicate groups: {cross_label_groups}")

# IMPORTANT:
# Duplicate embeddings are documented rather than automatically
# deleting them. They may arise from repeated environmental profiles.

duplicate_documented_pass = True

# ----------------------------------------------------------------
# VALIDATION RESULTS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("E6 FINAL VALIDATION RESULTS")
print("=" * 70)

results = {
    "Expected sample count": sample_count_pass,
    "Expected 32-D representation": dimension_pass,
    "Sample ID column present": ID_COL in df.columns,
    "No duplicate sample IDs": id_unique_pass,
    "No missing sample IDs": id_missing_pass,
    "Target column present": TARGET_COL in df.columns,
    "No target NaN": target_nan == 0,
    "Valid binary target": binary_target_pass,
    "Target distribution matches source": target_distribution_match,
    "All embedding columns numeric": numeric_pass,
    "No NaN in embeddings": nan_pass,
    "No Inf in embeddings": inf_pass,
    "No constant embedding dimensions": constant_pass,
    "Duplicate embeddings documented": duplicate_documented_pass
}

for name, status in results.items():

    print(
        f"{name:<45}: "
        f"{'PASSED' if status else 'FAILED'}"
    )

overall_pass = all(results.values())

# ----------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------

print("\n" + "=" * 70)

if overall_pass:

    print("VALIDATION STATUS : PASSED")
    print()
    print(
        "Environmental 32-D representation is VALIDATED."
    )
    print(
        "It is ready for the final multimodal fusion stage."
    )

else:

    print("VALIDATION STATUS : FAILED")
    print()
    print(
        "Environmental 32-D representation requires "
        "correction before final fusion."
    )

print("=" * 70)

# ----------------------------------------------------------------
# SAVE REPORT
# ----------------------------------------------------------------

report_lines = []

report_lines.append(
    "ENVIRONMENTAL MODEL — STEP E6\n"
)

report_lines.append(
    f"Input file: {INPUT_FILE}\n"
)

report_lines.append(
    f"Samples: {actual_samples}\n"
)

report_lines.append(
    f"Embedding dimensions: {len(embedding_cols)}\n"
)

report_lines.append(
    f"Duplicate sample IDs: {duplicate_ids}\n"
)

report_lines.append(
    f"Missing sample IDs: {missing_ids}\n"
)

report_lines.append(
    f"Target NaN: {target_nan}\n"
)

report_lines.append(
    f"Target distribution: {actual_distribution}\n"
)

report_lines.append(
    f"NaN embeddings: {nan_values}\n"
)

report_lines.append(
    f"Inf embeddings: {inf_values}\n"
)

report_lines.append(
    f"Constant dimensions: {len(constant_dimensions)}\n"
)

report_lines.append(
    f"Duplicate embedding groups: {duplicate_groups}\n"
)

report_lines.append(
    f"Same-label duplicate groups: {same_label_groups}\n"
)

report_lines.append(
    f"Cross-label duplicate groups: {cross_label_groups}\n"
)

report_lines.append(
    "\nVALIDATION RESULTS\n"
)

for name, status in results.items():

    report_lines.append(
        f"{name}: {'PASSED' if status else 'FAILED'}\n"
    )

report_lines.append(
    f"\nOVERALL STATUS: "
    f"{'PASSED' if overall_pass else 'FAILED'}\n"
)

with open(REPORT_FILE, "w") as f:
    f.writelines(report_lines)

print("\nReport saved:")
print(REPORT_FILE)

print("\n" + "=" * 70)
print("E6 COMPLETED")
print("=" * 70)

ENVIRONMENTAL MODEL — STEP E6
32-D ENVIRONMENTAL REPRESENTATION VALIDATION

Loading environmental 32-D representation...
Done.

DATASET STRUCTURE
Sample ID column      : Sample_Index
Target column         : Target
Embedding dimensions  : 32

Expected samples      : 47768
Actual samples        : 47768

Expected dimensions   : 32
Actual dimensions     : 32

SAMPLE ID VALIDATION
Duplicate sample IDs : 0
Missing sample IDs   : 0
Source dataset has no Sample_Index column.
Order comparison skipped.

TARGET VALIDATION
Target NaN values: 0

Target distribution:
Target
0    45650
1     2118
Name: count, dtype: int64

Unique target values: [0, 1]
Binary target valid: True

Source distribution:
{0: 45650, 1: 2118}

32-D distribution:
{0: 45650, 1: 2118}

Target distribution preserved: True

EMBEDDING VALUE VALIDATION
Non-numeric embedding columns: []
NaN values                  : 0
Inf values                  : 0

EMBEDDING DIMENSION CHECK
Constant embedding dimensions: 0

32-D EMBEDDING STATISTI

In [ ]:
# ==================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10
# ENVIRONMENTAL BRANCH — COMPLETE ARTIFACT + CHECKPOINT INSPECTION
# NO RECONSTRUCTION / NO TRAINING / NO CHECKPOINT MODIFICATION
# ==================================================================================================

import os
import glob
import json
import joblib
import torch
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10")
print("ENVIRONMENTAL BRANCH — COMPLETE ARTIFACT + CHECKPOINT INSPECTION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. VERIFY DRIVE / PROJECT
# --------------------------------------------------------------------------------------------------

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}\n\n"
        "Reconnect Google Drive and rerun this cell."
    )

print("[PASSED] Project directory available.")
print(f"Project : {PROJECT_DIR}")

if not os.path.isdir(DEMO_DIR):
    raise FileNotFoundError(f"Demo directory not found:\n{DEMO_DIR}")

print(f"[FOUND] Demo directory\n        {DEMO_DIR}")

# --------------------------------------------------------------------------------------------------
# 2. DISCOVER ENVIRONMENTAL-RELATED ARTIFACTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. DISCOVERING ENVIRONMENTAL ARTIFACTS")
print("=" * 100)

environmental_keywords = [
    "environment",
    "environmental",
    "env",
]

all_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for fname in files:
        full_path = os.path.join(root, fname)
        all_files.append(full_path)

environmental_files = []

for path in all_files:
    lower = path.lower()
    if any(k in lower for k in environmental_keywords):
        environmental_files.append(path)

environmental_files = sorted(set(environmental_files))

print(f"Environmental-related files discovered : {len(environmental_files)}")

for path in environmental_files:
    try:
        size_mb = os.path.getsize(path) / (1024 * 1024)
    except:
        size_mb = 0.0

    print(f"[FOUND] {size_mb:.4f} MB")
    print(f"        {path}")

# --------------------------------------------------------------------------------------------------
# 3. DISCOVER ENVIRONMENTAL PYTORCH CHECKPOINTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. DISCOVERING ENVIRONMENTAL PYTORCH CHECKPOINTS")
print("=" * 100)

pt_files = []

for path in all_files:
    if path.lower().endswith((".pt", ".pth")):
        pt_files.append(path)

environmental_checkpoints = []

for path in pt_files:
    lower = path.lower()

    if any(k in lower for k in environmental_keywords):
        environmental_checkpoints.append(path)

environmental_checkpoints = sorted(set(environmental_checkpoints))

print(f"[FOUND] Environmental checkpoint candidates: "
      f"{len(environmental_checkpoints)}")

for i, path in enumerate(environmental_checkpoints, 1):
    print(f"\n[{i}] {path}")

# --------------------------------------------------------------------------------------------------
# 4. DISCOVER ENVIRONMENTAL PREPROCESSING ARTIFACTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. DISCOVERING ENVIRONMENTAL PREPROCESSING ARTIFACTS")
print("=" * 100)

preprocessor_files = []

for path in all_files:
    lower = path.lower()

    if lower.endswith((".joblib", ".pkl", ".pickle")):
        if any(k in lower for k in environmental_keywords):
            preprocessor_files.append(path)

preprocessor_files = sorted(set(preprocessor_files))

print(f"Environmental preprocessing artifacts: "
      f"{len(preprocessor_files)}")

for path in preprocessor_files:
    print(f"[FOUND] {path}")

# --------------------------------------------------------------------------------------------------
# 5. INSPECT P001-P005 ENVIRONMENTAL INPUTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. INSPECTING P001-P005 ENVIRONMENTAL INPUTS")
print("=" * 100)

required_patients = ["P001", "P002", "P003", "P004", "P005"]

environmental_inputs = {}

for pid in required_patients:

    patient_dir = os.path.join(DEMO_DIR, pid)

    if not os.path.isdir(patient_dir):
        print(f"[MISSING] {pid} patient directory")
        continue

    candidates = []

    for fname in os.listdir(patient_dir):
        full_path = os.path.join(patient_dir, fname)

        if not os.path.isfile(full_path):
            continue

        lower = fname.lower()

        if (
            "environment" in lower
            or "environmental" in lower
            or lower.startswith("env")
        ):
            candidates.append(full_path)

    # Also explicitly check expected filename.
    expected = os.path.join(patient_dir, "environmental_input.csv")

    if os.path.isfile(expected):
        candidates.insert(0, expected)

    candidates = list(dict.fromkeys(candidates))

    if not candidates:
        print(f"[MISSING] {pid} environmental input")
        continue

    selected = candidates[0]
    environmental_inputs[pid] = selected

    print(f"[FOUND] {pid}")
    print(f"        {selected}")

print(
    f"\n[PASSED] Environmental inputs resolved: "
    f"{len(environmental_inputs)}/5"
)

# --------------------------------------------------------------------------------------------------
# 6. INSPECT ENVIRONMENTAL INPUT STRUCTURE
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. ENVIRONMENTAL INPUT STRUCTURE")
print("=" * 100)

environmental_structure = {}

for pid in required_patients:

    if pid not in environmental_inputs:
        continue

    path = environmental_inputs[pid]

    try:
        df = pd.read_csv(path)

        numeric_cols = df.select_dtypes(
            include=[np.number]
        ).columns.tolist()

        categorical_cols = [
            c for c in df.columns
            if c not in numeric_cols
        ]

        environmental_structure[pid] = {
            "shape": tuple(df.shape),
            "columns": list(df.columns),
            "numeric_columns": numeric_cols,
            "categorical_columns": categorical_cols,
        }

        print(f"\n--- {pid} ---")
        print(f"Shape              : {df.shape}")
        print(f"Total columns      : {len(df.columns)}")
        print(f"Numeric columns    : {len(numeric_cols)}")
        print(f"Categorical columns : {len(categorical_cols)}")

        print("Columns:")
        for col in df.columns:
            print(f"  {col}")

    except Exception as e:
        print(f"[ERROR] Could not inspect {pid}: {e}")

# --------------------------------------------------------------------------------------------------
# 7. INSPECT ALL ENVIRONMENTAL CHECKPOINTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. EXACT ENVIRONMENTAL CHECKPOINT INSPECTION")
print("=" * 100)

checkpoint_reports = []

for idx, path in enumerate(environmental_checkpoints, 1):

    print("\n" + "-" * 100)
    print(f"CHECKPOINT {idx}")
    print("-" * 100)

    print(f"File: {path}")

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print(f"Checkpoint type: {type(checkpoint)}")

        # Determine state dict
        if isinstance(checkpoint, dict):

            if "model_state_dict" in checkpoint:
                state_dict = checkpoint["model_state_dict"]
                print("[FOUND] model_state_dict")

            elif all(
                isinstance(v, torch.Tensor)
                for v in checkpoint.values()
            ):
                state_dict = checkpoint
                print("[FOUND] checkpoint itself appears to be state_dict")

            else:
                state_dict = None
                print("[INFO] No direct state_dict identified")

            print("\nTop-level keys:")

            for key, value in checkpoint.items():

                if isinstance(value, torch.Tensor):
                    print(
                        f"  {key:<55} "
                        f"tensor {tuple(value.shape)}"
                    )
                else:
                    print(
                        f"  {key:<55} "
                        f"type={type(value)} value={value}"
                    )

            if state_dict is not None:

                print("\nExact state-dict keys:")

                linear_candidates = []
                bn_candidates = []

                for key, tensor in state_dict.items():

                    print(
                        f"  {key:<65} "
                        f"shape={tuple(tensor.shape)}"
                    )

                    if key.endswith(".weight") and tensor.ndim == 2:
                        linear_candidates.append(
                            (key, tuple(tensor.shape))
                        )

                    if (
                        "running_mean" in key
                        or "running_var" in key
                    ):
                        bn_candidates.append(key)

                print("\nLinear layer candidates:")

                if linear_candidates:
                    for key, shape in linear_candidates:
                        print(
                            f"  {key:<55} "
                            f"{shape[1]} -> {shape[0]}"
                        )
                else:
                    print("  [NONE]")

                print("\nBatchNorm-related keys:")

                if bn_candidates:
                    for key in bn_candidates:
                        print(f"  {key}")
                else:
                    print("  [NONE FOUND]")

                checkpoint_reports.append({
                    "path": path,
                    "state_dict_keys": list(state_dict.keys()),
                    "linear_candidates": linear_candidates,
                    "batchnorm_keys": bn_candidates,
                })

        else:
            print("[WARNING] Unsupported checkpoint structure.")

    except Exception as e:
        print(f"[ERROR] Failed to inspect checkpoint:")
        print(e)

# --------------------------------------------------------------------------------------------------
# 8. INSPECT PREPROCESSING ARTIFACTS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. ENVIRONMENTAL PREPROCESSING INSPECTION")
print("=" * 100)

preprocessor_reports = []

for path in preprocessor_files:

    print("\n" + "-" * 100)
    print(f"FILE: {path}")
    print("-" * 100)

    try:

        obj = joblib.load(path)

        print(f"Type: {type(obj)}")

        if hasattr(obj, "n_features_in_"):
            print(
                f"n_features_in_: "
                f"{obj.n_features_in_}"
            )

        if hasattr(obj, "feature_names_in_"):
            print(
                f"feature_names_in_: "
                f"{list(obj.feature_names_in_)}"
            )

        if hasattr(obj, "transformers_"):

            print("\nColumnTransformer details:")

            for name, transformer, columns in obj.transformers_:
                print(f"\n  Transformer: {name}")
                print(f"  Columns    : {columns}")

        preprocessor_reports.append({
            "path": path,
            "type": str(type(obj)),
            "n_features_in": getattr(
                obj,
                "n_features_in_",
                None
            ),
        })

    except Exception as e:
        print(f"[ERROR] Could not load artifact: {e}")

# --------------------------------------------------------------------------------------------------
# 9. MANIFEST CROSS-CHECK
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("9. DEMO MANIFEST CROSS-CHECK")
print("=" * 100)

manifest_path = os.path.join(
    DEMO_DIR,
    "M4_demo_patient_manifest.csv"
)

if os.path.isfile(manifest_path):

    manifest = pd.read_csv(manifest_path)

    print(f"[FOUND] Manifest")
    print(f"Shape: {manifest.shape}")
    print(f"Columns: {list(manifest.columns)}")

    if "patient_id" in manifest.columns:

        for pid in required_patients:

            row = manifest[
                manifest["patient_id"].astype(str) == pid
            ]

            if len(row) == 0:
                print(f"[MISSING] {pid} manifest entry")
                continue

            print(f"\n{pid}")

            for col in manifest.columns:
                print(
                    f"  {col}: "
                    f"{row.iloc[0][col]}"
                )

else:
    print("[WARNING] Demo manifest not found.")

# --------------------------------------------------------------------------------------------------
# 10. FINAL STATUS
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("M4.4.10 ENVIRONMENTAL INSPECTION STATUS")
print("=" * 100)

print(
    f"[INFO] Environmental inputs resolved : "
    f"{len(environmental_inputs)}/5"
)

print(
    f"[INFO] Environmental checkpoints     : "
    f"{len(environmental_checkpoints)}"
)

print(
    f"[INFO] Environmental preprocessors   : "
    f"{len(preprocessor_files)}"
)

if len(environmental_inputs) == 5:
    print("[PASSED] All P001-P005 environmental inputs found.")
else:
    print("[BLOCKED] Not all environmental inputs were found.")

if len(environmental_checkpoints) > 0:
    print("[PASSED] Environmental checkpoint(s) discovered.")
else:
    print("[INFO] No environmental PyTorch checkpoint discovered.")

print("\nNO MODEL WAS RECONSTRUCTED.")
print("NO TRAINING WAS PERFORMED.")
print("NO CHECKPOINT WAS MODIFIED.")
print("NO PRODUCTION ARTIFACT WAS OVERWRITTEN.")

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)

print("""
Paste the COMPLETE FINAL SUMMARY/output.

We will then reconstruct ONLY the environmental architecture
proven by the actual artifacts/source, validate P001-P005,
and immediately proceed to multimodal fusion.
""")

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10
ENVIRONMENTAL BRANCH — COMPLETE ARTIFACT + CHECKPOINT INSPECTION
[PASSED] Project directory available.
Project : /content/gdrive/MyDrive/ASD_Multimodal_Project
[FOUND] Demo directory
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients

2. DISCOVERING ENVIRONMENTAL ARTIFACTS
Environmental-related files discovered : 27
[FOUND] 0.0218 MB
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt
[FOUND] 0.0013 MB
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_scaler.joblib
[FOUND] 0.0003 MB
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P001/environmental_input.csv
[FOUND] 0.0003 MB
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P002/environmental_input.csv
[FOUND] 0.0003 MB
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_m

In [ ]:
# ====================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.1
# EXACT ENVIRONMENTAL BRANCH ARTIFACT + CHECKPOINT INSPECTION
# INSPECTION ONLY — NO MODEL RECONSTRUCTION / NO TRAINING
# ====================================================================================================

import os
import glob
import json
import joblib
import torch
import pandas as pd
from collections import OrderedDict

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.1")
print("EXACT ENVIRONMENTAL BRANCH ARTIFACT + CHECKPOINT INSPECTION")
print("=" * 100)

# ----------------------------------------------------------------------------------------------------
# 1. DRIVE / PROJECT VERIFICATION
# ----------------------------------------------------------------------------------------------------

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
DEMO_DIR = os.path.join(PROJECT_DIR, "05_multimodal_fusion", "M4_demo_patients")

print("\n" + "=" * 100)
print("1. DRIVE / PROJECT VERIFICATION")
print("=" * 100)

if not os.path.isdir("/content/gdrive"):
    try:
        from google.colab import drive
        drive.mount("/content/gdrive")
        print("[PASSED] Google Drive mounted.")
    except Exception as e:
        print("[WARNING] Drive mount attempt failed:", e)

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}\n"
        "Reconnect Google Drive and run this cell again."
    )

print("[PASSED] Project directory available.")
print("Project :", PROJECT_DIR)

if os.path.isdir(DEMO_DIR):
    print("[FOUND] Demo directory :", DEMO_DIR)
else:
    print("[WARNING] Demo directory not found :", DEMO_DIR)

# ----------------------------------------------------------------------------------------------------
# 2. DISCOVER ENVIRONMENTAL-RELATED FILES
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. DISCOVERING ENVIRONMENTAL-RELATED FILES")
print("=" * 100)

all_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for f in files:
        all_files.append(os.path.join(root, f))

environmental_keywords = [
    "environment",
    "environmental",
    "env_",
    "_env",
    "context",
    "social",
    "clinical",
    "demographic"
]

genetic_keywords = ["genetic", "transformer", "genome"]
behavioral_keywords = ["behavioral", "behavioural", "behavior"]

environmental_files = []

for path in all_files:
    name = os.path.basename(path).lower()
    full = path.lower()

    if any(k in name or k in full for k in environmental_keywords):
        environmental_files.append(path)

print(f"[INFO] Total project files discovered : {len(all_files)}")
print(f"[INFO] Environmental-related files   : {len(environmental_files)}")

for path in sorted(environmental_files):
    print(" ", path)

# ----------------------------------------------------------------------------------------------------
# 3. ENVIRONMENTAL PYTORCH CHECKPOINT DISCOVERY
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. ENVIRONMENTAL PYTORCH CHECKPOINT DISCOVERY")
print("=" * 100)

pt_candidates = []

for path in all_files:
    if path.lower().endswith((".pt", ".pth", ".ckpt")):
        name = os.path.basename(path).lower()
        full = path.lower()

        # Environmental candidates
        if any(k in name or k in full for k in environmental_keywords):
            pt_candidates.append(path)

# Also inspect likely multimodal fusion folders for environmental models
fusion_dir = os.path.join(PROJECT_DIR, "05_multimodal_fusion")

if os.path.isdir(fusion_dir):
    for path in glob.glob(os.path.join(fusion_dir, "**", "*"), recursive=True):
        if os.path.isfile(path) and path.lower().endswith((".pt", ".pth", ".ckpt")):
            if path not in pt_candidates:
                pt_candidates.append(path)

pt_candidates = sorted(set(pt_candidates))

if not pt_candidates:
    print("[WARNING] No environmental checkpoint identified automatically.")
else:
    print(f"[FOUND] Environmental/fusion checkpoint candidates : {len(pt_candidates)}")

    for i, path in enumerate(pt_candidates, 1):
        print(f"\n[{i}] {path}")

# ----------------------------------------------------------------------------------------------------
# 4. ENVIRONMENTAL PREPROCESSING ARTIFACT DISCOVERY
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. ENVIRONMENTAL PREPROCESSING ARTIFACT DISCOVERY")
print("=" * 100)

preprocess_candidates = []

for path in all_files:
    lower = path.lower()

    if lower.endswith((
        ".joblib",
        ".pkl",
        ".pickle",
        ".json"
    )):
        name = os.path.basename(path).lower()

        if (
            any(k in name or k in lower for k in environmental_keywords)
            or "preprocess" in name
            or "scaler" in name
            or "encoder" in name
        ):
            preprocess_candidates.append(path)

preprocess_candidates = sorted(set(preprocess_candidates))

print(f"[INFO] Candidate preprocessing artifacts : {len(preprocess_candidates)}")

for path in preprocess_candidates:
    print(" ", path)

# ----------------------------------------------------------------------------------------------------
# 5. ENVIRONMENTAL DEMO INPUT DISCOVERY
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. ENVIRONMENTAL DEMO INPUT DISCOVERY")
print("=" * 100)

patient_ids = ["P001", "P002", "P003", "P004", "P005"]

environmental_demo_files = {}

for pid in patient_ids:

    patient_dir = os.path.join(DEMO_DIR, pid)

    candidates = []

    if os.path.isdir(patient_dir):
        for path in glob.glob(os.path.join(patient_dir, "**", "*"), recursive=True):

            if not os.path.isfile(path):
                continue

            name = os.path.basename(path).lower()

            if (
                "environment" in name
                or "environmental" in name
                or name.startswith("env")
                or "context" in name
                or "social" in name
            ):
                candidates.append(path)

    if candidates:
        environmental_demo_files[pid] = sorted(candidates)[0]
        print(f"[FOUND] {pid} -> {sorted(candidates)[0]}")
    else:
        print(f"[NOT FOUND] {pid} -> no obvious environmental input")

print(
    f"\nResolved environmental demo patients : "
    f"{len(environmental_demo_files)}/5"
)

# ----------------------------------------------------------------------------------------------------
# 6. INSPECT ENVIRONMENTAL DEMO FILE STRUCTURES
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. ENVIRONMENTAL DEMO INPUT STRUCTURE INSPECTION")
print("=" * 100)

for pid, path in environmental_demo_files.items():

    print("\n" + "-" * 100)
    print(pid)
    print("-" * 100)

    try:

        if path.lower().endswith(".csv"):

            df = pd.read_csv(path)

            print("File       :", path)
            print("Shape      :", df.shape)
            print("Columns    :", len(df.columns))
            print("Numeric    :", len(df.select_dtypes(include="number").columns))
            print("Non-numeric:", len(df.select_dtypes(exclude="number").columns))

            print("\nColumns:")
            for c in df.columns:
                print(f"  {c!r}")

        elif path.lower().endswith((".json", ".jsonl")):

            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            print("File :", path)
            print("JSON type :", type(data))

            if isinstance(data, dict):
                print("Keys:")
                for k in data.keys():
                    print(" ", k)

            elif isinstance(data, list):
                print("List length :", len(data))

                if data:
                    print("First item type :", type(data[0]))
                    if isinstance(data[0], dict):
                        print("First item keys:")
                        for k in data[0].keys():
                            print(" ", k)

        else:
            print("File :", path)
            print("Extension not directly tabular; structure inspection skipped.")

    except Exception as e:
        print("[WARNING] Could not inspect:", e)

# ----------------------------------------------------------------------------------------------------
# 7. CHECKPOINT STRUCTURE INSPECTION
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. EXACT ENVIRONMENTAL CHECKPOINT STRUCTURE INSPECTION")
print("=" * 100)

checkpoint_reports = []

for path in pt_candidates:

    print("\n" + "-" * 100)
    print("CHECKPOINT")
    print("-" * 100)
    print("File :", path)

    try:

        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print("Checkpoint type :", type(checkpoint))

        # Determine state dict
        if isinstance(checkpoint, OrderedDict):
            state_dict = checkpoint
            print("[FOUND] Checkpoint itself appears to be a state_dict.")

        elif isinstance(checkpoint, dict):

            print("Top-level keys:")

            for k, v in checkpoint.items():

                if isinstance(v, dict):
                    print(
                        f"  {k:<35} type={type(v)} "
                        f"entries={len(v)}"
                    )

                elif torch.is_tensor(v):
                    print(
                        f"  {k:<35} Tensor shape={tuple(v.shape)}"
                    )

                else:
                    print(
                        f"  {k:<35} "
                        f"type={type(v)} value={v}"
                    )

            if "model_state_dict" in checkpoint:
                state_dict = checkpoint["model_state_dict"]
                print("[FOUND] model_state_dict")

            elif "state_dict" in checkpoint:
                state_dict = checkpoint["state_dict"]
                print("[FOUND] state_dict")

            else:
                # Look for a dictionary containing tensor values
                possible = []

                for k, v in checkpoint.items():
                    if isinstance(v, (dict, OrderedDict)):
                        tensor_count = sum(
                            torch.is_tensor(x)
                            for x in v.values()
                        )

                        if tensor_count > 0:
                            possible.append((k, v))

                if len(possible) == 1:
                    state_dict = possible[0][1]
                    print(
                        f"[FOUND] Tensor state_dict under key "
                        f"'{possible[0][0]}'"
                    )
                else:
                    state_dict = None

        else:
            state_dict = None

        if state_dict is None:
            print("[WARNING] No state_dict-like structure identified.")
            continue

        print("\nExact state-dict keys :", len(state_dict))

        linear_candidates = []
        bn_candidates = []
        transformer_candidates = []

        print("\nState-dict entries:")

        for key, tensor in state_dict.items():

            shape = tuple(tensor.shape)

            print(
                f"{key:<65} "
                f"shape={str(shape):<22} "
                f"dtype={tensor.dtype}"
            )

            if key.endswith(".weight") and len(shape) == 2:

                out_features, in_features = shape

                linear_candidates.append(
                    (key, in_features, out_features)
                )

            if (
                "running_mean" in key
                or "running_var" in key
            ):
                bn_candidates.append(key)

            if (
                "transformer" in key.lower()
                or "self_attn" in key.lower()
                or "multihead" in key.lower()
            ):
                transformer_candidates.append(key)

        print("\n" + "-" * 100)
        print("LINEAR LAYER CANDIDATES")
        print("-" * 100)

        for key, inp, out in linear_candidates:
            print(
                f"{key:<55} "
                f"Linear candidate: in={inp:<5} out={out}"
            )

        print("\n" + "-" * 100)
        print("BATCH NORMALIZATION DETECTION")
        print("-" * 100)

        if bn_candidates:
            modules = sorted(
                set(
                    k.rsplit(".", 1)[0]
                    for k in bn_candidates
                )
            )

            for module in modules:
                print("[FOUND] BatchNorm-related module:", module)
        else:
            print("[NONE FOUND] BatchNorm running-stat buffers.")

        print("\n" + "-" * 100)
        print("TRANSFORMER STRUCTURE DETECTION")
        print("-" * 100)

        if transformer_candidates:
            print(
                f"[FOUND] Transformer-related keys: "
                f"{len(transformer_candidates)}"
            )

            for k in transformer_candidates[:20]:
                print(" ", k)

            if len(transformer_candidates) > 20:
                print(
                    f" ... {len(transformer_candidates)-20} more"
                )
        else:
            print("[NONE FOUND] Transformer-specific keys.")

        checkpoint_reports.append({
            "path": path,
            "state_dict_keys": list(state_dict.keys()),
            "linear_candidates": linear_candidates,
            "batchnorm": bool(bn_candidates),
            "transformer": bool(transformer_candidates)
        })

    except Exception as e:

        print("[ERROR] Could not inspect checkpoint:")
        print(type(e).__name__, e)

# ----------------------------------------------------------------------------------------------------
# 8. PREPROCESSING ARTIFACT DETAILS
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. PREPROCESSING ARTIFACT DETAILS")
print("=" * 100)

for path in preprocess_candidates:

    print("\n" + "-" * 100)
    print(path)
    print("-" * 100)

    try:

        if path.lower().endswith((".joblib", ".pkl", ".pickle")):

            obj = joblib.load(path)

            print("Type :", type(obj))

            if hasattr(obj, "n_features_in_"):
                print(
                    "n_features_in_ :",
                    obj.n_features_in_
                )

            if hasattr(obj, "feature_names_in_"):
                print(
                    "feature_names_in_ :"
                )

                for x in obj.feature_names_in_:
                    print(" ", x)

            if hasattr(obj, "transformers_"):

                print("\nColumnTransformer components:")

                for item in obj.transformers_:
                    print(" ", item[0], "->", type(item[1]))

        elif path.lower().endswith(".json"):

            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            print("JSON type :", type(data))

            if isinstance(data, dict):
                for k, v in data.items():
                    print(f"  {k}: {type(v).__name__}")

    except Exception as e:
        print("[WARNING] Could not load artifact:", e)

# ----------------------------------------------------------------------------------------------------
# 9. FINAL INSPECTION SUMMARY
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("M4.4.10.1 FINAL INSPECTION SUMMARY")
print("=" * 100)

print(
    "Environmental-related files discovered :",
    len(environmental_files)
)

print(
    "Environmental checkpoint candidates    :",
    len(pt_candidates)
)

print(
    "Environmental preprocessing candidates:",
    len(preprocess_candidates)
)

print(
    "Environmental demo patients resolved   :",
    f"{len(environmental_demo_files)}/5"
)

print(
    "Checkpoint structures successfully inspected:",
    len(checkpoint_reports)
)

print("\n[PASSED] Environmental artifact discovery completed.")
print("[PASSED] Environmental demo inputs inspected where available.")
print("[PASSED] Environmental checkpoint structures inspected.")
print("[PASSED] Preprocessing artifacts inspected where available.")
print("[PASSED] No environmental model reconstructed.")
print("[PASSED] No training performed.")
print("[PASSED] No checkpoint modified.")
print("[PASSED] No production artifact overwritten.")

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)

print("""
Paste the COMPLETE output here.

The next cell will reconstruct ONLY the environmental architecture
that is proven by the actual checkpoint/source structure.

Then we will proceed directly:

ENVIRONMENTAL INFERENCE
        ↓
MULTIMODAL FUSION INSPECTION
        ↓
FUSION RECONSTRUCTION
        ↓
P001-P005 COMPLETE 3-MODAL INFERENCE
        ↓
FINAL INTERFACE

Behavioral and genetic checkpoints will NOT be reconstructed again.
""")

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.1
EXACT ENVIRONMENTAL BRANCH ARTIFACT + CHECKPOINT INSPECTION

1. DRIVE / PROJECT VERIFICATION
[PASSED] Project directory available.
Project : /content/gdrive/MyDrive/ASD_Multimodal_Project
[FOUND] Demo directory : /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients

2. DISCOVERING ENVIRONMENTAL-RELATED FILES
[INFO] Total project files discovered : 90
[INFO] Environmental-related files   : 27
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_scaler.joblib
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P001/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P002/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P003/environmenta

In [ ]:
# ====================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.2
# EXACT ENVIRONMENTAL CHECKPOINT + PREPROCESSING INSPECTION
# INSPECTION ONLY — NO RECONSTRUCTION / NO TRAINING
# ====================================================================================================

import os
import glob
import json
import joblib
import torch
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
DEVICE = torch.device("cpu")

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.2")
print("EXACT ENVIRONMENTAL CHECKPOINT + PREPROCESSING INSPECTION")
print("=" * 100)

# ----------------------------------------------------------------------------------------------------
# 1. PROJECT CHECK
# ----------------------------------------------------------------------------------------------------

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}\n"
        "Reconnect Google Drive before running this cell."
    )

print("[PASSED] Project directory available.")
print(f"Project : {PROJECT_DIR}")

# ----------------------------------------------------------------------------------------------------
# 2. DISCOVER ENVIRONMENTAL ARTIFACTS
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. DISCOVERING ENVIRONMENTAL ARTIFACTS")
print("=" * 100)

all_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for f in files:
        all_files.append(os.path.join(root, f))

environmental_files = [
    p for p in all_files
    if any(
        token in os.path.basename(p).lower()
        or token in os.path.dirname(p).lower()
        for token in ["environment", "environmental", "env_"]
    )
]

print(f"[FOUND] Environmental-related artifacts: {len(environmental_files)}")

for p in environmental_files[:100]:
    print(" ", p)

# ----------------------------------------------------------------------------------------------------
# 3. DISCOVER PYTORCH CHECKPOINTS
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. ENVIRONMENTAL PYTORCH CHECKPOINTS")
print("=" * 100)

pt_candidates = [
    p for p in environmental_files
    if p.lower().endswith((".pt", ".pth", ".ckpt"))
]

# Also inspect any checkpoint whose filename strongly suggests environmental role
if not pt_candidates:
    pt_candidates = [
        p for p in all_files
        if p.lower().endswith((".pt", ".pth", ".ckpt"))
        and any(
            x in os.path.basename(p).lower()
            for x in ["environment", "environmental", "env_"]
        )
    ]

print(f"[FOUND] Environmental checkpoint candidates: {len(pt_candidates)}")

for i, p in enumerate(pt_candidates, 1):
    print(f"\n[{i}] {p}")

if not pt_candidates:
    print("[WARNING] No environmental PyTorch checkpoint found.")

# ----------------------------------------------------------------------------------------------------
# 4. INSPECT EVERY CHECKPOINT — EXACTLY
# ----------------------------------------------------------------------------------------------------

checkpoint_records = []

for idx, path in enumerate(pt_candidates, 1):

    print("\n" + "-" * 100)
    print(f"CHECKPOINT {idx}")
    print("-" * 100)
    print(f"File : {path}")

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print(f"Checkpoint type : {type(checkpoint)}")

        # Determine state dict
        if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
            print("[FOUND] model_state_dict")
        elif isinstance(checkpoint, dict):
            tensor_values = [
                v for v in checkpoint.values()
                if torch.is_tensor(v)
            ]

            if len(tensor_values) > 0:
                state_dict = checkpoint
                print("[FOUND] Checkpoint itself appears to be a state_dict")
            else:
                state_dict = None
                print("[INFO] No direct state_dict detected.")
        else:
            state_dict = None

        if state_dict is not None:

            print(f"State-dict keys : {len(state_dict)}")

            linear_candidates = []
            bn_candidates = []
            layernorm_candidates = []

            for key, tensor in state_dict.items():

                if not torch.is_tensor(tensor):
                    continue

                shape = tuple(tensor.shape)

                print(
                    f"{key:<65} "
                    f"shape={str(shape):<22} "
                    f"dtype={str(tensor.dtype)}"
                )

                # Linear candidates
                if key.endswith(".weight") and len(shape) == 2:
                    out_dim, in_dim = shape
                    linear_candidates.append(
                        (key, in_dim, out_dim)
                    )

                # BatchNorm candidates
                if (
                    "running_mean" in key
                    or "running_var" in key
                ):
                    module_name = key.rsplit(".", 1)[0]
                    if module_name not in bn_candidates:
                        bn_candidates.append(module_name)

            # Detect modules with weight+bias but no running statistics
            module_names = {}

            for key in state_dict.keys():
                if "." in key:
                    module, param = key.rsplit(".", 1)
                    module_names.setdefault(module, set()).add(param)

            for module, params in module_names.items():
                if {"weight", "bias"}.issubset(params):
                    if module not in bn_candidates:
                        # Candidate LayerNorm if 1-D weight and no running stats
                        weight = state_dict.get(module + ".weight")

                        if (
                            torch.is_tensor(weight)
                            and weight.ndim == 1
                            and module not in bn_candidates
                        ):
                            layernorm_candidates.append(module)

            print("\n" + "-" * 100)
            print("LINEAR LAYER CANDIDATES")
            print("-" * 100)

            if linear_candidates:
                for key, in_dim, out_dim in linear_candidates:
                    print(
                        f"{key:<60} "
                        f"Linear candidate: in={in_dim:<6} out={out_dim}"
                    )
            else:
                print("[NONE FOUND]")

            print("\n" + "-" * 100)
            print("BATCH NORMALIZATION DETECTION")
            print("-" * 100)

            if bn_candidates:
                for module in bn_candidates:
                    print(f"[FOUND] BatchNorm-related module: {module}")
            else:
                print("[NONE FOUND]")

            print("\n" + "-" * 100)
            print("LAYERNORM CANDIDATE DETECTION")
            print("-" * 100)

            if layernorm_candidates:
                for module in layernorm_candidates:
                    print(f"[FOUND] LayerNorm-related module: {module}")
            else:
                print("[NONE FOUND]")

            checkpoint_records.append({
                "path": path,
                "state_dict": state_dict,
                "linear_candidates": linear_candidates,
                "batchnorm": bn_candidates,
                "layernorm": layernorm_candidates
            })

        else:
            print("[INFO] No inspectable state_dict found.")

    except Exception as e:
        print(f"[ERROR] Could not inspect checkpoint: {e}")

# ----------------------------------------------------------------------------------------------------
# 5. PREPROCESSING ARTIFACTS
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. ENVIRONMENTAL PREPROCESSING ARTIFACTS")
print("=" * 100)

preprocessor_candidates = [
    p for p in all_files
    if p.lower().endswith((
        ".joblib",
        ".pkl",
        ".pickle"
    ))
    and any(
        token in os.path.basename(p).lower()
        or token in os.path.dirname(p).lower()
        for token in [
            "environment",
            "environmental",
            "env_",
            "preprocess",
            "scaler"
        ]
    )
]

print(
    f"[FOUND] Environmental preprocessing candidates: "
    f"{len(preprocessor_candidates)}"
)

for i, p in enumerate(preprocessor_candidates, 1):
    print(f"[{i}] {p}")

# ----------------------------------------------------------------------------------------------------
# 6. INSPECT PREPROCESSING OBJECTS
# ----------------------------------------------------------------------------------------------------

preprocessor_records = []

for path in preprocessor_candidates:

    print("\n" + "-" * 100)
    print("PREPROCESSING ARTIFACT")
    print("-" * 100)
    print(path)

    try:
        obj = joblib.load(path)

        print(f"Object type : {type(obj)}")

        attrs = [
            "n_features_in_",
            "feature_names_in_",
            "n_components_",
            "mean_",
            "scale_",
            "categories_"
        ]

        for attr in attrs:
            if hasattr(obj, attr):
                value = getattr(obj, attr)

                if isinstance(value, np.ndarray):
                    if value.size <= 30:
                        print(f"{attr:<25}: {value}")
                    else:
                        print(
                            f"{attr:<25}: ndarray "
                            f"shape={value.shape}"
                        )
                else:
                    print(f"{attr:<25}: {value}")

        if hasattr(obj, "transformers_"):
            print("\nColumnTransformer transformers:")
            for name, transformer, columns in obj.transformers_:
                print(
                    f"  {name:<20} "
                    f"transformer={type(transformer)} "
                    f"columns={columns}"
                )

        preprocessor_records.append({
            "path": path,
            "object": obj
        })

    except Exception as e:
        print(f"[WARNING] Could not load: {e}")

# ----------------------------------------------------------------------------------------------------
# 7. DEMO PATIENTS
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. ENVIRONMENTAL DEMO PATIENT DISCOVERY")
print("=" * 100)

demo_dir = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

patient_ids = ["P001", "P002", "P003", "P004", "P005"]

environmental_demo_files = {}

for pid in patient_ids:

    patient_dir = os.path.join(demo_dir, pid)

    candidates = []

    if os.path.isdir(patient_dir):
        for root, dirs, files in os.walk(patient_dir):
            for f in files:
                full = os.path.join(root, f)

                low = f.lower()

                if (
                    "environment" in low
                    or "environmental" in low
                    or low.startswith("env")
                ):
                    if low.endswith((".csv", ".xlsx", ".xls", ".json")):
                        candidates.append(full)

    if candidates:
        environmental_demo_files[pid] = candidates[0]
        print(f"[FOUND] {pid}: {candidates[0]}")
    else:
        print(f"[WARNING] {pid}: no environmental demo file resolved")

print(
    f"\nResolved environmental demo files: "
    f"{len(environmental_demo_files)}/5"
)

# ----------------------------------------------------------------------------------------------------
# 8. INSPECT DEMO INPUT STRUCTURES
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. ENVIRONMENTAL DEMO INPUT STRUCTURES")
print("=" * 100)

for pid, path in environmental_demo_files.items():

    print("\n" + "-" * 100)
    print(pid)
    print("-" * 100)
    print(f"File : {path}")

    try:

        if path.lower().endswith(".csv"):
            df = pd.read_csv(path)

        elif path.lower().endswith((".xlsx", ".xls")):
            df = pd.read_excel(path)

        elif path.lower().endswith(".json"):
            df = pd.read_json(path)

        else:
            print("[SKIP] Unsupported format")
            continue

        print(f"Shape : {df.shape}")
        print(f"Columns: {len(df.columns)}")

        numeric_cols = df.select_dtypes(
            include=[np.number]
        ).columns.tolist()

        print(f"Numeric columns : {len(numeric_cols)}")

        print("Column names:")
        for col in df.columns:
            print(f"  {col}")

    except Exception as e:
        print(f"[ERROR] Could not inspect {pid}: {e}")

# ----------------------------------------------------------------------------------------------------
# 9. FINAL INSPECTION SUMMARY
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("M4.4.10.2 FINAL INSPECTION SUMMARY")
print("=" * 100)

print(
    f"Environmental checkpoints inspected : "
    f"{len(checkpoint_records)}"
)

print(
    f"Environmental preprocessing objects : "
    f"{len(preprocessor_records)}"
)

print(
    f"Environmental demo patients resolved: "
    f"{len(environmental_demo_files)}/5"
)

print("\nCheckpoint roles will NOT be guessed in this cell.")

print("\n" + "=" * 100)
print("M4.4.10.2 COMPLETED")
print("=" * 100)

print("[PASSED] Environmental checkpoint structures inspected.")
print("[PASSED] Exact tensor shapes inspected.")
print("[PASSED] BatchNorm/LayerNorm structures inspected.")
print("[PASSED] Environmental preprocessing artifacts inspected.")
print("[PASSED] P001-P005 environmental inputs inspected where available.")

print("\nNO MODEL RECONSTRUCTED.")
print("NO TRAINING PERFORMED.")
print("NO CHECKPOINT MODIFIED.")
print("NO PRODUCTION ARTIFACT OVERWRITTEN.")

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)

print("""
The next step will classify the actual environmental checkpoints
into encoder / alignment / expert roles using ONLY the structures
printed above.

After that:

ENVIRONMENTAL RECONSTRUCTION
        ↓
ENVIRONMENTAL P001-P005 INFERENCE
        ↓
MULTIMODAL FUSION INSPECTION
        ↓
FUSION RECONSTRUCTION
        ↓
P001-P005 COMPLETE 3-MODAL INFERENCE
        ↓
FINAL INTERFACE
""")

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.2
EXACT ENVIRONMENTAL CHECKPOINT + PREPROCESSING INSPECTION
[PASSED] Project directory available.
Project : /content/gdrive/MyDrive/ASD_Multimodal_Project

2. DISCOVERING ENVIRONMENTAL ARTIFACTS
[FOUND] Environmental-related artifacts: 27
  /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/01_raw_data/environmental_dataset.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/02_preprocessing/environmental_train_processed.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/02_preprocessing/environmental_test_processed_unseen.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/02_preprocessing/environmental_preprocessor.joblib
  /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/02_preprocessing/environmental_processed_feature_names.txt
  /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/02_preprocessing/environmental_cleaned_no_leakage.csv
  /c

In [ ]:
# ====================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.3
# EXACT ENVIRONMENTAL ROLE CLASSIFICATION
# KEY/SHAPE BASED — NO ARCHITECTURE GUESSING
# ====================================================================================================

import os
import glob
import torch

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.3")
print("EXACT ENVIRONMENTAL ROLE CLASSIFICATION")
print("=" * 100)

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory missing:\n{PROJECT_DIR}\n"
        "Reconnect Google Drive and rerun."
    )

print("[PASSED] Project directory available.")

# ----------------------------------------------------------------------------------------------------
# 1. DISCOVER ENVIRONMENTAL CHECKPOINTS
# ----------------------------------------------------------------------------------------------------

all_pt = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for f in files:
        if f.lower().endswith((".pt", ".pth", ".ckpt")):
            all_pt.append(os.path.join(root, f))

env_pt = [
    p for p in all_pt
    if any(
        token in p.lower()
        for token in [
            "environment",
            "environmental",
            "env_"
        ]
    )
]

# Remove duplicates
env_pt = sorted(set(env_pt))

print("\n" + "=" * 100)
print("1. ENVIRONMENTAL CHECKPOINTS")
print("=" * 100)

print(f"[FOUND] {len(env_pt)} environmental checkpoint(s)")

for i, p in enumerate(env_pt, 1):
    print(f"[{i}] {p}")

if len(env_pt) == 0:
    raise FileNotFoundError(
        "No environmental checkpoint found."
    )

# ----------------------------------------------------------------------------------------------------
# 2. INSPECT EXACT CHECKPOINT
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. EXACT CHECKPOINT STRUCTURE")
print("=" * 100)

checkpoint = torch.load(
    env_pt[0],
    map_location="cpu",
    weights_only=True
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state = checkpoint["model_state_dict"]
    print("[FOUND] model_state_dict")
else:
    state = checkpoint
    print("[FOUND] checkpoint appears to be state_dict")

print(f"State-dict entries: {len(state)}")

# ----------------------------------------------------------------------------------------------------
# 3. COMPACT KEY/SHAPE REPORT
# ----------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("EXACT KEYS + SHAPES")
print("-" * 100)

for key, value in state.items():
    if torch.is_tensor(value):
        print(f"{key:<70} {tuple(value.shape)}")

# ----------------------------------------------------------------------------------------------------
# 4. STRUCTURAL ANALYSIS
# ----------------------------------------------------------------------------------------------------

linear_layers = []
bn_modules = []
ln_candidates = []

for key, value in state.items():

    if not torch.is_tensor(value):
        continue

    shape = tuple(value.shape)

    if key.endswith(".weight") and len(shape) == 2:
        out_dim, in_dim = shape
        linear_layers.append(
            (key, in_dim, out_dim)
        )

    if "running_mean" in key or "running_var" in key:
        module = key.rsplit(".", 1)[0]
        if module not in bn_modules:
            bn_modules.append(module)

# LayerNorm candidates = 1D weight+bias without running stats
modules = {}

for key in state.keys():
    if "." in key:
        module, param = key.rsplit(".", 1)
        modules.setdefault(module, set()).add(param)

for module, params in modules.items():

    if {"weight", "bias"}.issubset(params):

        weight = state.get(module + ".weight")

        if (
            torch.is_tensor(weight)
            and weight.ndim == 1
            and module not in bn_modules
        ):
            ln_candidates.append(module)

print("\n" + "-" * 100)
print("LINEAR LAYERS")
print("-" * 100)

for key, in_dim, out_dim in linear_layers:
    print(
        f"{key:<60} "
        f"{in_dim} -> {out_dim}"
    )

if not linear_layers:
    print("[NONE FOUND]")

print("\n" + "-" * 100)
print("BATCH NORMALIZATION")
print("-" * 100)

if bn_modules:
    for x in bn_modules:
        print("[FOUND]", x)
else:
    print("[NONE FOUND]")

print("\n" + "-" * 100)
print("LAYERNORM CANDIDATES")
print("-" * 100)

if ln_candidates:
    for x in ln_candidates:
        print("[FOUND]", x)
else:
    print("[NONE FOUND]")

# ----------------------------------------------------------------------------------------------------
# 5. ROLE CLASSIFICATION
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. ENVIRONMENTAL ROLE CLASSIFICATION")
print("=" * 100)

role = "UNRESOLVED"
confidence = "LOW"

if linear_layers:

    first_in = linear_layers[0][1]
    last_out = linear_layers[-1][2]

    # Strong binary expert signature
    if last_out == 1:
        role = "ENVIRONMENTAL_EXPERT"
        confidence = "VERY HIGH"

    # Compression/alignment signature
    elif first_in > last_out:
        role = "ENVIRONMENTAL_ALIGNMENT_OR_COMPRESSOR"
        confidence = "HIGH"

    # Representation expansion / encoder signature
    elif first_in < last_out:
        role = "ENVIRONMENTAL_ENCODER"
        confidence = "MEDIUM"

print(f"Detected role : {role}")
print(f"Confidence    : {confidence}")

if role == "ENVIRONMENTAL_EXPERT":
    print(
        "[REASON] Final parameterized layer produces one output, "
        "consistent with the ASD expert."
    )

elif role == "ENVIRONMENTAL_ALIGNMENT_OR_COMPRESSOR":
    print(
        "[REASON] Network reduces dimensionality; "
        "alignment/compression is structurally plausible."
    )

elif role == "ENVIRONMENTAL_ENCODER":
    print(
        "[REASON] Network increases representation dimensionality."
    )

else:
    print(
        "[WARNING] Role cannot be safely determined from this checkpoint alone."
    )

# ----------------------------------------------------------------------------------------------------
# 6. PREPROCESSING ARTIFACT SUMMARY
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. ENVIRONMENTAL PREPROCESSING ARTIFACTS")
print("=" * 100)

preprocess_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for f in files:

        if not f.lower().endswith(
            (".joblib", ".pkl", ".pickle")
        ):
            continue

        full = os.path.join(root, f)
        low = full.lower()

        if any(
            token in low
            for token in [
                "environment",
                "environmental",
                "env_"
            ]
        ):
            preprocess_files.append(full)

preprocess_files = sorted(set(preprocess_files))

print(f"Environmental preprocessing artifacts: {len(preprocess_files)}")

for p in preprocess_files:
    print(" ", p)

# ----------------------------------------------------------------------------------------------------
# 7. FINAL DECISION
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("M4.4.10.3 FINAL DECISION")
print("=" * 100)

print(f"Checkpoint : {env_pt[0]}")
print(f"Role       : {role}")
print(f"Confidence : {confidence}")

print("\n" + "=" * 100)
print("M4.4.10.3 COMPLETED")
print("=" * 100)

print("[PASSED] Environmental checkpoint located.")
print("[PASSED] Exact checkpoint keys inspected.")
print("[PASSED] Exact tensor shapes inspected.")
print("[PASSED] Linear structures identified.")
print("[PASSED] Normalization structures identified.")
print("[PASSED] Environmental role classified where structurally proven.")

print("\nNO TRAINING PERFORMED.")
print("NO CHECKPOINT MODIFIED.")
print("NO PRODUCTION ARTIFACT OVERWRITTEN.")

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)

if role == "ENVIRONMENTAL_EXPERT":
    print("""
The environmental expert is structurally proven.

NEXT:
M4.4.10.4 — exact environmental expert reconstruction
             + preprocessing validation
             + P001-P005 environmental probabilities.

Then immediately:
M4.4.11 — MULTIMODAL FUSION INSPECTION
""")

elif role == "ENVIRONMENTAL_ALIGNMENT_OR_COMPRESSOR":
    print("""
The checkpoint is a dimensionality-reduction network.

NEXT:
Inspect the remaining environmental artifacts/checkpoints for
the missing encoder or expert before reconstruction.

No architecture will be invented.
""")

elif role == "ENVIRONMENTAL_ENCODER":
    print("""
The checkpoint is an encoder candidate.

NEXT:
Inspect the remaining environmental artifacts for alignment
and expert components, then reconstruct the complete branch.
""")

else:
    print("""
The role is not structurally proven.

NEXT:
Use the environmental source/reference implementation to resolve
the role before reconstruction.
""")

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.3
EXACT ENVIRONMENTAL ROLE CLASSIFICATION
[PASSED] Project directory available.

1. ENVIRONMENTAL CHECKPOINTS
[FOUND] 3 environmental checkpoint(s)
[1] /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt
[2] /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/03_representation_learning/environmental_encoder_128d.pt
[3] /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/04_cross_modal_alignment/environmental_128_to_32_alignment.pt

2. EXACT CHECKPOINT STRUCTURE
[FOUND] checkpoint appears to be state_dict
State-dict entries: 11

----------------------------------------------------------------------------------------------------
EXACT KEYS + SHAPES
----------------------------------------------------------------------------------------------------
network.0.weight                                                       (64, 32)
network.0.bias                                          

In [ ]:
# ====================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.4
# EXACT ENVIRONMENTAL EXPERT RECONSTRUCTION + P001-P005 PRODUCTION INFERENCE
# ====================================================================================================
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# NO ARTIFACT OVERWRITE
# ====================================================================================================

import os
import glob
import joblib
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

DEVICE = torch.device("cpu")

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.4")
print("EXACT ENVIRONMENTAL EXPERT RECONSTRUCTION + P001-P005 INFERENCE")
print("=" * 100)

# ----------------------------------------------------------------------------------------------------
# 1. PROJECT CHECK
# ----------------------------------------------------------------------------------------------------

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory missing:\n{PROJECT_DIR}"
    )

print("[PASSED] Project directory available.")

# ----------------------------------------------------------------------------------------------------
# 2. FIND ENVIRONMENTAL EXPERT CHECKPOINT
# ----------------------------------------------------------------------------------------------------

expert_candidates = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for f in files:
        low = f.lower()

        if not low.endswith((".pt", ".pth", ".ckpt")):
            continue

        full = os.path.join(root, f)

        # Prefer explicit expert naming
        if (
            "environment" in low
            or "environmental" in low
            or "env_" in low
        ) and "expert" in low:
            expert_candidates.append(full)

# If exact expert filename wasn't obvious, use checkpoint classification
if not expert_candidates:

    for root, dirs, files in os.walk(PROJECT_DIR):
        for f in files:

            if not f.lower().endswith((".pt", ".pth", ".ckpt")):
                continue

            full = os.path.join(root, f)

            try:
                ckpt = torch.load(
                    full,
                    map_location="cpu",
                    weights_only=True
                )

                if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
                    state = ckpt["model_state_dict"]
                else:
                    state = ckpt

                if not isinstance(state, dict):
                    continue

                # Strong expert signature: final 2-D weight has output dimension 1
                linear_weights = []

                for k, v in state.items():
                    if torch.is_tensor(v) and k.endswith(".weight") and v.ndim == 2:
                        linear_weights.append((k, tuple(v.shape)))

                if linear_weights:
                    last_key, last_shape = linear_weights[-1]

                    if last_shape[0] == 1:
                        if any(
                            x in full.lower()
                            for x in ["environment", "environmental", "env_"]
                        ):
                            expert_candidates.append(full)

            except Exception:
                pass

expert_candidates = sorted(set(expert_candidates))

print("\n" + "=" * 100)
print("2. ENVIRONMENTAL EXPERT CHECKPOINT")
print("=" * 100)

if not expert_candidates:
    raise FileNotFoundError(
        "Environmental expert checkpoint could not be resolved."
    )

for p in expert_candidates:
    print("[FOUND]", p)

EXPERT_PATH = expert_candidates[0]

print(f"\n[SELECTED] {EXPERT_PATH}")

# ----------------------------------------------------------------------------------------------------
# 3. LOAD CHECKPOINT
# ----------------------------------------------------------------------------------------------------

checkpoint = torch.load(
    EXPERT_PATH,
    map_location="cpu",
    weights_only=True
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    expert_state = checkpoint["model_state_dict"]
else:
    expert_state = checkpoint

print("\n" + "=" * 100)
print("3. EXACT CHECKPOINT VALIDATION")
print("=" * 100)

print(f"Checkpoint entries: {len(expert_state)}")

for key, value in expert_state.items():
    if torch.is_tensor(value):
        print(f"{key:<65} {tuple(value.shape)}")

# ----------------------------------------------------------------------------------------------------
# 4. INFER EXACT NETWORK STRUCTURE FROM STATE-DICT
# ----------------------------------------------------------------------------------------------------

linear_layers = []

for key, value in expert_state.items():

    if (
        torch.is_tensor(value)
        and key.endswith(".weight")
        and value.ndim == 2
    ):
        out_dim, in_dim = tuple(value.shape)

        linear_layers.append({
            "key": key,
            "in": in_dim,
            "out": out_dim
        })

if len(linear_layers) < 2:
    raise RuntimeError(
        "Not enough Linear layers to reconstruct environmental expert."
    )

print("\n" + "=" * 100)
print("4. EXACT LINEAR STRUCTURE")
print("=" * 100)

for layer in linear_layers:
    print(
        f"{layer['key']:<60} "
        f"{layer['in']} -> {layer['out']}"
    )

input_dim = linear_layers[0]["in"]
hidden_dims = [x["out"] for x in linear_layers[:-1]]
output_dim = linear_layers[-1]["out"]

print(f"\nInput dimension : {input_dim}")
print(f"Hidden sizes    : {hidden_dims}")
print(f"Output dimension: {output_dim}")

if output_dim != 1:
    raise RuntimeError(
        f"Environmental checkpoint does not end in one output. "
        f"Detected output dimension={output_dim}"
    )

# ----------------------------------------------------------------------------------------------------
# 5. DETECT NORMALIZATION MODULES
# ----------------------------------------------------------------------------------------------------

bn_modules = set()

for key in expert_state.keys():

    if (
        "running_mean" in key
        or "running_var" in key
    ):
        bn_modules.add(
            key.rsplit(".", 1)[0]
        )

print("\n" + "=" * 100)
print("5. NORMALIZATION STRUCTURE")
print("=" * 100)

if bn_modules:
    for module in sorted(bn_modules):
        print("[FOUND] BatchNorm:", module)
else:
    print("[INFO] No BatchNorm running-statistics found.")

# ----------------------------------------------------------------------------------------------------
# 6. RECONSTRUCT USING PROVEN STATE-DICT INDEX PATTERN
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. RECONSTRUCTING EXACT ENVIRONMENTAL EXPERT")
print("=" * 100)

# The known environmental expert structure is inferred from its actual
# parameterized network indices. We preserve those exact indices.

network_indices = sorted(
    int(k.split(".")[1])
    for k in expert_state.keys()
    if k.startswith("network.")
    and len(k.split(".")) >= 3
    and k.split(".")[1].isdigit()
)

print("Detected network indices:", network_indices)

layers = []

# Build modules at their exact checkpoint indices.
# Parameterized indices determine where Linear/BN occur.
max_index = max(network_indices)

# Identify linear module indices
linear_by_index = {}

for layer in linear_layers:
    parts = layer["key"].split(".")
    idx = int(parts[1])
    linear_by_index[idx] = layer

# Identify BN module indices
bn_by_index = {}

for module in bn_modules:
    parts = module.split(".")
    if len(parts) >= 2 and parts[1].isdigit():
        bn_by_index[int(parts[1])] = module

# Determine expected sizes from checkpoint
for idx in range(max_index + 1):

    if idx in linear_by_index:

        layer = linear_by_index[idx]

        layers.append(
            nn.Linear(
                layer["in"],
                layer["out"]
            )
        )

    elif idx in bn_by_index:

        # Exact BN feature count from checkpoint weight
        module_name = bn_by_index[idx]
        weight_key = module_name + ".weight"

        features = int(
            expert_state[weight_key].shape[0]
        )

        layers.append(
            nn.BatchNorm1d(features)
        )

    else:
        # Parameter-free layers must be inferred from the known
        # sequential expert pattern. We preserve the parameterized
        # checkpoint indices and use standard activation/dropout
        # placement between them.
        #
        # For a final binary expert, activation is not inserted after
        # the final Linear layer.

        previous_linear = None

        # Find previous parameterized linear
        previous_linear_indices = [
            x for x in linear_by_index.keys()
            if x < idx
        ]

        if previous_linear_indices:
            previous_linear = max(previous_linear_indices)

        # Reconstruct activation/dropout gaps only when they occur
        # between known parameterized layers.
        #
        # Standard structure used by the discovered expert:
        # Linear -> BN -> ReLU -> Dropout -> Linear -> ReLU -> Dropout -> Linear
        #
        # The exact checkpoint keys themselves remain authoritative.

        if idx > 0:
            pass

# ----------------------------------------------------------------------------------------------------
# 7. BUILD THE KNOWN EXPERT PATTERN
# ----------------------------------------------------------------------------------------------------

# Based on the actual checkpoint structure, construct the expert with
# parameterized layers at the exact network indices.

modules = []

for idx in range(max_index + 1):

    if idx in linear_by_index:

        layer = linear_by_index[idx]

        modules.append(
            nn.Linear(
                layer["in"],
                layer["out"]
            )
        )

    elif idx in bn_by_index:

        module_name = bn_by_index[idx]

        features = int(
            expert_state[
                module_name + ".weight"
            ].shape[0]
        )

        modules.append(
            nn.BatchNorm1d(features)
        )

    else:

        # Parameter-free layers between parameterized layers.
        # Use the standard expert sequence inferred from the checkpoint
        # index gaps.
        #
        # Do NOT put an activation/dropout after the final output layer.

        if idx < max_index:

            # Determine whether the next parameterized layer exists.
            future_linear = [
                x for x in linear_by_index.keys()
                if x > idx
            ]

            if future_linear:

                # First gap after Linear/BN:
                # activation
                #
                # Additional gap:
                # dropout
                #
                # This preserves the discovered 0,1,4,7 style structure
                # when present.

                if idx == 2:
                    modules.append(nn.ReLU())

                elif idx == 3:
                    modules.append(nn.Dropout())

                elif idx == 5:
                    modules.append(nn.ReLU())

                elif idx == 6:
                    modules.append(nn.Dropout())

                else:
                    # Parameter-free index with no proven role.
                    # Do not silently invent a layer.
                    raise RuntimeError(
                        f"Unresolved parameter-free network index: network.{idx}"
                    )

expert = nn.Sequential(*modules).to(DEVICE)

# ----------------------------------------------------------------------------------------------------
# 8. STRICT LOAD
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. STRICT CHECKPOINT LOADING")
print("=" * 100)

try:
    expert.load_state_dict(
        expert_state,
        strict=True
    )
except RuntimeError as e:

    print("[ERROR] Strict reconstruction failed.")
    print(e)

    raise RuntimeError(
        "Environmental expert architecture does not match the "
        "checkpoint exactly. STOPPING instead of guessing."
    )

expert.eval()

print("[PASSED] Environmental expert loaded with strict=True.")
print("[PASSED] Environmental expert set to eval().")

# ----------------------------------------------------------------------------------------------------
# 9. STATE-DICT EXACTNESS
# ----------------------------------------------------------------------------------------------------

model_keys = list(expert.state_dict().keys())
checkpoint_keys = list(expert_state.keys())

if model_keys != checkpoint_keys:

    missing = [
        k for k in checkpoint_keys
        if k not in model_keys
    ]

    unexpected = [
        k for k in model_keys
        if k not in checkpoint_keys
    ]

    raise RuntimeError(
        f"State-dict mismatch.\n"
        f"Missing: {missing}\n"
        f"Unexpected: {unexpected}"
    )

print("[PASSED] State-dict keys exactly match checkpoint.")

# ----------------------------------------------------------------------------------------------------
# 10. SHAPE VALIDATION
# ----------------------------------------------------------------------------------------------------

for key in checkpoint_keys:

    if tuple(expert.state_dict()[key].shape) != tuple(
        expert_state[key].shape
    ):
        raise RuntimeError(
            f"Shape mismatch: {key}"
        )

print("[PASSED] All parameter/buffer shapes exactly match.")

# ----------------------------------------------------------------------------------------------------
# 11. FIND ENVIRONMENTAL PREPROCESSORS
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. ENVIRONMENTAL PREPROCESSING")
print("=" * 100)

preprocessor_files = []

for root, dirs, files in os.walk(PROJECT_DIR):

    for f in files:

        if not f.lower().endswith(
            (".joblib", ".pkl", ".pickle")
        ):
            continue

        full = os.path.join(root, f)
        low = full.lower()

        if any(
            token in low
            for token in [
                "environment",
                "environmental",
                "env_"
            ]
        ):
            preprocessor_files.append(full)

preprocessor_files = sorted(set(preprocessor_files))

print(
    f"Environmental preprocessing candidates: "
    f"{len(preprocessor_files)}"
)

# Load objects and select those capable of transform()
transformers = []

for path in preprocessor_files:

    try:
        obj = joblib.load(path)

        if hasattr(obj, "transform"):
            transformers.append(
                (path, obj)
            )

            print(
                f"[FOUND] Transform-capable: "
                f"{path} | {type(obj)}"
            )

    except Exception as e:
        print(
            f"[WARNING] Could not load {path}: {e}"
        )

# ----------------------------------------------------------------------------------------------------
# 12. FIND ENVIRONMENTAL DEMO INPUTS
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("9. ENVIRONMENTAL DEMO INPUTS")
print("=" * 100)

patient_files = {}

for pid in [
    "P001",
    "P002",
    "P003",
    "P004",
    "P005"
]:

    patient_dir = os.path.join(
        DEMO_DIR,
        pid
    )

    candidates = []

    if os.path.isdir(patient_dir):

        for root, dirs, files in os.walk(patient_dir):

            for f in files:

                low = f.lower()

                if not low.endswith(
                    (".csv", ".xlsx", ".xls", ".json")
                ):
                    continue

                if (
                    "environment" in low
                    or "environmental" in low
                    or low.startswith("env")
                ):
                    candidates.append(
                        os.path.join(root, f)
                    )

    if not candidates:
        print(f"[WARNING] {pid}: environmental file not found")
        continue

    patient_files[pid] = candidates[0]

    print(
        f"[FOUND] {pid}: "
        f"{candidates[0]}"
    )

if len(patient_files) != 5:

    raise RuntimeError(
        f"Expected 5 environmental demo files, "
        f"resolved {len(patient_files)}."
    )

# ----------------------------------------------------------------------------------------------------
# 13. LOAD INPUT
# ----------------------------------------------------------------------------------------------------

def load_table(path):

    low = path.lower()

    if low.endswith(".csv"):
        return pd.read_csv(path)

    if low.endswith((".xlsx", ".xls")):
        return pd.read_excel(path)

    if low.endswith(".json"):
        return pd.read_json(path)

    raise ValueError(
        f"Unsupported input: {path}"
    )

# ----------------------------------------------------------------------------------------------------
# 14. PRODUCTION INFERENCE
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("10. ENVIRONMENTAL P001-P005 PRODUCTION INFERENCE")
print("=" * 100)

results = []

for pid in [
    "P001",
    "P002",
    "P003",
    "P004",
    "P005"
]:

    print("\n" + "-" * 100)
    print(pid)
    print("-" * 100)

    df = load_table(
        patient_files[pid]
    )

    print(
        f"Raw input shape: {df.shape}"
    )

    # Remove obvious ID/target columns only if present.
    # Target is never used for inference.
    drop_cols = []

    for col in df.columns:

        low = str(col).lower()

        if low in [
            "id",
            "patient_id",
            "patient",
            "label",
            "target",
            "class",
            "class/asd"
        ]:
            drop_cols.append(col)

    raw_df = df.drop(
        columns=drop_cols,
        errors="ignore"
    )

    numeric_df = raw_df.select_dtypes(
        include=[np.number]
    )

    print(
        f"Numeric input shape: "
        f"{numeric_df.shape}"
    )

    # -----------------------------------------------------------------------------------------------
    # PREPROCESSING
    # -----------------------------------------------------------------------------------------------

    X = numeric_df.copy()

    transformed = None

    # Try saved environmental preprocessors first.
    for path, prep in transformers:

        try:

            candidate = prep.transform(X)

            candidate = np.asarray(
                candidate,
                dtype=np.float32
            )

            if candidate.ndim == 1:
                candidate = candidate.reshape(
                    1, -1
                )

            # Only accept a transformed representation compatible
            # with the expert's expected input dimension.
            if candidate.shape[1] == input_dim:

                transformed = candidate

                print(
                    f"[PASSED] Preprocessor selected: {path}"
                )

                break

        except Exception:
            continue

    # If preprocessing already produced the expert input dimension,
    # use the numeric data directly.
    if transformed is None:

        direct = numeric_df.to_numpy(
            dtype=np.float32
        )

        if direct.shape[1] == input_dim:

            transformed = direct

            print(
                "[INFO] Numeric input already matches "
                f"expert input dimension {input_dim}."
            )

    if transformed is None:

        raise RuntimeError(
            f"{pid}: Could not produce the environmental "
            f"expert's required input dimension {input_dim}. "
            f"Raw numeric dimensions={numeric_df.shape[1]}."
        )

    X_tensor = torch.tensor(
        transformed,
        dtype=torch.float32,
        device=DEVICE
    )

    print(
        f"Expert input shape: "
        f"{tuple(X_tensor.shape)}"
    )

    # -----------------------------------------------------------------------------------------------
    # EXPERT
    # -----------------------------------------------------------------------------------------------

    with torch.no_grad():

        logits = expert(
            X_tensor
        )

        probabilities = torch.sigmoid(
            logits
        ).cpu().numpy().reshape(-1)

    probability = float(
        probabilities[0]
    )

    prediction = (
        "ASD"
        if probability >= 0.5
        else "NON-ASD"
    )

    print(
        f"ASD probability : {probability:.6f}"
    )

    print(
        f"Prediction      : {prediction}"
    )

    if not np.isfinite(probability):
        raise RuntimeError(
            f"{pid}: invalid probability."
        )

    results.append({
        "patient_id": pid,
        "environmental_probability": probability,
        "environmental_prediction": prediction,
        "environmental_prediction_numeric": int(
            probability >= 0.5
        )
    })

# ----------------------------------------------------------------------------------------------------
# 15. RESULTS
# ----------------------------------------------------------------------------------------------------

results_df = pd.DataFrame(
    results
)

print("\n" + "=" * 100)
print("11. ENVIRONMENTAL RESULTS — P001-P005")
print("=" * 100)

print(
    results_df.to_string(
        index=False
    )
)

# ----------------------------------------------------------------------------------------------------
# 16. FINAL VALIDATION
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("M4.4.10.4 COMPLETED")
print("=" * 100)

print("[PASSED] Environmental expert checkpoint resolved.")
print("[PASSED] Exact checkpoint structure reconstructed.")
print("[PASSED] Strict checkpoint loading succeeded.")
print("[PASSED] State-dict keys exactly match.")
print("[PASSED] Parameter shapes exactly match.")
print("[PASSED] Environmental preprocessing validated.")
print("[PASSED] P001-P005 environmental inference completed.")
print("[PASSED] Environmental probabilities are finite and valid.")

print("\nNO TRAINING PERFORMED.")
print("NO CHECKPOINT MODIFIED.")
print("NO PRODUCTION ARTIFACT OVERWRITTEN.")

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)

print("""
M4.4.11 — MULTIMODAL FUSION INSPECTION

Verified branches now available:

BEHAVIORAL
RAW 18-D
   ↓
87-D
   ↓
128-D
   ↓
32-D
   ↓
ASD probability

GENETIC
1000-D
   ↓
128-D
   ↓
32-D
   ↓
ASD probability

ENVIRONMENTAL
saved preprocessing
   ↓
environmental expert
   ↓
ASD probability

NEXT:
Inspect the actual fusion checkpoint/source.
Then reconstruct the fusion layer.
Then run P001-P005 complete 3-modal inference.
Then build the final interface.
""")

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.4
EXACT ENVIRONMENTAL EXPERT RECONSTRUCTION + P001-P005 INFERENCE
[PASSED] Project directory available.

2. ENVIRONMENTAL EXPERT CHECKPOINT
[FOUND] /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt

[SELECTED] /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt

3. EXACT CHECKPOINT VALIDATION
Checkpoint entries: 11
network.0.weight                                                  (64, 32)
network.0.bias                                                    (64,)
network.1.weight                                                  (64,)
network.1.bias                                                    (64,)
network.1.running_mean                                            (64,)
network.1.running_var                                             (64,)
network.1.num_batches_tracked                                     ()
network.4.weight                                      

RuntimeError: Environmental expert architecture does not match the checkpoint exactly. STOPPING instead of guessing.

In [ ]:
# ==================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.4.1
# EXACT ENVIRONMENTAL EXPERT RECONSTRUCTION — NAMESPACE FIX
# ==================================================================================================

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

DEVICE = torch.device("cpu")

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
EXPERT_PATH = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M3_1_environmental_expert.pt"
)

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.4.1")
print("EXACT ENVIRONMENTAL EXPERT RECONSTRUCTION — NAMESPACE FIX")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. VERIFY CHECKPOINT
# --------------------------------------------------------------------------------------------------

if not os.path.isfile(EXPERT_PATH):
    raise FileNotFoundError(
        f"Environmental expert checkpoint missing:\n{EXPERT_PATH}"
    )

print("[FOUND] Environmental expert checkpoint")
print(f"        {EXPERT_PATH}")

expert_state = torch.load(
    EXPERT_PATH,
    map_location=DEVICE,
    weights_only=True
)

if not isinstance(expert_state, dict):
    raise TypeError("Environmental expert checkpoint is not a state_dict.")

print(f"[PASSED] Checkpoint loaded.")
print(f"State-dict entries: {len(expert_state)}")

# --------------------------------------------------------------------------------------------------
# 2. VERIFY EXACT KEYS
# --------------------------------------------------------------------------------------------------

expected_keys = [
    "network.0.weight",
    "network.0.bias",
    "network.1.weight",
    "network.1.bias",
    "network.1.running_mean",
    "network.1.running_var",
    "network.1.num_batches_tracked",
    "network.4.weight",
    "network.4.bias",
    "network.7.weight",
    "network.7.bias",
]

actual_keys = list(expert_state.keys())

if set(actual_keys) != set(expected_keys):
    print("[ERROR] Checkpoint keys differ from expected keys.")
    print("Actual:")
    for k in actual_keys:
        print(" ", k)

    raise RuntimeError(
        "Environmental expert checkpoint keys are not the previously "
        "verified 11-key structure."
    )

print("[PASSED] Exact 11 checkpoint keys confirmed.")

# --------------------------------------------------------------------------------------------------
# 3. EXACT PARAMETER SHAPES
# --------------------------------------------------------------------------------------------------

expected_shapes = {
    "network.0.weight": (64, 32),
    "network.0.bias": (64,),
    "network.1.weight": (64,),
    "network.1.bias": (64,),
    "network.1.running_mean": (64,),
    "network.1.running_var": (64,),
    "network.1.num_batches_tracked": (),
    "network.4.weight": (32, 64),
    "network.4.bias": (32,),
    "network.7.weight": (1, 32),
    "network.7.bias": (1,),
}

for key, expected_shape in expected_shapes.items():
    actual_shape = tuple(expert_state[key].shape)

    if actual_shape != expected_shape:
        raise RuntimeError(
            f"Shape mismatch for {key}: "
            f"expected {expected_shape}, got {actual_shape}"
        )

    print(f"[PASSED] {key:<45} {actual_shape}")

# --------------------------------------------------------------------------------------------------
# 4. EXACT NAMESPACE-MATCHED ARCHITECTURE
# --------------------------------------------------------------------------------------------------

class EnvironmentalExpert(nn.Module):
    def __init__(self):
        super().__init__()

        # IMPORTANT:
        # The checkpoint uses "network.*" keys.
        # Therefore the Sequential MUST be stored as self.network.

        self.network = nn.Sequential(
            nn.Linear(32, 64),       # network.0
            nn.BatchNorm1d(64),      # network.1
            nn.ReLU(),               # network.2
            nn.Dropout(),            # network.3
            nn.Linear(64, 32),       # network.4
            nn.ReLU(),              # network.5
            nn.Dropout(),            # network.6
            nn.Linear(32, 1)         # network.7
        )

    def forward(self, x):
        return self.network(x)


environmental_expert = EnvironmentalExpert().to(DEVICE)

print("=" * 100)
print("5. EXACT ENVIRONMENTAL EXPERT CREATED")
print("=" * 100)

print("[PASSED] EnvironmentalExpert created.")
print("[PASSED] self.network namespace matches checkpoint.")
print("[PASSED] network.0 = Linear(32 → 64)")
print("[PASSED] network.1 = BatchNorm1d(64)")
print("[PASSED] network.4 = Linear(64 → 32)")
print("[PASSED] network.7 = Linear(32 → 1)")

# --------------------------------------------------------------------------------------------------
# 6. STRICT CHECKPOINT LOADING
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("6. STRICT CHECKPOINT LOADING")
print("=" * 100)

environmental_expert.load_state_dict(
    expert_state,
    strict=True
)

environmental_expert.eval()

print("[PASSED] Environmental expert loaded with strict=True.")
print("[PASSED] No missing keys.")
print("[PASSED] No unexpected keys.")
print("[PASSED] Environmental expert set to eval().")

# --------------------------------------------------------------------------------------------------
# 7. STATE-DICT EXACTNESS
# --------------------------------------------------------------------------------------------------

reconstructed_keys = list(environmental_expert.state_dict().keys())

if reconstructed_keys != actual_keys:
    raise RuntimeError(
        "Reconstructed state-dict key order/content does not exactly match checkpoint."
    )

print("[PASSED] Reconstructed state-dict keys exactly match checkpoint.")

# --------------------------------------------------------------------------------------------------
# 8. FORWARD VALIDATION
# --------------------------------------------------------------------------------------------------

test_input = torch.randn(2, 32, device=DEVICE)

with torch.no_grad():
    test_logits = environmental_expert(test_input)

print("=" * 100)
print("7. ENVIRONMENTAL EXPERT FORWARD VALIDATION")
print("=" * 100)

print(f"Input shape  : {tuple(test_input.shape)}")
print(f"Output shape : {tuple(test_logits.shape)}")

if tuple(test_logits.shape) != (2, 1):
    raise RuntimeError(
        f"Unexpected expert output shape: {tuple(test_logits.shape)}"
    )

if not torch.isfinite(test_logits).all():
    raise RuntimeError("Environmental expert produced non-finite logits.")

test_probabilities = torch.sigmoid(test_logits)

if not torch.isfinite(test_probabilities).all():
    raise RuntimeError("Environmental expert produced non-finite probabilities.")

if not ((test_probabilities >= 0).all() and
        (test_probabilities <= 1).all()):
    raise RuntimeError("Environmental probabilities outside [0,1].")

print("[PASSED] 32-D input accepted.")
print("[PASSED] One output logit produced per sample.")
print("[PASSED] Sigmoid probabilities valid.")
print()
print("Test probabilities:")
print(test_probabilities.cpu().numpy().reshape(-1))

# --------------------------------------------------------------------------------------------------
# 9. FINAL ARCHITECTURE SUMMARY
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("8. EXACT ENVIRONMENTAL EXPERT ARCHITECTURE")
print("=" * 100)

print("""
32
 ↓
Linear(32 → 64)
 ↓
BatchNorm1d(64)
 ↓
ReLU
 ↓
Dropout
 ↓
Linear(64 → 32)
 ↓
ReLU
 ↓
Dropout
 ↓
Linear(32 → 1)
 ↓
Sigmoid
 ↓
Environmental ASD probability
""")

print("=" * 100)
print("M4.4.10.4.1 COMPLETED")
print("=" * 100)

print("[PASSED] Exact checkpoint namespace reconstructed.")
print("[PASSED] Exact 11 checkpoint keys loaded with strict=True.")
print("[PASSED] Parameter shapes exactly match.")
print("[PASSED] 32-D input accepted.")
print("[PASSED] One ASD logit produced.")
print("[PASSED] Probability validation passed.")
print()
print("NO TRAINING PERFORMED.")
print("NO CHECKPOINT MODIFIED.")
print("NO PRODUCTION ARTIFACT OVERWRITTEN.")
print()
print("NEXT:")
print("M4.4.10.5 — ENVIRONMENTAL PREPROCESSING + P001-P005 INFERENCE")
print("Then immediately:")
print("M4.4.11 — MULTIMODAL FUSION INSPECTION")
print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.4.1
EXACT ENVIRONMENTAL EXPERT RECONSTRUCTION — NAMESPACE FIX
[FOUND] Environmental expert checkpoint
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt
[PASSED] Checkpoint loaded.
State-dict entries: 11
[PASSED] Exact 11 checkpoint keys confirmed.
[PASSED] network.0.weight                              (64, 32)
[PASSED] network.0.bias                                (64,)
[PASSED] network.1.weight                              (64,)
[PASSED] network.1.bias                                (64,)
[PASSED] network.1.running_mean                        (64,)
[PASSED] network.1.running_var                         (64,)
[PASSED] network.1.num_batches_tracked                 ()
[PASSED] network.4.weight                              (32, 64)
[PASSED] network.4.bias                                (32,)
[PASSED] network.7.weight                              (1, 32)
[PASSED] network.7.bias                          

In [ ]:
# ==================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.5A
# EXACT ENVIRONMENTAL PREPROCESSING INSPECTION
# NO MODEL RECONSTRUCTION — NO TRAINING
# ==================================================================================================

import os
import glob
import joblib
import json
import pickle
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.5A")
print("EXACT ENVIRONMENTAL PREPROCESSING INSPECTION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. VERIFY DRIVE / PROJECT
# --------------------------------------------------------------------------------------------------

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}"
    )

if not os.path.isdir(DEMO_DIR):
    raise FileNotFoundError(
        f"Demo directory not found:\n{DEMO_DIR}"
    )

print("[PASSED] Project directory available.")
print("[PASSED] Environmental demo directory available.")

# --------------------------------------------------------------------------------------------------
# 2. DISCOVER ENVIRONMENTAL PREPROCESSING ARTIFACTS
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("2. DISCOVERING ENVIRONMENTAL PREPROCESSING ARTIFACTS")
print("=" * 100)

candidate_extensions = [
    "*.joblib",
    "*.pkl",
    "*.pickle",
    "*.json",
]

all_candidates = []

for ext in candidate_extensions:
    all_candidates.extend(
        glob.glob(
            os.path.join(PROJECT_DIR, "**", ext),
            recursive=True
        )
    )

# Remove duplicates
all_candidates = sorted(set(all_candidates))

environmental_candidates = []

environmental_keywords = [
    "environment",
    "environmental",
    "env_",
    "_env",
    "environment_model",
    "m3_1_environmental",
]

for path in all_candidates:
    lower = path.lower()

    if any(keyword in lower for keyword in environmental_keywords):
        environmental_candidates.append(path)

print(f"[INFO] Total preprocessing-like artifacts discovered: {len(all_candidates)}")
print(
    f"[INFO] Environmental preprocessing candidates discovered: "
    f"{len(environmental_candidates)}"
)

for i, path in enumerate(environmental_candidates, 1):
    print(f"\n[{i}] {path}")

if not environmental_candidates:
    print()
    print("[WARNING] No environmental preprocessing artifact matched by filename.")
    print("The demo inputs and source directories will be inspected before deciding.")

# --------------------------------------------------------------------------------------------------
# 3. INSPECT JOBLIB / PICKLE ARTIFACTS
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("3. INSPECTING SERIALIZED PREPROCESSING OBJECTS")
print("=" * 100)

loaded_objects = {}

for path in environmental_candidates:

    print()
    print("-" * 100)
    print("FILE:", path)
    print("-" * 100)

    ext = os.path.splitext(path)[1].lower()

    try:

        if ext in [".joblib", ".pkl", ".pickle"]:

            obj = joblib.load(path)
            loaded_objects[path] = obj

            print("Object type:", type(obj))

            # sklearn-style metadata
            for attr in [
                "n_features_in_",
                "feature_names_in_",
                "mean_",
                "scale_",
                "categories_",
                "transformers_",
                "transformers",
                "steps",
                "n_components",
            ]:
                if hasattr(obj, attr):
                    try:
                        value = getattr(obj, attr)

                        if isinstance(value, np.ndarray):
                            print(
                                f"{attr}: ndarray shape={value.shape}"
                            )

                        elif isinstance(value, (list, tuple)):
                            print(
                                f"{attr}: "
                                f"type={type(value).__name__} "
                                f"len={len(value)}"
                            )

                            if len(value) <= 20:
                                print(" ", value)

                        else:
                            print(
                                f"{attr}: {value}"
                            )

                    except Exception as e:
                        print(
                            f"{attr}: <inspection failed: {e}>"
                        )

            # Pipeline structure
            if hasattr(obj, "named_steps"):

                print()
                print("[FOUND] sklearn Pipeline")

                for name, step in obj.named_steps.items():
                    print(
                        f"  step={name:<25} "
                        f"type={type(step)}"
                    )

            # ColumnTransformer structure
            if hasattr(obj, "transformers_"):

                print()
                print("[FOUND] ColumnTransformer-like structure")

                try:
                    for item in obj.transformers_:
                        print(" ", item[0], type(item[1]))
                except Exception:
                    pass

            # Generic dictionary
            if isinstance(obj, dict):
                print()
                print("[FOUND] Dictionary object")
                print("Keys:")

                for key in obj.keys():
                    print(" ", key)

        elif ext == ".json":

            with open(path, "r") as f:
                obj = json.load(f)

            loaded_objects[path] = obj

            print("JSON object type:", type(obj))

            if isinstance(obj, dict):
                print("Keys:")
                for key in obj.keys():
                    print(" ", key)

    except Exception as e:

        print(
            f"[WARNING] Could not load/inspect artifact:\n{e}"
        )

# --------------------------------------------------------------------------------------------------
# 4. INSPECT ENVIRONMENTAL DEMO INPUTS
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("4. INSPECTING P001-P005 ENVIRONMENTAL INPUTS")
print("=" * 100)

patients = ["P001", "P002", "P003", "P004", "P005"]

resolved_inputs = {}

for patient in patients:

    patient_dir = os.path.join(DEMO_DIR, patient)

    if not os.path.isdir(patient_dir):
        print(f"[MISSING] {patient}: patient directory not found")
        continue

    candidates = []

    for ext in ["*.csv", "*.xlsx", "*.xls", "*.json"]:
        candidates.extend(
            glob.glob(
                os.path.join(patient_dir, ext)
            )
        )

    environmental_files = []

    for path in candidates:

        lower = os.path.basename(path).lower()

        if (
            "environment" in lower
            or "environmental" in lower
            or "env" in lower
        ):
            environmental_files.append(path)

    print()
    print(f"--- {patient} ---")

    if not environmental_files:
        print("[WARNING] No explicitly named environmental file found.")

        print("Available files:")
        for path in candidates:
            print(" ", path)

        continue

    selected = environmental_files[0]
    resolved_inputs[patient] = selected

    print("[FOUND]", selected)

    try:

        if selected.lower().endswith(".csv"):

            df = pd.read_csv(selected)

        elif selected.lower().endswith((".xlsx", ".xls")):

            df = pd.read_excel(selected)

        elif selected.lower().endswith(".json"):

            df = pd.read_json(selected)

        else:
            continue

        print("Shape:", df.shape)
        print("Columns:", len(df.columns))

        print()
        print("Column names:")

        for col in df.columns:
            print(
                f"  {col:<45} dtype={df[col].dtype}"
            )

        numeric_cols = df.select_dtypes(
            include=[np.number]
        ).columns.tolist()

        print()
        print("Numeric feature count:", len(numeric_cols))

    except Exception as e:

        print(
            f"[WARNING] Could not inspect {selected}: {e}"
        )

# --------------------------------------------------------------------------------------------------
# 5. CROSS-CHECK INPUT DIMENSION AGAINST VERIFIED EXPERT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("5. ENVIRONMENTAL INPUT COMPATIBILITY")
print("=" * 100)

print("Verified environmental expert input dimension: 32")
print()
print("Important:")
print("The environmental expert does NOT necessarily consume raw environmental data.")
print("It consumes the 32-D representation produced by the environmental upstream branch.")
print()
print("Therefore we will NOT assume that raw demo feature count = 32.")
print("The preprocessing / representation artifacts must establish the transformation.")

# --------------------------------------------------------------------------------------------------
# 6. SEARCH FOR ENVIRONMENTAL SOURCE / MODEL REFERENCES
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("6. SEARCHING PROJECT SOURCE FOR ENVIRONMENTAL PIPELINE REFERENCES")
print("=" * 100)

source_extensions = [
    "*.py",
    "*.ipynb",
]

source_files = []

for ext in source_extensions:
    source_files.extend(
        glob.glob(
            os.path.join(PROJECT_DIR, "**", ext),
            recursive=True
        )
    )

source_files = sorted(set(source_files))

keywords = [
    "environmental",
    "environmental_expert",
    "environment",
    "env_scaler",
    "env_preprocessor",
    "environmental_preprocessor",
    "environmental_transformer",
]

matches = []

for path in source_files:

    try:

        if path.endswith(".py"):

            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:
                text = f.read()

        else:

            # Notebook inspection without requiring notebook execution
            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:
                text = f.read()

        lower = text.lower()

        matched_keywords = [
            k for k in keywords
            if k.lower() in lower
        ]

        if matched_keywords:

            matches.append(
                (path, matched_keywords)
            )

    except Exception:
        pass

print(
    f"[INFO] Source files containing environmental references: "
    f"{len(matches)}"
)

for path, matched_keywords in matches[:30]:

    print()
    print(path)
    print("Matched:", ", ".join(matched_keywords))

# --------------------------------------------------------------------------------------------------
# 7. FINAL DECISION
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("M4.4.10.5A SUMMARY")
print("=" * 100)

print(
    f"Environmental preprocessing candidates : "
    f"{len(environmental_candidates)}"
)

print(
    f"Environmental demo inputs resolved      : "
    f"{len(resolved_inputs)}/5"
)

print(
    f"Environmental source references found   : "
    f"{len(matches)}"
)

print()
print("[PASSED] No environmental checkpoint modified.")
print("[PASSED] No training performed.")
print("[PASSED] No environmental architecture guessed.")
print("[PASSED] Verified environmental expert preserved.")

print()
print("=" * 100)
print("NEXT")
print("=" * 100)

print("""
Use the inspection results above to identify the ACTUAL
environmental upstream representation pipeline.

Then:

ENVIRONMENTAL RAW INPUT
        ↓
EXACT SAVED PREPROCESSING
        ↓
EXACT ENVIRONMENTAL REPRESENTATION
        ↓
32-D REPRESENTATION
        ↓
VERIFIED ENVIRONMENTAL EXPERT
        ↓
1 ASD PROBABILITY

After that we move directly to:

M4.4.11 — MULTIMODAL FUSION INSPECTION
        ↓
M4.4.12 — FUSION RECONSTRUCTION
        ↓
P001-P005 THREE-MODAL INFERENCE
        ↓
ANY-ONE / ANY-TWO / ALL-THREE INPUT INTERFACE
""")

print("=" * 100)
print("M4.4.10.5A COMPLETED")
print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.5A
EXACT ENVIRONMENTAL PREPROCESSING INSPECTION
[PASSED] Project directory available.
[PASSED] Environmental demo directory available.

2. DISCOVERING ENVIRONMENTAL PREPROCESSING ARTIFACTS
[INFO] Total preprocessing-like artifacts discovered: 13
[INFO] Environmental preprocessing candidates discovered: 2

[1] /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_scaler.joblib

[2] /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/02_preprocessing/environmental_preprocessor.joblib

3. INSPECTING SERIALIZED PREPROCESSING OBJECTS

----------------------------------------------------------------------------------------------------
FILE: /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_scaler.joblib
----------------------------------------------------------------------------------------------------
Object type: <class 'sklearn.preprocessing._data.StandardScaler'>
n_featur

In [ ]:
# ================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.5B
# EXACT ENVIRONMENTAL PREPROCESSING COMPATIBILITY INSPECTION
# NO TRAINING — NO CHECKPOINT MODIFICATION — NO ARCHITECTURE GUESSING
# ================================================================================================

import os
import glob
import json
import joblib
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.5B")
print("EXACT ENVIRONMENTAL PREPROCESSING COMPATIBILITY INSPECTION")
print("=" * 100)

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}\n"
        "Reconnect Google Drive before continuing."
    )

print("[PASSED] Project directory available.")
print("Project :", PROJECT_DIR)

# ================================================================================================
# 1. RESOLVE ENVIRONMENTAL DEMO FILES
# ================================================================================================

print("\n" + "=" * 100)
print("1. RESOLVING ENVIRONMENTAL DEMO INPUTS")
print("=" * 100)

patients = ["P001", "P002", "P003", "P004", "P005"]

environmental_files = {}

for pid in patients:

    patient_dir = os.path.join(DEMO_DIR, pid)

    candidates = []

    if os.path.isdir(patient_dir):
        candidates = sorted(
            glob.glob(os.path.join(patient_dir, "*environment*.csv"))
            + glob.glob(os.path.join(patient_dir, "*Environment*.csv"))
            + glob.glob(os.path.join(patient_dir, "*ENV*.csv"))
        )

        # fallback: explicitly named environmental_input.csv
        exact_path = os.path.join(patient_dir, "environmental_input.csv")
        if os.path.isfile(exact_path):
            candidates.insert(0, exact_path)

    # remove duplicates
    candidates = list(dict.fromkeys(candidates))

    if candidates:
        environmental_files[pid] = candidates[0]
        print(f"[FOUND] {pid}  {candidates[0]}")
    else:
        print(f"[MISSING] {pid}")

if len(environmental_files) != 5:
    raise RuntimeError(
        f"Only {len(environmental_files)}/5 environmental demo inputs resolved."
    )

print("[PASSED] P001-P005 environmental files resolved.")

# ================================================================================================
# 2. INSPECT ACTUAL ENVIRONMENTAL INPUT STRUCTURE
# ================================================================================================

print("\n" + "=" * 100)
print("2. ACTUAL ENVIRONMENTAL INPUT STRUCTURE")
print("=" * 100)

environmental_data = {}

for pid, path in environmental_files.items():

    df = pd.read_csv(path)
    environmental_data[pid] = df

    print(f"\n--- {pid} ---")
    print("File       :", path)
    print("Shape      :", df.shape)
    print("Columns    :", len(df.columns))
    print("Numeric    :", len(df.select_dtypes(include=[np.number]).columns))
    print("Non-numeric:", len(df.select_dtypes(exclude=[np.number]).columns))

    print("Column names:")
    print(list(df.columns))

# ================================================================================================
# 3. DISCOVER PREPROCESSING CANDIDATES
# ================================================================================================

print("\n" + "=" * 100)
print("3. DISCOVERING ENVIRONMENTAL PREPROCESSING ARTIFACTS")
print("=" * 100)

all_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for filename in files:
        all_files.append(os.path.join(root, filename))

preprocessor_candidates = []

for path in all_files:

    lower = os.path.basename(path).lower()

    if not lower.endswith((
        ".joblib",
        ".pkl",
        ".pickle"
    )):
        continue

    # Environmental naming indicators
    if any(
        token in lower
        for token in [
            "environment",
            "environmental",
            "env_",
            "_env",
            "m3_1_environment"
        ]
    ):
        preprocessor_candidates.append(path)

# Deduplicate
preprocessor_candidates = sorted(set(preprocessor_candidates))

print("Environmental preprocessing candidates:",
      len(preprocessor_candidates))

for i, path in enumerate(preprocessor_candidates, 1):
    print(f"[{i}] {path}")

if len(preprocessor_candidates) == 0:
    print(
        "\n[WARNING] No environmental preprocessing artifacts were found."
    )

# ================================================================================================
# 4. INSPECT EVERY PREPROCESSING OBJECT
# ================================================================================================

print("\n" + "=" * 100)
print("4. EXACT PREPROCESSING OBJECT INSPECTION")
print("=" * 100)

preprocessor_info = []

for idx, path in enumerate(preprocessor_candidates, 1):

    print("\n" + "-" * 100)
    print(f"PREPROCESSOR CANDIDATE {idx}")
    print("-" * 100)

    print("Path:", path)

    try:
        obj = joblib.load(path)

        print("Type:", type(obj))

        info = {
            "path": path,
            "type": str(type(obj)),
            "object": obj,
            "n_features_in": None,
            "n_features_out": None,
            "feature_names_in": None,
        }

        # ------------------------------------------------------------------
        # Generic n_features_in_
        # ------------------------------------------------------------------

        if hasattr(obj, "n_features_in_"):
            info["n_features_in"] = int(obj.n_features_in_)
            print("n_features_in_:", info["n_features_in"])

        else:
            print("n_features_in_: NOT AVAILABLE")

        # ------------------------------------------------------------------
        # Generic get_feature_names_out
        # ------------------------------------------------------------------

        if hasattr(obj, "get_feature_names_out"):

            try:
                names = obj.get_feature_names_out()

                info["n_features_out"] = len(names)

                print("n_features_out:", len(names))

                print("First output feature names:")
                print(list(names[:20]))

            except Exception as e:

                print(
                    "[INFO] get_feature_names_out unavailable:",
                    str(e)
                )

        else:
            print("get_feature_names_out: NOT AVAILABLE")

        # ------------------------------------------------------------------
        # Feature names
        # ------------------------------------------------------------------

        if hasattr(obj, "feature_names_in_"):

            try:
                names = list(obj.feature_names_in_)

                info["feature_names_in"] = names

                print("feature_names_in_ count:", len(names))
                print("feature_names_in_:")
                print(names)

            except Exception:
                pass

        # ------------------------------------------------------------------
        # Pipeline structure
        # ------------------------------------------------------------------

        if hasattr(obj, "steps"):

            print("\nPipeline steps:")

            for name, transformer in obj.steps:

                print(
                    f"  {name:<30} "
                    f"{type(transformer)}"
                )

        # ------------------------------------------------------------------
        # ColumnTransformer structure
        # ------------------------------------------------------------------

        if hasattr(obj, "transformers_"):

            print("\nColumnTransformer structure:")

            for name, transformer, columns in obj.transformers_:

                print(
                    f"\n  Transformer : {name}"
                )

                print(
                    f"  Type        : {type(transformer)}"
                )

                try:
                    print(
                        f"  Columns     : {list(columns)}"
                    )
                except Exception:
                    print(
                        f"  Columns     : {columns}"
                    )

        preprocessor_info.append(info)

        print("[PASSED] Artifact inspected.")

    except Exception as e:

        print("[ERROR] Could not load artifact:")
        print(str(e))

# ================================================================================================
# 5. TEST EVERY CANDIDATE AGAINST REAL P001 INPUT
# ================================================================================================

print("\n" + "=" * 100)
print("5. REAL-DATA TRANSFORMATION COMPATIBILITY TEST")
print("=" * 100)

reference_pid = "P001"
reference_df = environmental_data[reference_pid].copy()

print("Reference patient:", reference_pid)
print("Reference shape  :", reference_df.shape)

compatible_candidates = []

for idx, info in enumerate(preprocessor_info, 1):

    obj = info["object"]
    path = info["path"]

    print("\n" + "-" * 100)
    print(f"TESTING CANDIDATE {idx}")
    print("Path:", path)
    print("-" * 100)

    # ------------------------------------------------------------------
    # Test DataFrame directly
    # ------------------------------------------------------------------

    try:

        transformed = obj.transform(reference_df)

        arr = np.asarray(transformed)

        print("[PASSED] DataFrame transformation succeeded.")
        print("Output type :", type(transformed))
        print("Output shape:", arr.shape)
        print("Output dtype:", arr.dtype)

        if np.all(np.isfinite(arr)):
            print("[PASSED] Output contains only finite values.")
        else:
            print("[FAILED] Output contains non-finite values.")
            continue

        compatible_candidates.append({
            "path": path,
            "object": obj,
            "output_shape": arr.shape,
            "mode": "dataframe"
        })

        continue

    except Exception as e:

        print("[INFO] Direct DataFrame transform failed:")
        print("       ", str(e))

    # ------------------------------------------------------------------
    # Test numeric-only DataFrame
    # ------------------------------------------------------------------

    try:

        numeric_df = reference_df.select_dtypes(
            include=[np.number]
        ).copy()

        transformed = obj.transform(numeric_df)

        arr = np.asarray(transformed)

        print("[PASSED] Numeric-only transformation succeeded.")
        print("Numeric input shape:", numeric_df.shape)
        print("Output shape        :", arr.shape)

        if np.all(np.isfinite(arr)):

            print("[PASSED] Output contains only finite values.")

            compatible_candidates.append({
                "path": path,
                "object": obj,
                "output_shape": arr.shape,
                "mode": "numeric_dataframe"
            })

        else:

            print("[FAILED] Output contains non-finite values.")

    except Exception as e:

        print("[INFO] Numeric-only transform failed:")
        print("       ", str(e))

# ================================================================================================
# 6. COMPATIBILITY DECISION
# ================================================================================================

print("\n" + "=" * 100)
print("6. ENVIRONMENTAL PREPROCESSING COMPATIBILITY DECISION")
print("=" * 100)

if len(compatible_candidates) == 0:

    print("[BLOCKED] No preprocessing candidate successfully transformed P001.")
    print()
    print("This is intentional.")
    print("We will NOT guess the environmental preprocessing pipeline.")
    print()
    print("NEXT ACTION:")
    print("Inspect the exact environmental input columns and")
    print("the internal structure of the two preprocessing candidates.")

else:

    print(
        "[PASSED] Compatible preprocessing candidates:",
        len(compatible_candidates)
    )

    for i, item in enumerate(compatible_candidates, 1):

        print(
            f"[{i}] {item['path']}"
        )

        print(
            f"    mode         : {item['mode']}"
        )

        print(
            f"    output shape : {item['output_shape']}"
        )

# ================================================================================================
# 7. CHECK WHETHER ANY CANDIDATE PRODUCES THE REQUIRED 32-D INPUT
# ================================================================================================

print("\n" + "=" * 100)
print("7. REQUIRED 32-D COMPATIBILITY CHECK")
print("=" * 100)

thirty_two_candidates = []

for item in compatible_candidates:

    shape = item["output_shape"]

    if len(shape) == 2 and shape[1] == 32:

        thirty_two_candidates.append(item)

        print(
            "[PASSED] Candidate directly produces required 32-D representation:"
        )
        print(
            "        ",
            item["path"]
        )

if len(thirty_two_candidates) == 0:

    print(
        "[INFO] No preprocessing artifact directly produces 32-D."
    )

    print(
        "This means an environmental representation model may exist "
        "between preprocessing and the verified expert."
    )

else:

    print(
        "[PASSED] Environmental expert-compatible preprocessing found."
    )

# ================================================================================================
# 8. FINAL STATUS
# ================================================================================================

print("\n" + "=" * 100)
print("M4.4.10.5B STATUS")
print("=" * 100)

print(
    "[PASSED] Actual P001-P005 environmental inputs inspected."
)

print(
    "[PASSED] Every discovered preprocessing candidate inspected."
)

print(
    "[PASSED] Real-data compatibility tested."
)

print(
    "[PASSED] No preprocessing artifact modified."
)

print(
    "[PASSED] No checkpoint modified."
)

print(
    "[PASSED] No training performed."
)

print(
    "[PASSED] No architecture guessed."
)

print("=" * 100)
print("NEXT")
print("=" * 100)

if len(thirty_two_candidates) > 0:

    print(
        "A verified preprocessing path produces the required 32-D input."
    )

    print(
        "Next: M4.4.10.6 — P001-P005 ENVIRONMENTAL PRODUCTION INFERENCE"
    )

else:

    print(
        "The next step is to inspect the environmental upstream "
        "representation checkpoint(s) using the actual artifact "
        "structures before reconstruction."
    )

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.5B
EXACT ENVIRONMENTAL PREPROCESSING COMPATIBILITY INSPECTION
[PASSED] Project directory available.
Project : /content/gdrive/MyDrive/ASD_Multimodal_Project

1. RESOLVING ENVIRONMENTAL DEMO INPUTS
[FOUND] P001  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P001/environmental_input.csv
[FOUND] P002  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P002/environmental_input.csv
[FOUND] P003  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P003/environmental_input.csv
[FOUND] P004  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P004/environmental_input.csv
[FOUND] P005  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P005/environmental_input.csv
[PASSED] P001-P005 environmental files resolved.

2. ACTUAL ENVIRONMENTAL INPUT STRUCTURE

--- P001 ---
File       : /content/gdri

In [ ]:
# ================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.6
# EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION CHECKPOINT INSPECTION
# ================================================================================================
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# NO ARCHITECTURE GUESSING
# ================================================================================================

import os
import glob
import torch
from collections import OrderedDict

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.6")
print("EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION CHECKPOINT INSPECTION")
print("=" * 100)

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}\n"
        "Reconnect Google Drive first."
    )

print("[PASSED] Project directory available.")
print("Project :", PROJECT_DIR)

# ================================================================================================
# 1. SEARCH ENVIRONMENTAL MODEL DIRECTORY
# ================================================================================================

print("\n" + "=" * 100)
print("1. ENVIRONMENTAL MODEL DIRECTORY")
print("=" * 100)

env_dir = os.path.join(PROJECT_DIR, "Environmental_Model")

if os.path.isdir(env_dir):
    print("[FOUND]", env_dir)

    for root, dirs, files in os.walk(env_dir):

        level = root.replace(env_dir, "").count(os.sep)
        indent = "  " * level

        print(f"{indent}{os.path.basename(root)}/")

        for f in sorted(files):
            print(f"{indent}  {f}")

else:
    print("[WARNING] Environmental_Model directory not found.")

# ================================================================================================
# 2. DISCOVER ALL ENVIRONMENTAL PYTORCH CHECKPOINTS
# ================================================================================================

print("\n" + "=" * 100)
print("2. ENVIRONMENTAL PYTORCH CHECKPOINT DISCOVERY")
print("=" * 100)

all_pt = []

for root, dirs, files in os.walk(PROJECT_DIR):

    for filename in files:

        if filename.lower().endswith((".pt", ".pth", ".bin")):

            path = os.path.join(root, filename)

            if "environment" in path.lower() or "env" in path.lower():

                all_pt.append(path)

all_pt = sorted(set(all_pt))

print("Environmental checkpoint candidates:", len(all_pt))

for i, path in enumerate(all_pt, 1):
    print(f"[{i}] {path}")

# ================================================================================================
# 3. LOAD AND INSPECT EVERY CHECKPOINT
# ================================================================================================

print("\n" + "=" * 100)
print("3. EXACT ENVIRONMENTAL CHECKPOINT STRUCTURE")
print("=" * 100)

inspected = []

for idx, path in enumerate(all_pt, 1):

    print("\n" + "-" * 100)
    print(f"CHECKPOINT {idx}")
    print("-" * 100)
    print("File:", path)

    try:

        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print("Checkpoint type:", type(checkpoint))

        # ------------------------------------------------------------
        # Extract state dict
        # ------------------------------------------------------------

        if isinstance(checkpoint, OrderedDict):

            state = checkpoint

        elif isinstance(checkpoint, dict):

            possible_keys = [
                "state_dict",
                "model_state_dict",
                "model",
                "network"
            ]

            state = None

            for key in possible_keys:

                if key in checkpoint and isinstance(
                    checkpoint[key],
                    (dict, OrderedDict)
                ):

                    state = checkpoint[key]
                    print("[FOUND] Nested state dict:", key)
                    break

            if state is None:
                print("[INFO] Dictionary does not expose a standard state_dict.")
                continue

        else:

            print("[INFO] Unsupported checkpoint container.")
            continue

        print("State-dict entries:", len(state))

        print("\nExact state-dict keys and shapes:")

        for key, tensor in state.items():

            if hasattr(tensor, "shape"):

                print(
                    f"{key:<75} "
                    f"{tuple(tensor.shape)!s:<20} "
                    f"{str(tensor.dtype)}"
                )

            else:

                print(
                    f"{key:<75} "
                    f"type={type(tensor)}"
                )

        # ------------------------------------------------------------
        # Detect Linear candidates
        # ------------------------------------------------------------

        print("\nLinear layer candidates:")

        linear_found = False

        for key, tensor in state.items():

            if key.endswith(".weight") and len(tensor.shape) == 2:

                out_features = tensor.shape[0]
                in_features = tensor.shape[1]

                print(
                    f"{key:<70} "
                    f"{in_features} -> {out_features}"
                )

                linear_found = True

        if not linear_found:
            print("[NONE] No 2-D Linear weight candidates.")

        # ------------------------------------------------------------
        # Detect normalization
        # ------------------------------------------------------------

        print("\nNormalization candidates:")

        norm_found = False

        for key, tensor in state.items():

            if (
                "running_mean" in key
                or "running_var" in key
                or key.endswith(".norm.weight")
                or key.endswith(".norm.bias")
            ):

                print(key)
                norm_found = True

        if not norm_found:
            print("[NONE] No obvious normalization buffers.")

        # ------------------------------------------------------------
        # Save inspected information
        # ------------------------------------------------------------

        inspected.append({
            "path": path,
            "state": state
        })

        print("\n[PASSED] Checkpoint inspection completed.")

    except Exception as e:

        print("[ERROR] Could not inspect checkpoint:")
        print(str(e))

# ================================================================================================
# 4. IDENTIFY CHECKPOINTS THAT COULD PRODUCE 32-D
# ================================================================================================

print("\n" + "=" * 100)
print("4. 32-D OUTPUT CANDIDATE ANALYSIS")
print("=" * 100)

candidates_32 = []

for item in inspected:

    path = item["path"]
    state = item["state"]

    for key, tensor in state.items():

        if key.endswith(".weight") and len(tensor.shape) == 2:

            out_features = tensor.shape[0]

            if out_features == 32:

                candidates_32.append(
                    (path, key, tuple(tensor.shape))
                )

                print(
                    "[FOUND]",
                    os.path.basename(path),
                    "|",
                    key,
                    "|",
                    tuple(tensor.shape)
                )

if not candidates_32:

    print(
        "[INFO] No checkpoint contains an explicit Linear layer "
        "ending in 32 dimensions."
    )

# ================================================================================================
# 5. IDENTIFY CHECKPOINTS THAT COULD PRODUCE 128-D
# ================================================================================================

print("\n" + "=" * 100)
print("5. 128-D OUTPUT CANDIDATE ANALYSIS")
print("=" * 100)

candidates_128 = []

for item in inspected:

    path = item["path"]
    state = item["state"]

    for key, tensor in state.items():

        if key.endswith(".weight") and len(tensor.shape) == 2:

            out_features = tensor.shape[0]

            if out_features == 128:

                candidates_128.append(
                    (path, key, tuple(tensor.shape))
                )

                print(
                    "[FOUND]",
                    os.path.basename(path),
                    "|",
                    key,
                    "|",
                    tuple(tensor.shape)
                )

if not candidates_128:

    print(
        "[INFO] No explicit 128-D Linear output candidate found."
    )

# ================================================================================================
# 6. FINAL DECISION
# ================================================================================================

print("\n" + "=" * 100)
print("M4.4.10.6 DECISION")
print("=" * 100)

print(
    "[PASSED] Environmental checkpoint search completed."
)

print(
    "[PASSED] Actual checkpoint state-dict structures inspected."
)

print(
    "[PASSED] Linear dimensions inspected."
)

print(
    "[PASSED] Normalization structures inspected."
)

print(
    "[PASSED] 32-D candidates identified where present."
)

print(
    "[PASSED] 128-D candidates identified where present."
)

print(
    "[PASSED] No model reconstructed."
)

print(
    "[PASSED] No training performed."
)

print(
    "[PASSED] No checkpoint modified."
)

print(
    "[PASSED] No production artifact overwritten."
)

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)

if candidates_32:

    print(
        "A checkpoint contains an explicit 32-D representation candidate."
    )
    print(
        "Next: exact environmental upstream reconstruction."
    )

elif candidates_128:

    print(
        "A 128-D environmental representation candidate was found."
    )
    print(
        "Next: determine whether an alignment checkpoint maps "
        "128-D → 32-D."
    )

else:

    print(
        "No direct 32-D/128-D candidate identified."
    )
    print(
        "Next: inspect environmental source/notebook references "
        "before any reconstruction."
    )

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.6
EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION CHECKPOINT INSPECTION
[PASSED] Project directory available.
Project : /content/gdrive/MyDrive/ASD_Multimodal_Project

1. ENVIRONMENTAL MODEL DIRECTORY
[FOUND] /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model
Environmental_Model/
  01_raw_data/
    environmental_dataset.csv
  02_preprocessing/
    environmental_cleaned_no_leakage.csv
    environmental_preprocessing_report.txt
    environmental_preprocessor.joblib
    environmental_processed_feature_names.txt
    environmental_test_processed_unseen.csv
    environmental_train_processed.csv
  03_representation_learning/
    environmental_128d_corrected_validation_report.txt
    environmental_128d_duplicate_embedding_investigation.txt
    environmental_128d_embeddings.csv
    environmental_128d_training_report.txt
    environmental_128d_validation_report.txt
    environmental_encoder_128d.pt
    environmental_test_predictions.csv
  04_cross_mod

In [ ]:
# ================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.7
# EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION DISCOVERY + CHECKPOINT COMPATIBILITY
# ================================================================================================
#
# PURPOSE:
#   Identify the ACTUAL environmental upstream representation pipeline that produces
#   the verified 32-D input expected by M3_1_environmental_expert.pt.
#
# IMPORTANT:
#   - No training
#   - No checkpoint modification
#   - No architecture guessing
#   - No behavioral/genetic reconstruction
#   - No production artifact overwrite
#
# ================================================================================================

import os
import glob
import json
import joblib
import torch
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
ENV_DIR = os.path.join(PROJECT_DIR, "Environmental_Model")
FUSION_DIR = os.path.join(PROJECT_DIR, "05_multimodal_fusion")
DEMO_DIR = os.path.join(FUSION_DIR, "M4_demo_patients")

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.7")
print("EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION DISCOVERY")
print("=" * 100)

# ------------------------------------------------------------------------------------------------
# 1. PROJECT VALIDATION
# ------------------------------------------------------------------------------------------------

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}\n"
        "Reconnect Google Drive before running this cell."
    )

print("[PASSED] Project directory available.")
print("Project:", PROJECT_DIR)

# ------------------------------------------------------------------------------------------------
# 2. DISCOVER ENVIRONMENTAL ARTIFACTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. ENVIRONMENTAL ARTIFACT DISCOVERY")
print("=" * 100)

patterns = [
    "**/*.pt",
    "**/*.pth",
    "**/*.joblib",
    "**/*.pkl",
    "**/*.pickle",
    "**/*.json",
    "**/*.csv",
    "**/*.parquet",
    "**/*.npy",
    "**/*.npz",
]

env_candidates = []

for pattern in patterns:
    env_candidates.extend(
        glob.glob(
            os.path.join(PROJECT_DIR, pattern),
            recursive=True
        )
    )

# Keep only artifacts whose path/name appears environmental
env_candidates = sorted(set(
    p for p in env_candidates
    if any(
        token in p.lower()
        for token in [
            "environment",
            "environmental",
            "env_",
            "_env",
            "ecological",
            "exposure"
        ]
    )
))

print(f"[FOUND] Environmental-related artifacts: {len(env_candidates)}")

for p in env_candidates:
    print(" ", p)

# ------------------------------------------------------------------------------------------------
# 3. CLASSIFY PREPROCESSING CANDIDATES
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. PREPROCESSING ARTIFACT INSPECTION")
print("=" * 100)

preprocessors = []

for p in env_candidates:
    low = os.path.basename(p).lower()

    if any(
        token in low
        for token in [
            "preprocess",
            "preprocessor",
            "scaler",
            "transform",
            "encoder",
            "normalizer"
        ]
    ):
        preprocessors.append(p)

print(f"[FOUND] Environmental preprocessing candidates: {len(preprocessors)}")

for p in preprocessors:
    print("\n[ARTIFACT]", p)

    try:
        obj = joblib.load(p)

        print("  Type:", type(obj))

        if hasattr(obj, "n_features_in_"):
            print("  n_features_in_:", obj.n_features_in_)

        if hasattr(obj, "feature_names_in_"):
            names = list(obj.feature_names_in_)
            print("  feature_names_in_ count:", len(names))
            print("  first features:", names[:10])

        if hasattr(obj, "transformers_"):
            print("  ColumnTransformer detected")

        if hasattr(obj, "mean_"):
            try:
                print("  mean_ shape:", np.asarray(obj.mean_).shape)
            except:
                pass

        if hasattr(obj, "scale_"):
            try:
                print("  scale_ shape:", np.asarray(obj.scale_).shape)
            except:
                pass

    except Exception as e:
        print("  [INFO] Could not load with joblib:", repr(e))

# ------------------------------------------------------------------------------------------------
# 4. INSPECT ALL ENVIRONMENTAL PYTORCH CHECKPOINTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. ENVIRONMENTAL PYTORCH CHECKPOINT INSPECTION")
print("=" * 100)

pt_files = [
    p for p in env_candidates
    if p.lower().endswith((".pt", ".pth"))
]

print(f"[FOUND] Environmental PyTorch artifacts: {len(pt_files)}")

for path in pt_files:

    print("\n" + "-" * 100)
    print("CHECKPOINT")
    print("-" * 100)
    print(path)

    try:
        state = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        if isinstance(state, dict):
            keys = list(state.keys())

            print("State-dict type:", type(state))
            print("Number of entries:", len(keys))

            for key in keys:
                value = state[key]

                if torch.is_tensor(value):
                    print(
                        f"{key:70s}",
                        "shape=",
                        tuple(value.shape),
                        "dtype=",
                        value.dtype
                    )

        else:
            print("Checkpoint object type:", type(state))

    except Exception as e:
        print("[INFO] Could not inspect:", repr(e))

# ------------------------------------------------------------------------------------------------
# 5. LOCATE DEMO ENVIRONMENTAL INPUTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. ENVIRONMENTAL DEMO INPUT DISCOVERY")
print("=" * 100)

patients = [f"P{i:03d}" for i in range(1, 6)]

environmental_files = {}

for pid in patients:

    patient_dir = os.path.join(DEMO_DIR, pid)

    candidates = []

    if os.path.isdir(patient_dir):
        for root, dirs, files in os.walk(patient_dir):

            for fname in files:

                low = fname.lower()

                if (
                    "environment" in low
                    or "environmental" in low
                    or low.startswith("env")
                    or "_env" in low
                ):
                    candidates.append(
                        os.path.join(root, fname)
                    )

    environmental_files[pid] = sorted(candidates)

    print(f"\n{pid}: {len(candidates)} environmental file(s)")

    for p in candidates:
        print("   ", p)

# ------------------------------------------------------------------------------------------------
# 6. INSPECT DEMO INPUT DIMENSIONS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. ENVIRONMENTAL DEMO DIMENSION INSPECTION")
print("=" * 100)

for pid, files in environmental_files.items():

    for path in files:

        print("\n---", pid, "---")
        print(path)

        try:

            low = path.lower()

            if low.endswith(".csv"):

                df = pd.read_csv(path)

                print("Shape:", df.shape)
                print("Columns:", len(df.columns))

                numeric_cols = df.select_dtypes(
                    include=[np.number]
                ).columns

                print("Numeric columns:", len(numeric_cols))

                print(
                    "First columns:",
                    list(df.columns[:15])
                )

            elif low.endswith(".npy"):

                arr = np.load(path, allow_pickle=False)

                print("Array shape:", arr.shape)
                print("dtype:", arr.dtype)

            elif low.endswith(".npz"):

                data = np.load(path)

                print("NPZ keys:", list(data.keys()))

                for key in data.keys():

                    arr = data[key]

                    print(
                        f"  {key}: shape={arr.shape}, dtype={arr.dtype}"
                    )

        except Exception as e:

            print("[INFO] Inspection failed:", repr(e))

# ------------------------------------------------------------------------------------------------
# 7. SEARCH SOURCE / NOTEBOOK REFERENCES
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. ENVIRONMENTAL SOURCE REFERENCE DISCOVERY")
print("=" * 100)

source_files = []

for root, dirs, files in os.walk(PROJECT_DIR):

    for fname in files:

        low = fname.lower()

        if low.endswith(
            (
                ".py",
                ".ipynb",
                ".txt",
                ".md"
            )
        ):
            source_files.append(
                os.path.join(root, fname)
            )

print(f"[FOUND] Source/reference files: {len(source_files)}")

# Search source text for environmental + 32-D clues
matches = []

keywords = [
    "environmental",
    "environment",
    "32",
    "embedding",
    "representation",
    "encoder",
    "projection"
]

for path in source_files:

    try:

        if path.endswith(".ipynb"):

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:
                notebook = json.load(f)

            text = json.dumps(notebook)

        else:

            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:
                text = f.read()

        low = text.lower()

        if (
            ("environment" in low or "environmental" in low)
            and
            ("32" in low)
        ):
            matches.append(path)

    except Exception:
        pass

print(
    f"[FOUND] Source/reference files containing environmental + 32-D clues: "
    f"{len(matches)}"
)

for p in matches[:30]:
    print(" ", p)

# ------------------------------------------------------------------------------------------------
# 8. FINAL DECISION
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. ENVIRONMENTAL UPSTREAM DISCOVERY STATUS")
print("=" * 100)

if preprocessors:
    print("[FOUND] Environmental preprocessing candidates available.")
else:
    print("[INFO] No environmental preprocessing artifact conclusively identified.")

if pt_files:
    print("[FOUND] Environmental PyTorch artifacts available.")
else:
    print("[INFO] No environmental PyTorch artifact identified.")

resolved_count = sum(
    bool(files)
    for files in environmental_files.values()
)

print(
    f"[INFO] Environmental demo patients with discovered inputs: "
    f"{resolved_count}/5"
)

if matches:
    print("[FOUND] Environmental source/reference candidates available.")
else:
    print("[INFO] No environmental source/reference candidate identified.")

print("\n" + "=" * 100)
print("M4.4.10.7 COMPLETED")
print("=" * 100)

print("[PASSED] Environmental upstream artifacts inspected.")
print("[PASSED] Environmental preprocessing candidates inspected.")
print("[PASSED] Environmental checkpoints inspected.")
print("[PASSED] Environmental P001-P005 inputs inspected.")
print("[PASSED] Source/reference candidates searched.")

print("\nNO TRAINING PERFORMED.")
print("NO CHECKPOINT MODIFIED.")
print("NO PRODUCTION ARTIFACT OVERWRITTEN.")

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)

print("""
Use ONLY the actual output above to reconstruct the environmental
upstream representation.

Target:

ENVIRONMENTAL RAW INPUT
        ↓
ACTUAL PREPROCESSING
        ↓
ACTUAL ENVIRONMENTAL REPRESENTATION
        ↓
32-D
        ↓
VERIFIED ENVIRONMENTAL EXPERT
        ↓
ASD PROBABILITY

After successful validation, immediately proceed to:

M4.4.11 — MULTIMODAL FUSION INSPECTION
M4.4.12 — FUSION RECONSTRUCTION
M4.4.13 — P001-P005 COMPLETE THREE-MODAL INFERENCE
M4.4.14 — ANY-ONE / ANY-TWO / ALL-THREE INPUT INFERENCE
M4.4.15 — FINAL INTERFACE

DO NOT RECONSTRUCT BEHAVIORAL AGAIN.
DO NOT RECONSTRUCT GENETIC AGAIN.
DO NOT TRAIN ANY MODEL.
DO NOT MODIFY ANY CHECKPOINT.
""")

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.7
EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION DISCOVERY
[PASSED] Project directory available.
Project: /content/gdrive/MyDrive/ASD_Multimodal_Project

2. ENVIRONMENTAL ARTIFACT DISCOVERY
[FOUND] Environmental-related artifacts: 19
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_scaler.joblib
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P001/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P002/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P003/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P004/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/

In [ ]:
# ================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.7
# EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION DISCOVERY + CHECKPOINT COMPATIBILITY
# ================================================================================================
#
# PURPOSE:
#   Identify the ACTUAL environmental upstream representation pipeline that produces
#   the verified 32-D input expected by M3_1_environmental_expert.pt.
#
# IMPORTANT:
#   - No training
#   - No checkpoint modification
#   - No architecture guessing
#   - No behavioral/genetic reconstruction
#   - No production artifact overwrite
#
# ================================================================================================

import os
import glob
import json
import joblib
import torch
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
ENV_DIR = os.path.join(PROJECT_DIR, "Environmental_Model")
FUSION_DIR = os.path.join(PROJECT_DIR, "05_multimodal_fusion")
DEMO_DIR = os.path.join(FUSION_DIR, "M4_demo_patients")

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.7")
print("EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION DISCOVERY")
print("=" * 100)

# ------------------------------------------------------------------------------------------------
# 1. PROJECT VALIDATION
# ------------------------------------------------------------------------------------------------

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}\n"
        "Reconnect Google Drive before running this cell."
    )

print("[PASSED] Project directory available.")
print("Project:", PROJECT_DIR)

# ------------------------------------------------------------------------------------------------
# 2. DISCOVER ENVIRONMENTAL ARTIFACTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. ENVIRONMENTAL ARTIFACT DISCOVERY")
print("=" * 100)

patterns = [
    "**/*.pt",
    "**/*.pth",
    "**/*.joblib",
    "**/*.pkl",
    "**/*.pickle",
    "**/*.json",
    "**/*.csv",
    "**/*.parquet",
    "**/*.npy",
    "**/*.npz",
]

env_candidates = []

for pattern in patterns:
    env_candidates.extend(
        glob.glob(
            os.path.join(PROJECT_DIR, pattern),
            recursive=True
        )
    )

# Keep only artifacts whose path/name appears environmental
env_candidates = sorted(set(
    p for p in env_candidates
    if any(
        token in p.lower()
        for token in [
            "environment",
            "environmental",
            "env_",
            "_env",
            "ecological",
            "exposure"
        ]
    )
))

print(f"[FOUND] Environmental-related artifacts: {len(env_candidates)}")

for p in env_candidates:
    print(" ", p)

# ------------------------------------------------------------------------------------------------
# 3. CLASSIFY PREPROCESSING CANDIDATES
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. PREPROCESSING ARTIFACT INSPECTION")
print("=" * 100)

preprocessors = []

for p in env_candidates:
    low = os.path.basename(p).lower()

    if any(
        token in low
        for token in [
            "preprocess",
            "preprocessor",
            "scaler",
            "transform",
            "encoder",
            "normalizer"
        ]
    ):
        preprocessors.append(p)

print(f"[FOUND] Environmental preprocessing candidates: {len(preprocessors)}")

for p in preprocessors:
    print("\n[ARTIFACT]", p)

    try:
        obj = joblib.load(p)

        print("  Type:", type(obj))

        if hasattr(obj, "n_features_in_"):
            print("  n_features_in_:", obj.n_features_in_)

        if hasattr(obj, "feature_names_in_"):
            names = list(obj.feature_names_in_)
            print("  feature_names_in_ count:", len(names))
            print("  first features:", names[:10])

        if hasattr(obj, "transformers_"):
            print("  ColumnTransformer detected")

        if hasattr(obj, "mean_"):
            try:
                print("  mean_ shape:", np.asarray(obj.mean_).shape)
            except:
                pass

        if hasattr(obj, "scale_"):
            try:
                print("  scale_ shape:", np.asarray(obj.scale_).shape)
            except:
                pass

    except Exception as e:
        print("  [INFO] Could not load with joblib:", repr(e))

# ------------------------------------------------------------------------------------------------
# 4. INSPECT ALL ENVIRONMENTAL PYTORCH CHECKPOINTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. ENVIRONMENTAL PYTORCH CHECKPOINT INSPECTION")
print("=" * 100)

pt_files = [
    p for p in env_candidates
    if p.lower().endswith((".pt", ".pth"))
]

print(f"[FOUND] Environmental PyTorch artifacts: {len(pt_files)}")

for path in pt_files:

    print("\n" + "-" * 100)
    print("CHECKPOINT")
    print("-" * 100)
    print(path)

    try:
        state = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        if isinstance(state, dict):
            keys = list(state.keys())

            print("State-dict type:", type(state))
            print("Number of entries:", len(keys))

            for key in keys:
                value = state[key]

                if torch.is_tensor(value):
                    print(
                        f"{key:70s}",
                        "shape=",
                        tuple(value.shape),
                        "dtype=",
                        value.dtype
                    )

        else:
            print("Checkpoint object type:", type(state))

    except Exception as e:
        print("[INFO] Could not inspect:", repr(e))

# ------------------------------------------------------------------------------------------------
# 5. LOCATE DEMO ENVIRONMENTAL INPUTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. ENVIRONMENTAL DEMO INPUT DISCOVERY")
print("=" * 100)

patients = [f"P{i:03d}" for i in range(1, 6)]

environmental_files = {}

for pid in patients:

    patient_dir = os.path.join(DEMO_DIR, pid)

    candidates = []

    if os.path.isdir(patient_dir):
        for root, dirs, files in os.walk(patient_dir):

            for fname in files:

                low = fname.lower()

                if (
                    "environment" in low
                    or "environmental" in low
                    or low.startswith("env")
                    or "_env" in low
                ):
                    candidates.append(
                        os.path.join(root, fname)
                    )

    environmental_files[pid] = sorted(candidates)

    print(f"\n{pid}: {len(candidates)} environmental file(s)")

    for p in candidates:
        print("   ", p)

# ------------------------------------------------------------------------------------------------
# 6. INSPECT DEMO INPUT DIMENSIONS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. ENVIRONMENTAL DEMO DIMENSION INSPECTION")
print("=" * 100)

for pid, files in environmental_files.items():

    for path in files:

        print("\n---", pid, "---")
        print(path)

        try:

            low = path.lower()

            if low.endswith(".csv"):

                df = pd.read_csv(path)

                print("Shape:", df.shape)
                print("Columns:", len(df.columns))

                numeric_cols = df.select_dtypes(
                    include=[np.number]
                ).columns

                print("Numeric columns:", len(numeric_cols))

                print(
                    "First columns:",
                    list(df.columns[:15])
                )

            elif low.endswith(".npy"):

                arr = np.load(path, allow_pickle=False)

                print("Array shape:", arr.shape)
                print("dtype:", arr.dtype)

            elif low.endswith(".npz"):

                data = np.load(path)

                print("NPZ keys:", list(data.keys()))

                for key in data.keys():

                    arr = data[key]

                    print(
                        f"  {key}: shape={arr.shape}, dtype={arr.dtype}"
                    )

        except Exception as e:

            print("[INFO] Inspection failed:", repr(e))

# ------------------------------------------------------------------------------------------------
# 7. SEARCH SOURCE / NOTEBOOK REFERENCES
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. ENVIRONMENTAL SOURCE REFERENCE DISCOVERY")
print("=" * 100)

source_files = []

for root, dirs, files in os.walk(PROJECT_DIR):

    for fname in files:

        low = fname.lower()

        if low.endswith(
            (
                ".py",
                ".ipynb",
                ".txt",
                ".md"
            )
        ):
            source_files.append(
                os.path.join(root, fname)
            )

print(f"[FOUND] Source/reference files: {len(source_files)}")

# Search source text for environmental + 32-D clues
matches = []

keywords = [
    "environmental",
    "environment",
    "32",
    "embedding",
    "representation",
    "encoder",
    "projection"
]

for path in source_files:

    try:

        if path.endswith(".ipynb"):

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:
                notebook = json.load(f)

            text = json.dumps(notebook)

        else:

            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:
                text = f.read()

        low = text.lower()

        if (
            ("environment" in low or "environmental" in low)
            and
            ("32" in low)
        ):
            matches.append(path)

    except Exception:
        pass

print(
    f"[FOUND] Source/reference files containing environmental + 32-D clues: "
    f"{len(matches)}"
)

for p in matches[:30]:
    print(" ", p)

# ------------------------------------------------------------------------------------------------
# 8. FINAL DECISION
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. ENVIRONMENTAL UPSTREAM DISCOVERY STATUS")
print("=" * 100)

if preprocessors:
    print("[FOUND] Environmental preprocessing candidates available.")
else:
    print("[INFO] No environmental preprocessing artifact conclusively identified.")

if pt_files:
    print("[FOUND] Environmental PyTorch artifacts available.")
else:
    print("[INFO] No environmental PyTorch artifact identified.")

resolved_count = sum(
    bool(files)
    for files in environmental_files.values()
)

print(
    f"[INFO] Environmental demo patients with discovered inputs: "
    f"{resolved_count}/5"
)

if matches:
    print("[FOUND] Environmental source/reference candidates available.")
else:
    print("[INFO] No environmental source/reference candidate identified.")

print("\n" + "=" * 100)
print("M4.4.10.7 COMPLETED")
print("=" * 100)

print("[PASSED] Environmental upstream artifacts inspected.")
print("[PASSED] Environmental preprocessing candidates inspected.")
print("[PASSED] Environmental checkpoints inspected.")
print("[PASSED] Environmental P001-P005 inputs inspected.")
print("[PASSED] Source/reference candidates searched.")

print("\nNO TRAINING PERFORMED.")
print("NO CHECKPOINT MODIFIED.")
print("NO PRODUCTION ARTIFACT OVERWRITTEN.")

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)

print("""
Use ONLY the actual output above to reconstruct the environmental
upstream representation.

Target:

ENVIRONMENTAL RAW INPUT
        ↓
ACTUAL PREPROCESSING
        ↓
ACTUAL ENVIRONMENTAL REPRESENTATION
        ↓
32-D
        ↓
VERIFIED ENVIRONMENTAL EXPERT
        ↓
ASD PROBABILITY

After successful validation, immediately proceed to:

M4.4.11 — MULTIMODAL FUSION INSPECTION
M4.4.12 — FUSION RECONSTRUCTION
M4.4.13 — P001-P005 COMPLETE THREE-MODAL INFERENCE
M4.4.14 — ANY-ONE / ANY-TWO / ALL-THREE INPUT INFERENCE
M4.4.15 — FINAL INTERFACE

DO NOT RECONSTRUCT BEHAVIORAL AGAIN.
DO NOT RECONSTRUCT GENETIC AGAIN.
DO NOT TRAIN ANY MODEL.
DO NOT MODIFY ANY CHECKPOINT.
""")

print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.7
EXACT ENVIRONMENTAL UPSTREAM REPRESENTATION DISCOVERY
[PASSED] Project directory available.
Project: /content/gdrive/MyDrive/ASD_Multimodal_Project

2. ENVIRONMENTAL ARTIFACT DISCOVERY
[FOUND] Environmental-related artifacts: 19
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_scaler.joblib
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P001/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P002/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P003/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P004/environmental_input.csv
  /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/

In [ ]:
# ================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.8.1
# ENVIRONMENTAL ALIGNMENT CHECKPOINT — TRUSTED LOAD + EXACT INSPECTION
# ================================================================================================

import os
import torch
import numpy as np

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"

ALIGNMENT_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "04_cross_modal_alignment",
    "environmental_128_to_32_alignment.pt"
)

EXPERT_PATH = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M3_1_environmental_expert.pt"
)

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.8.1")
print("ENVIRONMENTAL ALIGNMENT — TRUSTED LOAD + EXACT INSPECTION")
print("=" * 100)

print(f"Project : {PROJECT_DIR}")

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_DIR}"
    )

if not os.path.isfile(ALIGNMENT_PATH):
    raise FileNotFoundError(
        f"Environmental alignment checkpoint not found:\n{ALIGNMENT_PATH}"
    )

print("[PASSED] Project directory available.")
print("[FOUND] Environmental alignment checkpoint")
print(f"        {ALIGNMENT_PATH}")

# ------------------------------------------------------------------------------------------------
# 1. TRUSTED LOAD
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("1. LOADING ENVIRONMENTAL ALIGNMENT CHECKPOINT")
print("-" * 100)

print("[INFO] Standard weights_only=True load failed because this trusted")
print("       checkpoint contains a serialized NumPy object.")
print("[INFO] Using weights_only=False ONLY for this trusted local checkpoint.")
print("[INFO] No checkpoint will be modified.")

alignment_obj = torch.load(
    ALIGNMENT_PATH,
    map_location="cpu",
    weights_only=False
)

print("[PASSED] Environmental alignment checkpoint loaded.")
print("Object type:", type(alignment_obj))

# ------------------------------------------------------------------------------------------------
# 2. IDENTIFY STATE DICTIONARY
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. IDENTIFYING ALIGNMENT STATE DICTIONARY")
print("-" * 100)

if isinstance(alignment_obj, dict):
    print("Dictionary keys:")
    for k in alignment_obj.keys():
        print(f"  {k}")

if isinstance(alignment_obj, dict):
    if "state_dict" in alignment_obj:
        alignment_state = alignment_obj["state_dict"]
        print("\n[FOUND] state_dict")
    elif "model_state_dict" in alignment_obj:
        alignment_state = alignment_obj["model_state_dict"]
        print("\n[FOUND] model_state_dict")
    elif all(isinstance(v, torch.Tensor) for v in alignment_obj.values()):
        alignment_state = alignment_obj
        print("\n[FOUND] Checkpoint itself is a state_dict")
    else:
        raise RuntimeError(
            "Could not identify the environmental alignment state_dict."
        )
else:
    raise RuntimeError(
        f"Unexpected alignment checkpoint type: {type(alignment_obj)}"
    )

print(f"Alignment state-dict entries: {len(alignment_state)}")

# ------------------------------------------------------------------------------------------------
# 3. EXACT KEY / SHAPE INSPECTION
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. EXACT ENVIRONMENTAL ALIGNMENT STATE-DICT")
print("-" * 100)

for key, tensor in alignment_state.items():
    if isinstance(tensor, torch.Tensor):
        print(
            f"{key:<65} "
            f"shape={tuple(tensor.shape)!s:<20} "
            f"dtype={tensor.dtype}"
        )
    else:
        print(
            f"{key:<65} "
            f"type={type(tensor)}"
        )

# ------------------------------------------------------------------------------------------------
# 4. LINEAR / NORMALIZATION STRUCTURE
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. STRUCTURAL INFERENCE FROM ACTUAL CHECKPOINT")
print("-" * 100)

linear_candidates = []
norm_candidates = []

for key, tensor in alignment_state.items():

    if not isinstance(tensor, torch.Tensor):
        continue

    if key.endswith(".weight") and tensor.ndim == 2:
        out_features, in_features = tensor.shape
        linear_candidates.append(
            (key, in_features, out_features)
        )

    if (
        key.endswith(".running_mean")
        or key.endswith(".running_var")
        or key.endswith(".num_batches_tracked")
    ):
        norm_candidates.append(key)

print("\nLinear candidates:")
for key, in_features, out_features in linear_candidates:
    print(
        f"  {key:<55} "
        f"{in_features} -> {out_features}"
    )

print("\nNormalization-related keys:")
if norm_candidates:
    for key in norm_candidates:
        print(f"  {key}")
else:
    print("  [NONE]")

# ------------------------------------------------------------------------------------------------
# 5. EXPECTED COMPATIBILITY
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. ENVIRONMENTAL ALIGNMENT COMPATIBILITY")
print("-" * 100)

has_128_to_32 = any(
    in_features == 128 and out_features == 32
    for _, in_features, out_features in linear_candidates
)

if has_128_to_32:
    print("[PASSED] Actual checkpoint contains a 128 -> 32 projection.")
else:
    print("[WARNING] 128 -> 32 projection was not found.")

print("\nNo model reconstructed.")
print("No training performed.")
print("No checkpoint modified.")
print("No production artifact overwritten.")

print("\n" + "=" * 100)
print("M4.4.10.8.1 COMPLETED")
print("=" * 100)

print("""
NEXT:
M4.4.10.8.2 — EXACT ENVIRONMENTAL ALIGNMENT RECONSTRUCTION
              + strict checkpoint loading
              + environmental preprocessing resolution
              + P001-P005 environmental inference

Then immediately:
M4.4.11 — MULTIMODAL FUSION INSPECTION
M4.4.12 — FUSION RECONSTRUCTION
M4.4.13 — ANY-ONE / ANY-TWO / ALL-THREE INFERENCE
M4.4.14 — FINAL INTERFACE
""")

ASD MULTIMODAL PROJECT — M4.4.10.8.1
ENVIRONMENTAL ALIGNMENT — TRUSTED LOAD + EXACT INSPECTION
Project : /content/gdrive/MyDrive/ASD_Multimodal_Project
[PASSED] Project directory available.
[FOUND] Environmental alignment checkpoint
        /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/04_cross_modal_alignment/environmental_128_to_32_alignment.pt

1. LOADING ENVIRONMENTAL ALIGNMENT CHECKPOINT
----------------------------------------------------------------------------------------------------
[INFO] Standard weights_only=True load failed because this trusted
       checkpoint contains a serialized NumPy object.
[INFO] Using weights_only=False ONLY for this trusted local checkpoint.
[INFO] No checkpoint will be modified.
[PASSED] Environmental alignment checkpoint loaded.
Object type: <class 'dict'>

2. IDENTIFYING ALIGNMENT STATE DICTIONARY
----------------------------------------------------------------------------------------------------
Dictionary keys:
  model_s

In [ ]:
# ================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.8.2
# EXACT ENVIRONMENTAL ALIGNMENT RECONSTRUCTION
# + ENVIRONMENTAL ENCODER COMPATIBILITY
# + ENVIRONMENTAL EXPERT COMPATIBILITY
# ================================================================================================

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"

ENCODER_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "03_representation_learning",
    "environmental_encoder_128d.pt"
)

ALIGNMENT_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "04_cross_modal_alignment",
    "environmental_128_to_32_alignment.pt"
)

EXPERT_PATH = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M3_1_environmental_expert.pt"
)

DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

DEVICE = torch.device("cpu")

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.8.2")
print("EXACT ENVIRONMENTAL ALIGNMENT RECONSTRUCTION")
print("+ ENVIRONMENTAL BRANCH COMPATIBILITY VALIDATION")
print("=" * 100)

# ------------------------------------------------------------------------------------------------
# 1. VERIFY FILES
# ------------------------------------------------------------------------------------------------

for name, path in [
    ("Environmental encoder", ENCODER_PATH),
    ("Environmental alignment", ALIGNMENT_PATH),
    ("Environmental expert", EXPERT_PATH),
]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print(f"[FOUND] {name}")
    print(f"        {path}")

# ------------------------------------------------------------------------------------------------
# 2. LOAD ENCODER CHECKPOINT
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. LOADING ENVIRONMENTAL ENCODER CHECKPOINT")
print("-" * 100)

encoder_checkpoint = torch.load(
    ENCODER_PATH,
    map_location=DEVICE,
    weights_only=False
)

print("[PASSED] Environmental encoder checkpoint loaded.")
print("Checkpoint type:", type(encoder_checkpoint))

encoder_state = encoder_checkpoint["model_state_dict"]

print(f"Encoder state-dict entries: {len(encoder_state)}")

print("\nEncoder metadata:")
for key in encoder_checkpoint:
    if key != "model_state_dict":
        value = encoder_checkpoint[key]

        if isinstance(value, (list, tuple)):
            print(f"  {key}: list/tuple length={len(value)}")
        elif isinstance(value, np.ndarray):
            print(
                f"  {key}: ndarray shape={value.shape} dtype={value.dtype}"
            )
        else:
            print(f"  {key}: {value}")

# ------------------------------------------------------------------------------------------------
# 3. VERIFY ENVIRONMENTAL ENCODER DIMENSIONS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. ENVIRONMENTAL ENCODER DIMENSIONS")
print("-" * 100)

first_weight = encoder_state["encoder.0.weight"]
second_weight = encoder_state["encoder.4.weight"]

input_dim = first_weight.shape[1]
embedding_dim = first_weight.shape[0]

print(f"Input dimension       : {input_dim}")
print(f"Embedding dimension   : {embedding_dim}")

if input_dim != 31:
    raise RuntimeError(
        f"Expected environmental encoder input dimension 31, "
        f"found {input_dim}"
    )

if embedding_dim != 128:
    raise RuntimeError(
        f"Expected environmental encoder output dimension 128, "
        f"found {embedding_dim}"
    )

print("[PASSED] Environmental encoder is 31-D → 128-D.")

# ------------------------------------------------------------------------------------------------
# 4. LOAD ALIGNMENT CHECKPOINT
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. LOADING ENVIRONMENTAL ALIGNMENT CHECKPOINT")
print("-" * 100)

alignment_checkpoint = torch.load(
    ALIGNMENT_PATH,
    map_location=DEVICE,
    weights_only=False
)

alignment_state = alignment_checkpoint["model_state_dict"]

print("[PASSED] Alignment checkpoint loaded.")
print(f"Alignment state-dict entries: {len(alignment_state)}")

# ------------------------------------------------------------------------------------------------
# 5. VERIFY ALIGNMENT METADATA
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. ALIGNMENT METADATA")
print("-" * 100)

for key, value in alignment_checkpoint.items():

    if key == "model_state_dict":
        continue

    if isinstance(value, np.ndarray):
        print(
            f"{key:<25}: ndarray "
            f"shape={value.shape} dtype={value.dtype}"
        )
    elif isinstance(value, torch.Tensor):
        print(
            f"{key:<25}: Tensor "
            f"shape={tuple(value.shape)}"
        )
    else:
        print(f"{key:<25}: {value}")

# ------------------------------------------------------------------------------------------------
# 6. EXACT ALIGNMENT ARCHITECTURE
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. RECONSTRUCTING EXACT ENVIRONMENTAL ALIGNMENT")
print("-" * 100)

class EnvironmentalAlignment(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128)
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return z, reconstructed


environmental_alignment = EnvironmentalAlignment().to(DEVICE)

expected_alignment_keys = set(alignment_state.keys())
actual_alignment_keys = set(
    environmental_alignment.state_dict().keys()
)

if actual_alignment_keys != expected_alignment_keys:

    print("[ERROR] Alignment keys do not match.")

    print("\nMissing from reconstruction:")
    for key in sorted(expected_alignment_keys - actual_alignment_keys):
        print(" ", key)

    print("\nUnexpected in reconstruction:")
    for key in sorted(actual_alignment_keys - expected_alignment_keys):
        print(" ", key)

    raise RuntimeError(
        "Environmental alignment architecture does not exactly "
        "match checkpoint."
    )

print("[PASSED] Exact environmental alignment keys match.")

# ------------------------------------------------------------------------------------------------
# 7. STRICT LOAD
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. STRICT ALIGNMENT CHECKPOINT LOADING")
print("-" * 100)

environmental_alignment.load_state_dict(
    alignment_state,
    strict=True
)

environmental_alignment.eval()

print("[PASSED] Environmental alignment loaded with strict=True.")
print("[PASSED] No missing keys.")
print("[PASSED] No unexpected keys.")
print("[PASSED] Alignment set to eval().")

# ------------------------------------------------------------------------------------------------
# 8. ALIGNMENT FORWARD VALIDATION
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. ALIGNMENT FORWARD VALIDATION")
print("-" * 100)

test_128 = torch.randn(2, 128)

with torch.no_grad():
    test_32, test_reconstruction = environmental_alignment(test_128)

print(f"Input shape          : {tuple(test_128.shape)}")
print(f"Aligned shape        : {tuple(test_32.shape)}")
print(f"Reconstruction shape : {tuple(test_reconstruction.shape)}")

if test_32.shape != (2, 32):
    raise RuntimeError(
        f"Expected aligned shape (2,32), got {tuple(test_32.shape)}"
    )

if test_reconstruction.shape != (2, 128):
    raise RuntimeError(
        "Environmental alignment decoder does not reconstruct 128-D."
    )

if not torch.isfinite(test_32).all():
    raise RuntimeError("Non-finite values detected in alignment.")

print("[PASSED] 128-D → 32-D alignment validated.")
print("[PASSED] 32-D → 128-D reconstruction validated.")

# ------------------------------------------------------------------------------------------------
# 9. LOAD VERIFIED ENVIRONMENTAL EXPERT
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("9. LOADING VERIFIED ENVIRONMENTAL EXPERT")
print("-" * 100)

expert_state = torch.load(
    EXPERT_PATH,
    map_location=DEVICE,
    weights_only=True
)

class EnvironmentalExpert(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


environmental_expert = EnvironmentalExpert().to(DEVICE)

environmental_expert.load_state_dict(
    expert_state,
    strict=True
)

environmental_expert.eval()

print("[PASSED] Environmental expert loaded strictly.")
print("[PASSED] Expert ready for inference.")

# ------------------------------------------------------------------------------------------------
# 10. COMPLETE REPRESENTATION-LEVEL TEST
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("10. COMPLETE ENVIRONMENTAL REPRESENTATION PIPELINE TEST")
print("-" * 100)

dummy_environmental = torch.randn(2, 31)

# The exact encoder architecture is reconstructed from its checkpoint.
class EnvironmentalEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(31, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        z = self.encoder(x)
        return z


environmental_encoder = EnvironmentalEncoder().to(DEVICE)

environmental_encoder.load_state_dict(
    encoder_state,
    strict=True
)

environmental_encoder.eval()

with torch.no_grad():

    z128 = environmental_encoder(dummy_environmental)

    z32, reconstructed = environmental_alignment(z128)

    logits = environmental_expert(z32)

    probabilities = torch.sigmoid(logits)

print(f"Raw environmental input : {tuple(dummy_environmental.shape)}")
print(f"128-D representation    : {tuple(z128.shape)}")
print(f"32-D alignment          : {tuple(z32.shape)}")
print(f"Expert output            : {tuple(logits.shape)}")

if z128.shape != (2, 128):
    raise RuntimeError("Environmental encoder output is not 128-D.")

if z32.shape != (2, 32):
    raise RuntimeError("Environmental alignment output is not 32-D.")

if logits.shape != (2, 1):
    raise RuntimeError("Environmental expert output is not one logit.")

if not torch.isfinite(probabilities).all():
    raise RuntimeError("Environmental probability contains non-finite values.")

if not torch.all((probabilities >= 0) & (probabilities <= 1)):
    raise RuntimeError("Environmental probability outside [0,1].")

print("[PASSED] 31-D → 128-D → 32-D → 1 pipeline validated.")
print("[PASSED] Environmental probabilities numerically valid.")

# ------------------------------------------------------------------------------------------------
# 11. IMPORTANT: PREPROCESSING INFORMATION
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("11. ENVIRONMENTAL PREPROCESSING STATUS")
print("-" * 100)

feature_columns = encoder_checkpoint.get("feature_columns", None)
target_column = encoder_checkpoint.get("target_column", None)

print("Feature columns available in encoder checkpoint:",
      feature_columns is not None)

if feature_columns is not None:
    print(f"Number of feature columns: {len(feature_columns)}")
    print("\nExact feature columns:")
    for i, col in enumerate(feature_columns):
        print(f"  {i:03d}: {col}")

print(f"\nTarget column: {target_column}")

# ------------------------------------------------------------------------------------------------
# 12. DO NOT GUESS DEMO PREPROCESSING
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("12. DEMO PREPROCESSING SAFETY CHECK")
print("-" * 100)

patient_files = {}

for patient_id in ["P001", "P002", "P003", "P004", "P005"]:

    patient_dir = os.path.join(DEMO_DIR, patient_id)

    candidates = [
        os.path.join(patient_dir, "environmental_input.csv"),
        os.path.join(patient_dir, "environmental.csv"),
    ]

    found = None

    for candidate in candidates:
        if os.path.isfile(candidate):
            found = candidate
            break

    if found is not None:
        patient_files[patient_id] = found
        print(f"[FOUND] {patient_id}: {found}")
    else:
        print(f"[WARNING] {patient_id}: environmental input file not found.")

print(f"\nEnvironmental demo files resolved: "
      f"{len(patient_files)}/5")

# ------------------------------------------------------------------------------------------------
# 13. INSPECT SHAPES ONLY — NO UNSAFE TRANSFORMATION
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("13. ENVIRONMENTAL DEMO INPUT COMPATIBILITY")
print("-" * 100)

for patient_id, path in patient_files.items():

    df = pd.read_csv(path)

    print(
        f"{patient_id}: "
        f"shape={df.shape}, "
        f"columns={len(df.columns)}, "
        f"numeric={df.select_dtypes(include=np.number).shape[1]}"
    )

    if feature_columns is not None:

        missing = [
            col for col in feature_columns
            if col not in df.columns
        ]

        if missing:
            print(
                f"  [BLOCKED] Missing {len(missing)} expected "
                f"encoder feature columns."
            )
        else:
            print(
                "  [PASSED] All encoder feature columns present."
            )

print("\n" + "=" * 100)
print("M4.4.10.8.2 COMPLETED")
print("=" * 100)

print("""
[PASSED] Environmental encoder reconstructed strictly.
[PASSED] Environmental alignment reconstructed strictly.
[PASSED] Environmental expert loaded strictly.
[PASSED] 31-D → 128-D → 32-D → 1 representation pipeline validated.

IMPORTANT:
P001-P005 production probabilities are generated ONLY when the
actual saved/input preprocessing is resolved. No preprocessing
assumptions are being introduced.

NEXT:
Use the exact feature_columns / preprocessing artifacts reported
above to perform P001-P005 environmental production inference.

Then immediately proceed to:
M4.4.11 — MULTIMODAL FUSION INSPECTION
M4.4.12 — FUSION RECONSTRUCTION
M4.4.13 — ANY-ONE / ANY-TWO / ALL-THREE INFERENCE
M4.4.14 — FINAL INTERFACE
""")

ASD MULTIMODAL PROJECT — M4.4.10.8.2
EXACT ENVIRONMENTAL ALIGNMENT RECONSTRUCTION
+ ENVIRONMENTAL BRANCH COMPATIBILITY VALIDATION
[FOUND] Environmental encoder
        /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/03_representation_learning/environmental_encoder_128d.pt
[FOUND] Environmental alignment
        /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/04_cross_modal_alignment/environmental_128_to_32_alignment.pt
[FOUND] Environmental expert
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M3_1_environmental_expert.pt

2. LOADING ENVIRONMENTAL ENCODER CHECKPOINT
----------------------------------------------------------------------------------------------------
[PASSED] Environmental encoder checkpoint loaded.
Checkpoint type: <class 'dict'>
Encoder state-dict entries: 16

Encoder metadata:
  input_dim: 31
  embedding_dim: 128
  feature_columns: list/tuple length=31
  target_column: Target
  target_definition: K2Q35

RuntimeError: Error(s) in loading state_dict for EnvironmentalEncoder:
	Missing key(s) in state_dict: "encoder.3.weight", "encoder.3.bias", "encoder.4.running_mean", "encoder.4.running_var". 
	Unexpected key(s) in state_dict: "encoder.5.weight", "encoder.5.bias", "encoder.5.running_mean", "encoder.5.running_var", "encoder.5.num_batches_tracked". 
	size mismatch for encoder.4.weight: copying a param with shape torch.Size([128, 128]) from checkpoint, the shape in current model is torch.Size([128]).

In [ ]:
# ==============================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.8.3
# EXACT ENVIRONMENTAL ENCODER RECONSTRUCTION — CHECKPOINT-KEY ALIGNED
# + 31-D → 128-D → 32-D → EXPERT VALIDATION
# ==============================================================================================

import os
import glob
import json
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

DEVICE = torch.device("cpu")

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"

ENCODER_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "03_representation_learning",
    "environmental_encoder_128d.pt"
)

ALIGNMENT_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "04_cross_modal_alignment",
    "environmental_128_to_32_alignment.pt"
)

EXPERT_PATH = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M3_1_environmental_expert.pt"
)

DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.8.3")
print("EXACT ENVIRONMENTAL ENCODER RECONSTRUCTION — CHECKPOINT-KEY ALIGNED")
print("=" * 100)

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(PROJECT_DIR)

print("[PASSED] Project directory available.")

# ------------------------------------------------------------------------------------------------
# 1. LOAD ENCODER CHECKPOINT
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("1. LOADING ENVIRONMENTAL ENCODER")
print("=" * 100)

encoder_ckpt = torch.load(
    ENCODER_PATH,
    map_location=DEVICE,
    weights_only=False
)

print("[PASSED] Environmental encoder checkpoint loaded.")
print("Checkpoint type:", type(encoder_ckpt))

if "model_state_dict" not in encoder_ckpt:
    raise RuntimeError("model_state_dict not found in environmental encoder checkpoint.")

encoder_state = encoder_ckpt["model_state_dict"]

print("Encoder state-dict entries:", len(encoder_state))

for k, v in encoder_state.items():
    print(f"{k:60s} {tuple(v.shape)}")

# ------------------------------------------------------------------------------------------------
# 2. VERIFY CHECKPOINT-DERIVED METADATA
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. CHECKPOINT METADATA")
print("=" * 100)

INPUT_DIM = int(encoder_ckpt["input_dim"])
EMBED_DIM = int(encoder_ckpt["embedding_dim"])

FEATURE_COLUMNS = list(encoder_ckpt["feature_columns"])

print("Input dimension       :", INPUT_DIM)
print("Embedding dimension   :", EMBED_DIM)
print("Feature columns       :", len(FEATURE_COLUMNS))
print("Target column         :", encoder_ckpt.get("target_column"))
print("Target definition     :", encoder_ckpt.get("target_definition"))

if INPUT_DIM != 31:
    raise RuntimeError(f"Expected checkpoint input_dim=31, found {INPUT_DIM}")

if EMBED_DIM != 128:
    raise RuntimeError(f"Expected checkpoint embedding_dim=128, found {EMBED_DIM}")

if len(FEATURE_COLUMNS) != 31:
    raise RuntimeError(
        f"Checkpoint feature_columns length is {len(FEATURE_COLUMNS)}, expected 31."
    )

print("[PASSED] Actual checkpoint metadata confirms 31-D → 128-D.")

# ------------------------------------------------------------------------------------------------
# 3. EXACT CHECKPOINT-KEY ALIGNED ARCHITECTURE
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. RECONSTRUCTING EXACT ENVIRONMENTAL ENCODER")
print("=" * 100)

class EnvironmentalEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        # IMPORTANT:
        # These indices are preserved because the checkpoint keys prove them.
        self.encoder = nn.Sequential(
            nn.Linear(31, 128),      # encoder.0
            nn.BatchNorm1d(128),     # encoder.1
            nn.ReLU(),              # encoder.2
            nn.Dropout(),           # encoder.3
            nn.Linear(128, 128),     # encoder.4
            nn.BatchNorm1d(128),     # encoder.5
            nn.ReLU(),              # encoder.6
            nn.Dropout(),           # encoder.7
        )

        self.classifier = nn.Linear(128, 1)

    def forward(self, x, return_embedding=False):
        z = self.encoder(x)

        if return_embedding:
            return z

        return self.classifier(z)


environmental_encoder = EnvironmentalEncoder().to(DEVICE)

print("[PASSED] EnvironmentalEncoder created.")

# ------------------------------------------------------------------------------------------------
# 4. EXACT KEY VALIDATION BEFORE LOADING
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. EXACT STATE-DICT KEY VALIDATION")
print("=" * 100)

model_keys = list(environmental_encoder.state_dict().keys())
checkpoint_keys = list(encoder_state.keys())

missing_before = sorted(set(model_keys) - set(checkpoint_keys))
unexpected_before = sorted(set(checkpoint_keys) - set(model_keys))

print("Model keys      :", len(model_keys))
print("Checkpoint keys :", len(checkpoint_keys))

if missing_before:
    print("[ERROR] Missing checkpoint keys:")
    for k in missing_before:
        print(" ", k)

if unexpected_before:
    print("[ERROR] Unexpected checkpoint keys:")
    for k in unexpected_before:
        print(" ", k)

if missing_before or unexpected_before:
    raise RuntimeError(
        "Environmental encoder namespace still does not exactly match checkpoint."
    )

print("[PASSED] Encoder namespaces exactly match.")

# ------------------------------------------------------------------------------------------------
# 5. EXACT SHAPE VALIDATION
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. EXACT PARAMETER SHAPE VALIDATION")
print("=" * 100)

for key in checkpoint_keys:

    checkpoint_shape = tuple(encoder_state[key].shape)
    model_shape = tuple(environmental_encoder.state_dict()[key].shape)

    if checkpoint_shape != model_shape:
        raise RuntimeError(
            f"Shape mismatch for {key}: "
            f"checkpoint={checkpoint_shape}, model={model_shape}"
        )

    print(f"[PASSED] {key:55s} {model_shape}")

# ------------------------------------------------------------------------------------------------
# 6. STRICT LOAD
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. STRICT CHECKPOINT LOADING")
print("=" * 100)

environmental_encoder.load_state_dict(
    encoder_state,
    strict=True
)

environmental_encoder.eval()

print("[PASSED] Environmental encoder loaded with strict=True.")
print("[PASSED] No missing keys.")
print("[PASSED] No unexpected keys.")
print("[PASSED] Environmental encoder set to eval().")

# ------------------------------------------------------------------------------------------------
# 7. FORWARD VALIDATION
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. ENVIRONMENTAL ENCODER FORWARD VALIDATION")
print("=" * 100)

test_input = torch.randn(2, 31)

with torch.no_grad():
    test_embedding = environmental_encoder(
        test_input,
        return_embedding=True
    )

    test_logit = environmental_encoder(test_input)

print("Input shape       :", tuple(test_input.shape))
print("Embedding shape   :", tuple(test_embedding.shape))
print("Classifier output :", tuple(test_logit.shape))

if tuple(test_embedding.shape) != (2, 128):
    raise RuntimeError("Environmental encoder did not produce 128-D representation.")

if tuple(test_logit.shape) != (2, 1):
    raise RuntimeError("Environmental classifier did not produce one logit.")

if not torch.isfinite(test_embedding).all():
    raise RuntimeError("Environmental embedding contains non-finite values.")

print("[PASSED] 31-D input accepted.")
print("[PASSED] Exact 128-D environmental representation produced.")
print("[PASSED] Environmental representation is finite.")

# ------------------------------------------------------------------------------------------------
# 8. LOAD ALIGNMENT
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. LOADING VERIFIED ENVIRONMENTAL ALIGNMENT")
print("=" * 100)

alignment_ckpt = torch.load(
    ALIGNMENT_PATH,
    map_location=DEVICE,
    weights_only=False
)

alignment_state = alignment_ckpt["model_state_dict"]

print("[PASSED] Environmental alignment loaded.")
print("Alignment keys:", len(alignment_state))

class EnvironmentalAlignment(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128)
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        return z, reconstruction

environmental_alignment = EnvironmentalAlignment().to(DEVICE)

environmental_alignment.load_state_dict(
    alignment_state,
    strict=True
)

environmental_alignment.eval()

print("[PASSED] Environmental alignment loaded with strict=True.")
print("[PASSED] 128 → 64 → 32 encoder validated.")
print("[PASSED] 32 → 64 → 128 decoder validated.")

# ------------------------------------------------------------------------------------------------
# 9. LOAD ENVIRONMENTAL EXPERT
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("9. LOADING VERIFIED ENVIRONMENTAL EXPERT")
print("=" * 100)

expert_state = torch.load(
    EXPERT_PATH,
    map_location=DEVICE,
    weights_only=True
)

class EnvironmentalExpert(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

environmental_expert = EnvironmentalExpert().to(DEVICE)

environmental_expert.load_state_dict(
    expert_state,
    strict=True
)

environmental_expert.eval()

print("[PASSED] Environmental expert loaded strictly.")

# ------------------------------------------------------------------------------------------------
# 10. COMPLETE REPRESENTATION PIPELINE TEST
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("10. COMPLETE ENVIRONMENTAL PIPELINE TEST")
print("=" * 100)

with torch.no_grad():

    embedding_128 = environmental_encoder(
        test_input,
        return_embedding=True
    )

    aligned_32, reconstructed_128 = environmental_alignment(
        embedding_128
    )

    environmental_logit = environmental_expert(
        aligned_32
    )

    environmental_probability = torch.sigmoid(
        environmental_logit
    )

print("Raw input       :", tuple(test_input.shape))
print("128-D embedding :", tuple(embedding_128.shape))
print("32-D alignment  :", tuple(aligned_32.shape))
print("Expert output   :", tuple(environmental_logit.shape))

print(
    "Probabilities   :",
    environmental_probability.squeeze(1).numpy()
)

if tuple(embedding_128.shape) != (2, 128):
    raise RuntimeError("128-D representation validation failed.")

if tuple(aligned_32.shape) != (2, 32):
    raise RuntimeError("32-D alignment validation failed.")

if tuple(environmental_logit.shape) != (2, 1):
    raise RuntimeError("Environmental expert output validation failed.")

if not torch.isfinite(environmental_probability).all():
    raise RuntimeError("Environmental probability contains non-finite values.")

print("[PASSED] 31-D → 128-D → 32-D → 1 pipeline works.")
print("[PASSED] Environmental probability numerically valid.")

# ------------------------------------------------------------------------------------------------
# 11. DISCOVER PREPROCESSING ARTIFACTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("11. ENVIRONMENTAL PREPROCESSING ARTIFACT DISCOVERY")
print("=" * 100)

patterns = [
    "**/*environmental*.joblib",
    "**/*environment*.joblib",
    "**/*environmental*.pkl",
    "**/*environment*.pkl",
    "**/*scaler*.joblib",
    "**/*preprocessor*.joblib",
    "**/*scaler*.pkl",
    "**/*preprocessor*.pkl",
]

preprocess_candidates = []

for pattern in patterns:
    preprocess_candidates.extend(
        glob.glob(
            os.path.join(PROJECT_DIR, pattern),
            recursive=True
        )
    )

preprocess_candidates = sorted(
    set(preprocess_candidates)
)

print("Preprocessing candidates:", len(preprocess_candidates))

for path in preprocess_candidates:
    print(" ", path)

# ------------------------------------------------------------------------------------------------
# 12. DISCOVER DEMO ENVIRONMENTAL INPUTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("12. ENVIRONMENTAL DEMO INPUT DISCOVERY")
print("=" * 100)

patient_inputs = {}

for patient_id in ["P001", "P002", "P003", "P004", "P005"]:

    patient_dir = os.path.join(
        DEMO_DIR,
        patient_id
    )

    candidates = []

    for pattern in [
        "*environment*.csv",
        "*environmental*.csv",
        "environmental_input.csv",
        "environment_input.csv",
    ]:
        candidates.extend(
            glob.glob(
                os.path.join(patient_dir, pattern)
            )
        )

    candidates = sorted(set(candidates))

    if candidates:
        patient_inputs[patient_id] = candidates[0]
        print(
            f"[FOUND] {patient_id}: {candidates[0]}"
        )
    else:
        print(
            f"[MISSING] {patient_id}: environmental CSV not found"
        )

print(
    "\nPatients resolved:",
    f"{len(patient_inputs)}/5"
)

# ------------------------------------------------------------------------------------------------
# 13. PRINT FEATURE-COLUMN REQUIREMENTS
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("13. VERIFIED ENVIRONMENTAL FEATURE SPACE")
print("=" * 100)

print("Required raw feature count:", len(FEATURE_COLUMNS))

for i, col in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{i:03d}. {col}")

# ------------------------------------------------------------------------------------------------
# 14. SAVE VERIFIED OBJECTS IN MEMORY
# ------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("14. ENVIRONMENTAL BRANCH STATUS")
print("=" * 100)

print("[PASSED] Exact environmental encoder reconstructed.")
print("[PASSED] Strict encoder checkpoint loading succeeded.")
print("[PASSED] 31-D → 128-D representation validated.")
print("[PASSED] Exact environmental alignment loaded.")
print("[PASSED] 128-D → 32-D alignment validated.")
print("[PASSED] Exact environmental expert loaded.")
print("[PASSED] 32-D → 1 expert inference validated.")
print("[PASSED] Existing preprocessing artifacts searched.")
print("[PASSED] Existing P001-P005 environmental inputs searched.")

print("\n" + "=" * 100)
print("M4.4.10.8.3 COMPLETED")
print("=" * 100)

print("""
NO TRAINING PERFORMED.
NO CHECKPOINT MODIFIED.
NO PRODUCTION ARTIFACT OVERWRITTEN.

VERIFIED ENVIRONMENTAL PIPELINE:

RAW ENVIRONMENTAL INPUT
        ↓
31-D
        ↓
EXACT ENVIRONMENTAL ENCODER
        ↓
128-D
        ↓
EXACT ENVIRONMENTAL ALIGNMENT
        ↓
32-D
        ↓
EXACT ENVIRONMENTAL EXPERT
        ↓
1 ASD LOGIT
        ↓
SIGMOID
        ↓
ASD PROBABILITY

NEXT:
Use the printed preprocessing candidates and demo-input structure
to perform P001-P005 environmental production inference.

Then immediately inspect/reconstruct multimodal fusion.
""")


ASD MULTIMODAL PROJECT — M4.4.10.8.3
EXACT ENVIRONMENTAL ENCODER RECONSTRUCTION — CHECKPOINT-KEY ALIGNED
[PASSED] Project directory available.

1. LOADING ENVIRONMENTAL ENCODER
[PASSED] Environmental encoder checkpoint loaded.
Checkpoint type: <class 'dict'>
Encoder state-dict entries: 16
encoder.0.weight                                             (128, 31)
encoder.0.bias                                               (128,)
encoder.1.weight                                             (128,)
encoder.1.bias                                               (128,)
encoder.1.running_mean                                       (128,)
encoder.1.running_var                                        (128,)
encoder.1.num_batches_tracked                                ()
encoder.4.weight                                             (128, 128)
encoder.4.bias                                               (128,)
encoder.5.weight                                             (128,)
encoder.5.bias             

In [ ]:
# ==============================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.9
# ENVIRONMENTAL P001-P005 PRODUCTION INFERENCE
# ARTIFACT-FIRST — NO PREPROCESSING GUESSING
# ==============================================================================================

import os
import glob
import joblib
import numpy as np
import pandas as pd
import torch

DEVICE = torch.device("cpu")

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

ENCODER_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "03_representation_learning",
    "environmental_encoder_128d.pt"
)

ALIGNMENT_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "04_cross_modal_alignment",
    "environmental_128_to_32_alignment.pt"
)

EXPERT_PATH = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M3_1_environmental_expert.pt"
)

PATIENTS = ["P001", "P002", "P003", "P004", "P005"]

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.9")
print("ENVIRONMENTAL P001-P005 PRODUCTION INFERENCE")
print("=" * 100)

# ==============================================================================================
# 1. LOAD VERIFIED ENCODER
# ==============================================================================================

encoder_ckpt = torch.load(
    ENCODER_PATH,
    map_location=DEVICE,
    weights_only=False
)

encoder_state = encoder_ckpt["model_state_dict"]
FEATURE_COLUMNS = list(encoder_ckpt["feature_columns"])

# Reuse exact architecture already proven by checkpoint
import torch.nn as nn

class EnvironmentalEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(31, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(),
        )

        self.classifier = nn.Linear(128, 1)

    def forward(self, x, return_embedding=False):
        z = self.encoder(x)

        if return_embedding:
            return z

        return self.classifier(z)

environmental_encoder = EnvironmentalEncoder().to(DEVICE)
environmental_encoder.load_state_dict(encoder_state, strict=True)
environmental_encoder.eval()

# ==============================================================================================
# 2. LOAD EXACT ALIGNMENT
# ==============================================================================================

alignment_ckpt = torch.load(
    ALIGNMENT_PATH,
    map_location=DEVICE,
    weights_only=False
)

alignment_state = alignment_ckpt["model_state_dict"]

class EnvironmentalAlignment(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128)
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        return z, reconstruction

environmental_alignment = EnvironmentalAlignment().to(DEVICE)
environmental_alignment.load_state_dict(alignment_state, strict=True)
environmental_alignment.eval()

# ==============================================================================================
# 3. LOAD EXACT EXPERT
# ==============================================================================================

expert_state = torch.load(
    EXPERT_PATH,
    map_location=DEVICE,
    weights_only=True
)

class EnvironmentalExpert(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

environmental_expert = EnvironmentalExpert().to(DEVICE)
environmental_expert.load_state_dict(expert_state, strict=True)
environmental_expert.eval()

print("[PASSED] Verified environmental encoder loaded.")
print("[PASSED] Verified environmental alignment loaded.")
print("[PASSED] Verified environmental expert loaded.")

# ==============================================================================================
# 4. DISCOVER ACTUAL ENVIRONMENTAL INPUT FILES
# ==============================================================================================

print("\n" + "=" * 100)
print("1. RESOLVING ENVIRONMENTAL INPUT FILES")
print("=" * 100)

environmental_files = {}

for patient in PATIENTS:

    patient_dir = os.path.join(DEMO_DIR, patient)

    candidates = []

    for pattern in [
        "*environmental*.csv",
        "*environment*.csv",
        "environmental_input.csv",
        "environment_input.csv",
    ]:
        candidates.extend(
            glob.glob(
                os.path.join(patient_dir, pattern)
            )
        )

    candidates = sorted(set(candidates))

    if not candidates:
        print(f"[MISSING] {patient}")
        continue

    environmental_files[patient] = candidates[0]

    print(f"[FOUND] {patient}")
    print(f"        {candidates[0]}")

if len(environmental_files) != 5:
    raise RuntimeError(
        f"Only {len(environmental_files)}/5 environmental inputs resolved."
    )

# ==============================================================================================
# 5. INSPECT INPUT COLUMNS
# ==============================================================================================

print("\n" + "=" * 100)
print("2. VALIDATING ENVIRONMENTAL INPUT SCHEMA")
print("=" * 100)

environmental_data = {}

for patient in PATIENTS:

    path = environmental_files[patient]

    df = pd.read_csv(path)

    environmental_data[patient] = df

    print(
        f"{patient}: shape={df.shape}, "
        f"columns={len(df.columns)}, "
        f"numeric={df.select_dtypes(include=np.number).shape[1]}"
    )

    missing = [
        c for c in FEATURE_COLUMNS
        if c not in df.columns
    ]

    if missing:
        print(f"[MISSING FEATURES] {patient}:")
        for c in missing:
            print(" ", c)

    else:
        print(
            f"[PASSED] {patient}: all 31 verified environmental "
            f"features are present."
        )

# ==============================================================================================
# 6. CHECK WHETHER RAW 31-D INPUT CAN BE USED DIRECTLY
# ==============================================================================================

print("\n" + "=" * 100)
print("3. DETERMINING VERIFIED INPUT REPRESENTATION")
print("=" * 100)

all_features_present = all(
    all(c in environmental_data[p].columns for c in FEATURE_COLUMNS)
    for p in PATIENTS
)

if not all_features_present:
    raise RuntimeError(
        "Environmental inputs do not contain the exact 31 checkpoint "
        "feature columns. STOPPING rather than guessing mappings."
    )

print("[PASSED] Exact 31 environmental feature columns available.")

# ==============================================================================================
# 7. NUMERIC VALIDATION
# ==============================================================================================

print("\n" + "=" * 100)
print("4. NUMERIC FEATURE VALIDATION")
print("=" * 100)

raw_rows = {}

for patient in PATIENTS:

    df = environmental_data[patient]

    row = df[FEATURE_COLUMNS].copy()

    # Convert using pandas numeric conversion ONLY.
    # No invented scaling/imputation.
    numeric_row = row.apply(pd.to_numeric, errors="coerce")

    if numeric_row.isna().any().any():

        bad_columns = list(
            numeric_row.columns[
                numeric_row.isna().any()
            ]
        )

        raise RuntimeError(
            f"{patient} contains non-numeric/missing values in "
            f"verified features: {bad_columns}"
        )

    raw_rows[patient] = numeric_row

    print(
        f"[PASSED] {patient}: "
        f"31 numeric features ready."
    )

# ==============================================================================================
# 8. ENVIRONMENTAL PRODUCTION INFERENCE
# ==============================================================================================

print("\n" + "=" * 100)
print("5. P001-P005 ENVIRONMENTAL PRODUCTION INFERENCE")
print("=" * 100)

results = []

for patient in PATIENTS:

    print(f"\n--- {patient} ---")

    raw_x = raw_rows[patient].to_numpy(
        dtype=np.float32
    )

    if raw_x.shape != (1, 31):
        raise RuntimeError(
            f"{patient}: expected (1,31), got {raw_x.shape}"
        )

    x = torch.tensor(
        raw_x,
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():

        embedding_128 = environmental_encoder(
            x,
            return_embedding=True
        )

        aligned_32, reconstructed_128 = (
            environmental_alignment(
                embedding_128
            )
        )

        logit = environmental_expert(
            aligned_32
        )

        probability = torch.sigmoid(
            logit
        ).item()

    prediction = (
        "ASD"
        if probability >= 0.5
        else "NON-ASD"
    )

    print("Raw input              :", tuple(x.shape))
    print("128-D representation   :", tuple(embedding_128.shape))
    print("32-D aligned           :", tuple(aligned_32.shape))
    print(f"ASD probability        : {probability:.6f}")
    print(f"Prediction             : {prediction}")

    results.append({
        "patient_id": patient,
        "environmental_probability": probability,
        "environmental_prediction": prediction,
        "environmental_prediction_numeric": int(
            probability >= 0.5
        )
    })

# ==============================================================================================
# 9. RESULTS TABLE
# ==============================================================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 100)
print("6. ENVIRONMENTAL RESULTS — P001-P005")
print("=" * 100)

print(
    results_df.to_string(index=False)
)

# ==============================================================================================
# 10. FINAL STATUS
# ==============================================================================================

print("\n" + "=" * 100)
print("M4.4.10.9 COMPLETED")
print("=" * 100)

print("""
[PASSED] Environmental input files resolved.
[PASSED] Exact 31-D environmental feature space validated.
[PASSED] No feature mapping was invented.
[PASSED] No preprocessing was invented.
[PASSED] Exact environmental encoder used.
[PASSED] Exact 128-D representation generated.
[PASSED] Exact 128 → 32 alignment used.
[PASSED] Exact environmental expert used.
[PASSED] P001-P005 environmental inference completed.

NO TRAINING PERFORMED.
NO CHECKPOINT MODIFIED.
NO PRODUCTION ARTIFACT OVERWRITTEN.

NEXT:
M4.4.11 — MULTIMODAL FUSION INSPECTION
""")

ASD MULTIMODAL PROJECT — M4.4.10.9
ENVIRONMENTAL P001-P005 PRODUCTION INFERENCE
[PASSED] Verified environmental encoder loaded.
[PASSED] Verified environmental alignment loaded.
[PASSED] Verified environmental expert loaded.

1. RESOLVING ENVIRONMENTAL INPUT FILES
[FOUND] P001
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P001/environmental_input.csv
[FOUND] P002
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P002/environmental_input.csv
[FOUND] P003
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P003/environmental_input.csv
[FOUND] P004
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P004/environmental_input.csv
[FOUND] P005
        /content/gdrive/MyDrive/ASD_Multimodal_Project/05_multimodal_fusion/M4_demo_patients/P005/environmental_input.csv

2. VALIDATING ENVIRONMENTAL INPUT SCHEMA
P001: shape=(1, 32

RuntimeError: P001 contains non-numeric/missing values in verified features: ['mpc_yn']

In [ ]:
# ================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.9.1
# ENVIRONMENTAL CATEGORICAL ENCODING RESOLUTION
# EXACT SOURCE/PREPROCESSOR BASED — NO VALUE GUESSING
# ================================================================================================

import os
import re
import json
import joblib
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"
DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

ENCODER_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "03_representation_learning",
    "environmental_encoder_128d.pt"
)

SOURCE_NOTEBOOK = "/content/gdrive/MyDrive/Colab Notebooks/Untitled0.ipynb"

PATIENTS = ["P001", "P002", "P003", "P004", "P005"]

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.9.1")
print("ENVIRONMENTAL CATEGORICAL ENCODING RESOLUTION")
print("=" * 100)

# ------------------------------------------------------------------------------------------------
# 1. VERIFY PROJECT
# ------------------------------------------------------------------------------------------------

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(PROJECT_DIR)

print("[PASSED] Project directory available.")

# ------------------------------------------------------------------------------------------------
# 2. LOAD VERIFIED ENCODER METADATA
# ------------------------------------------------------------------------------------------------

checkpoint = torch.load(
    ENCODER_PATH,
    map_location="cpu",
    weights_only=False
)

feature_columns = list(checkpoint["feature_columns"])

print()
print("=" * 100)
print("1. VERIFIED ENVIRONMENTAL FEATURE ORDER")
print("=" * 100)

for i, col in enumerate(feature_columns, 1):
    print(f"{i:02d}. {col}")

print()
print(f"[PASSED] Encoder expects exactly {len(feature_columns)} features.")

# ------------------------------------------------------------------------------------------------
# 3. LOAD P001-P005 INPUTS
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("2. INSPECTING mpc_yn VALUES")
print("=" * 100)

patient_frames = {}

for patient in PATIENTS:

    path = os.path.join(
        DEMO_DIR,
        patient,
        "environmental_input.csv"
    )

    if not os.path.isfile(path):
        raise FileNotFoundError(f"{patient}: {path}")

    df = pd.read_csv(path)

    patient_frames[patient] = df

    print(f"\n{patient}")
    print(f"Shape: {df.shape}")

    if "mpc_yn" not in df.columns:
        raise RuntimeError(f"{patient}: mpc_yn column missing.")

    value = df.loc[0, "mpc_yn"]

    print(f"mpc_yn raw value : {repr(value)}")
    print(f"Python type      : {type(value).__name__}")

# ------------------------------------------------------------------------------------------------
# 4. DISCOVER PREPROCESSING ARTIFACTS
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("3. SEARCHING ENVIRONMENTAL PREPROCESSING ARTIFACTS")
print("=" * 100)

candidate_files = []

for root, dirs, files in os.walk(PROJECT_DIR):

    # Avoid unrelated caches
    dirs[:] = [
        d for d in dirs
        if d not in {".git", "__pycache__", ".ipynb_checkpoints"}
    ]

    for file in files:

        lower = file.lower()

        if any(x in lower for x in [
            "preprocess",
            "preprocessor",
            "transformer",
            "encoder"
        ]):

            if lower.endswith((
                ".joblib",
                ".pkl",
                ".pickle"
            )):

                candidate_files.append(
                    os.path.join(root, file)
                )

print(f"Candidate preprocessing objects: {len(candidate_files)}")

for path in candidate_files:
    print(" ", path)

# ------------------------------------------------------------------------------------------------
# 5. INSPECT SKLEARN OBJECTS FOR mpc_yn
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("4. SEARCHING SAVED OBJECTS FOR mpc_yn ENCODING")
print("=" * 100)

encoding_evidence = []

for path in candidate_files:

    try:

        obj = joblib.load(path)

        print()
        print("[CHECK]", path)
        print("Type:", type(obj))

        # ----------------------------------------------------------------------
        # ColumnTransformer
        # ----------------------------------------------------------------------

        if hasattr(obj, "transformers_"):

            for name, transformer, columns in obj.transformers_:

                try:
                    cols = list(columns)
                except Exception:
                    cols = []

                if "mpc_yn" in cols:

                    print(
                        "[FOUND] mpc_yn handled by transformer:",
                        name
                    )

                    print(
                        "Transformer:",
                        type(transformer)
                    )

                    encoding_evidence.append(
                        (path, name, transformer)
                    )

                    # OneHotEncoder
                    if hasattr(transformer, "categories_"):

                        for idx, col in enumerate(cols):

                            if col == "mpc_yn":

                                cats = transformer.categories_[idx]

                                print(
                                    "[FOUND] OneHotEncoder categories:",
                                    cats
                                )

                    # OrdinalEncoder
                    if hasattr(transformer, "categories_"):

                        print(
                            "[INFO] Transformer categories:",
                            transformer.categories_
                        )

        # ----------------------------------------------------------------------
        # Pipeline
        # ----------------------------------------------------------------------

        if hasattr(obj, "steps"):

            for step_name, step in obj.steps:

                if hasattr(step, "categories_"):

                    print(
                        "[FOUND] Encoder step:",
                        step_name
                    )

                    print(
                        "Categories:",
                        step.categories_
                    )

    except Exception as e:

        print(
            "[SKIP] Could not inspect:",
            path,
            "|",
            repr(e)
        )

# ------------------------------------------------------------------------------------------------
# 6. SEARCH ORIGINAL NOTEBOOK FOR mpc_yn
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("5. SEARCHING ORIGINAL SOURCE FOR mpc_yn")
print("=" * 100)

source_hits = []

if os.path.isfile(SOURCE_NOTEBOOK):

    with open(
        SOURCE_NOTEBOOK,
        "r",
        encoding="utf-8"
    ) as f:

        notebook = json.load(f)

    for cell_idx, cell in enumerate(
        notebook.get("cells", [])
    ):

        source = "".join(
            cell.get("source", [])
        )

        if "mpc_yn" in source:

            source_hits.append(
                (cell_idx, source)
            )

            print()
            print("-" * 100)
            print(f"[FOUND] Notebook cell {cell_idx}")
            print("-" * 100)

            lines = source.splitlines()

            for line_no, line in enumerate(lines, 1):

                if "mpc_yn" in line.lower():

                    print(
                        f"{line_no:04d}: {line[:500]}"
                    )

else:

    print("[WARNING] Original notebook not found.")

print()
print(
    f"[INFO] Notebook cells containing mpc_yn: {len(source_hits)}"
)

# ------------------------------------------------------------------------------------------------
# 7. SHOW UNIQUE RAW VALUES
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("6. RAW mpc_yn VALUE DISTRIBUTION")
print("=" * 100)

all_values = []

for patient, df in patient_frames.items():

    value = df.loc[0, "mpc_yn"]

    all_values.append(value)

    print(
        f"{patient}: {repr(value)} "
        f"({type(value).__name__})"
    )

unique_values = pd.Series(all_values).unique()

print()
print("Unique demo values:")
for value in unique_values:
    print(" ", repr(value))

# ------------------------------------------------------------------------------------------------
# 8. DECISION
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("7. ENCODING DECISION")
print("=" * 100)

# We intentionally do NOT guess here.
#
# The next production cell will use the exact mapping discovered from:
#   1. Saved preprocessing object, OR
#   2. Original training source.
#
# If neither contains the mapping, stop.

if encoding_evidence:

    print("[PASSED] Saved preprocessing contains mpc_yn encoding evidence.")
    print("[PASSED] No manual categorical mapping required.")

elif source_hits:

    print("[PASSED] Original source contains mpc_yn processing evidence.")
    print("[PASSED] Source can be used to reproduce the original encoding.")

else:

    print("[BLOCKED] Original mpc_yn encoding could not yet be proven.")
    print()
    print("DO NOT manually convert mpc_yn to 0/1.")
    print("DO NOT continue production inference until the encoding is established.")

    raise RuntimeError(
        "Exact mpc_yn encoding is not proven by saved preprocessing or source."
    )

print()
print("=" * 100)
print("M4.4.10.9.1 COMPLETED")
print("=" * 100)

print("[PASSED] Environmental input files resolved.")
print("[PASSED] mpc_yn raw values inspected.")
print("[PASSED] Saved preprocessing searched.")
print("[PASSED] Original source searched.")
print("[PASSED] Encoding evidence located.")
print()
print("NEXT:")
print("M4.4.10.9.2 — APPLY EXACT ENVIRONMENTAL ENCODING")
print("                  ↓")
print("              P001-P005 inference")
print("                  ↓")
print("M4.4.11 — MULTIMODAL FUSION INSPECTION")
print("                  ↓")
print("M4.4.12 — FUSION RECONSTRUCTION")
print("                  ↓")
print("M4.4.13 — ANY-ONE / ANY-TWO / ALL-THREE")
print("                  ↓")
print("M4.4.14 — FINAL INTERFACE")
print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.9.1
ENVIRONMENTAL CATEGORICAL ENCODING RESOLUTION
[PASSED] Project directory available.

1. VERIFIED ENVIRONMENTAL FEATURE ORDER
01. fpl_i1
02. higrade
03. family_r
04. house_gen
05. metro_yn
06. mpc_yn
07. currins
08. insgap
09. foodsit
10. k11q61
11. k10q40_r
12. k10q30
13. k10q23
14. k10q22
15. k10q20
16. k10q11
17. k10q12
18. k10q13
19. k10q14
20. ace1
21. ace3
22. ace4
23. ace5
24. ace6
25. ace7
26. ace8
27. ace9
28. ace10
29. hoursleep
30. physactiv
31. screentime

[PASSED] Encoder expects exactly 31 features.

2. INSPECTING mpc_yn VALUES

P001
Shape: (1, 32)
mpc_yn raw value : np.float64(nan)
Python type      : float64

P002
Shape: (1, 32)
mpc_yn raw value : np.float64(1.0)
Python type      : float64

P003
Shape: (1, 32)
mpc_yn raw value : np.float64(2.0)
Python type      : float64

P004
Shape: (1, 32)
mpc_yn raw value : np.float64(2.0)
Python type      : float64

P005
Shape: (1, 32)
mpc_yn raw value : np.float64(1.0)
Python type      : float64



RuntimeError: Exact mpc_yn encoding is not proven by saved preprocessing or source.

In [ ]:
# ================================================================================================
# ASD MULTIMODAL PROJECT — M4.4.10.9.2
# ENVIRONMENTAL PRODUCTION INFERENCE — USE SAVED PREPROCESSOR
# ================================================================================================

import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

PROJECT_DIR = "/content/gdrive/MyDrive/ASD_Multimodal_Project"

DEMO_DIR = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M4_demo_patients"
)

PREPROCESSOR_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "02_preprocessing",
    "environmental_preprocessor.joblib"
)

ENCODER_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "03_representation_learning",
    "environmental_encoder_128d.pt"
)

ALIGNMENT_PATH = os.path.join(
    PROJECT_DIR,
    "Environmental_Model",
    "04_cross_modal_alignment",
    "environmental_128_to_32_alignment.pt"
)

EXPERT_PATH = os.path.join(
    PROJECT_DIR,
    "05_multimodal_fusion",
    "M3_1_environmental_expert.pt"
)

PATIENTS = ["P001", "P002", "P003", "P004", "P005"]

DEVICE = torch.device("cpu")

print("=" * 100)
print("ASD MULTIMODAL PROJECT — M4.4.10.9.2")
print("ENVIRONMENTAL PRODUCTION INFERENCE — SAVED PREPROCESSOR")
print("=" * 100)

# ------------------------------------------------------------------------------------------------
# 1. LOAD SAVED PREPROCESSOR
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("1. LOADING VERIFIED ENVIRONMENTAL PREPROCESSOR")
print("=" * 100)

preprocessor = joblib.load(PREPROCESSOR_PATH)

print("[FOUND]", PREPROCESSOR_PATH)
print("Type:", type(preprocessor))

if not hasattr(preprocessor, "transform"):
    raise RuntimeError("Saved environmental preprocessing object has no transform() method.")

print("[PASSED] Saved preprocessing object is transform-capable.")

# Print pipeline structure
if hasattr(preprocessor, "steps"):
    print()
    print("Pipeline steps:")
    for name, step in preprocessor.steps:
        print(f"  {name}: {type(step)}")

# ------------------------------------------------------------------------------------------------
# 2. LOAD ENCODER CHECKPOINT
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("2. LOADING VERIFIED ENVIRONMENTAL ENCODER")
print("=" * 100)

encoder_ckpt = torch.load(
    ENCODER_PATH,
    map_location=DEVICE,
    weights_only=False
)

encoder_state = encoder_ckpt["model_state_dict"]

print("[PASSED] Encoder checkpoint loaded.")
print("Input dimension :", encoder_ckpt["input_dim"])
print("Embedding dim   :", encoder_ckpt["embedding_dim"])

feature_columns = list(encoder_ckpt["feature_columns"])

print("Feature count   :", len(feature_columns))

# ------------------------------------------------------------------------------------------------
# 3. EXACT ENVIRONMENTAL ENCODER
# ------------------------------------------------------------------------------------------------

class EnvironmentalEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(31, 128),       # 0
            nn.BatchNorm1d(128),     # 1
            nn.ReLU(),               # 2
            nn.Dropout(),            # 3
            nn.Linear(128, 128),     # 4
            nn.BatchNorm1d(128),     # 5
            nn.ReLU(),               # 6
            nn.Dropout(),            # 7
        )

        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        z = self.encoder(x)
        return z


environmental_encoder = EnvironmentalEncoder().to(DEVICE)

environmental_encoder.load_state_dict(
    encoder_state,
    strict=True
)

environmental_encoder.eval()

print("[PASSED] Environmental encoder loaded strictly.")
print("[PASSED] Encoder state-dict exactly matches checkpoint.")

# ------------------------------------------------------------------------------------------------
# 4. EXACT ENVIRONMENTAL ALIGNMENT
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("3. LOADING VERIFIED ENVIRONMENTAL ALIGNMENT")
print("=" * 100)

alignment_ckpt = torch.load(
    ALIGNMENT_PATH,
    map_location=DEVICE,
    weights_only=False
)

alignment_state = alignment_ckpt["model_state_dict"]

class EnvironmentalAlignment(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(128, 64),      # 0
            nn.BatchNorm1d(64),      # 1
            nn.ReLU(),              # 2
            nn.Linear(64, 32),       # 3
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),       # 0
            nn.BatchNorm1d(64),      # 1
            nn.ReLU(),              # 2
            nn.Linear(64, 128),      # 3
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return z, reconstructed


environmental_alignment = EnvironmentalAlignment().to(DEVICE)

environmental_alignment.load_state_dict(
    alignment_state,
    strict=True
)

environmental_alignment.eval()

print("[PASSED] Environmental alignment loaded strictly.")
print("[PASSED] 128-D -> 32-D verified.")

# ------------------------------------------------------------------------------------------------
# 5. EXACT ENVIRONMENTAL EXPERT
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("4. LOADING VERIFIED ENVIRONMENTAL EXPERT")
print("=" * 100)

expert_state = torch.load(
    EXPERT_PATH,
    map_location=DEVICE,
    weights_only=False
)

class EnvironmentalExpert(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(32, 64),       # 0
            nn.BatchNorm1d(64),      # 1
            nn.ReLU(),              # 2
            nn.Dropout(),            # 3
            nn.Linear(64, 32),       # 4
            nn.ReLU(),              # 5
            nn.Dropout(),            # 6
            nn.Linear(32, 1),        # 7
        )

    def forward(self, x):
        return self.network(x)


environmental_expert = EnvironmentalExpert().to(DEVICE)

environmental_expert.load_state_dict(
    expert_state,
    strict=True
)

environmental_expert.eval()

print("[PASSED] Environmental expert loaded strictly.")

# ------------------------------------------------------------------------------------------------
# 6. PREPROCESSOR OUTPUT VALIDATION
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("5. TESTING SAVED PREPROCESSOR ON P001")
print("=" * 100)

test_path = os.path.join(
    DEMO_DIR,
    "P001",
    "environmental_input.csv"
)

test_df = pd.read_csv(test_path)

raw_test = test_df[feature_columns].copy()

print("Raw shape:", raw_test.shape)
print("P001 missing values:")
print(raw_test.isna().sum()[raw_test.isna().sum() > 0])

try:

    processed_test = preprocessor.transform(raw_test)

    if hasattr(processed_test, "toarray"):
        processed_test = processed_test.toarray()

    processed_test = np.asarray(processed_test, dtype=np.float32)

    print()
    print("[PASSED] Saved preprocessor successfully transformed P001.")
    print("Preprocessed shape:", processed_test.shape)

except Exception as e:

    print()
    print("[FAILED] Saved preprocessor itself cannot transform P001.")
    print("Exact error:")
    print(repr(e))

    raise

if processed_test.shape[1] != 31:

    raise RuntimeError(
        f"Expected 31 preprocessed features, got {processed_test.shape[1]}"
    )

print("[PASSED] Preprocessor output is exactly 31-D.")

# ------------------------------------------------------------------------------------------------
# 7. P001-P005 PRODUCTION INFERENCE
# ------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("6. ENVIRONMENTAL P001-P005 PRODUCTION INFERENCE")
print("=" * 100)

environmental_results = []

with torch.no_grad():

    for patient in PATIENTS:

        print()
        print("-" * 100)
        print(patient)
        print("-" * 100)

        path = os.path.join(
            DEMO_DIR,
            patient,
            "environmental_input.csv"
        )

        df = pd.read_csv(path)

        raw_x = df[feature_columns].copy()

        print("Raw input shape:", raw_x.shape)

        # IMPORTANT:
        # No manual mpc_yn conversion.
        # The saved preprocessor handles the raw feature exactly as trained.

        processed = preprocessor.transform(raw_x)

        if hasattr(processed, "toarray"):
            processed = processed.toarray()

        processed = np.asarray(
            processed,
            dtype=np.float32
        )

        print(
            "Preprocessed shape:",
            processed.shape
        )

        if processed.shape != (1, 31):
            raise RuntimeError(
                f"{patient}: unexpected preprocessed shape "
                f"{processed.shape}"
            )

        if not np.isfinite(processed).all():

            bad = np.argwhere(
                ~np.isfinite(processed)
            )

            raise RuntimeError(
                f"{patient}: saved preprocessor produced "
                f"non-finite values at {bad.tolist()}"
            )

        x = torch.tensor(
            processed,
            dtype=torch.float32,
            device=DEVICE
        )

        # 31 -> 128
        representation_128 = environmental_encoder(x)

        # 128 -> 32
        representation_32, _ = environmental_alignment(
            representation_128
        )

        # 32 -> 1
        logit = environmental_expert(
            representation_32
        )

        probability = torch.sigmoid(
            logit
        ).item()

        prediction = int(probability >= 0.5)

        prediction_label = (
            "ASD"
            if prediction == 1
            else "NON-ASD"
        )

        print(
            "31-D input            :",
            tuple(x.shape)
        )

        print(
            "128-D representation  :",
            tuple(representation_128.shape)
        )

        print(
            "32-D alignment        :",
            tuple(representation_32.shape)
        )

        print(
            "ASD probability       :",
            f"{probability:.6f}"
        )

        print(
            "Prediction             :",
            prediction_label
        )

        environmental_results.append({
            "patient_id": patient,
            "environmental_probability": probability,
            "environmental_prediction": prediction_label,
            "environmental_prediction_numeric": prediction
        })

# ------------------------------------------------------------------------------------------------
# 8. RESULTS TABLE
# ------------------------------------------------------------------------------------------------

environmental_results_df = pd.DataFrame(
    environmental_results
)

print()
print("=" * 100)
print("7. ENVIRONMENTAL RESULTS — P001-P005")
print("=" * 100)

print(
    environmental_results_df.to_string(
        index=False
    )
)

# Keep in notebook for fusion
ENVIRONMENTAL_RESULTS = environmental_results_df.copy()

print()
print("=" * 100)
print("M4.4.10.9.2 COMPLETED")
print("=" * 100)

print("[PASSED] Saved environmental preprocessor used.")
print("[PASSED] No manual mpc_yn encoding performed.")
print("[PASSED] P001-P005 environmental preprocessing completed.")
print("[PASSED] 31-D environmental representation validated.")
print("[PASSED] 128-D environmental representation produced.")
print("[PASSED] 32-D environmental alignment produced.")
print("[PASSED] Environmental expert inference completed.")
print("[PASSED] Environmental probabilities generated.")

print()
print("NEXT — DO NOT RECONSTRUCT ENVIRONMENTAL AGAIN:")
print("M4.4.11 — MULTIMODAL FUSION INSPECTION")
print("M4.4.12 — FUSION RECONSTRUCTION")
print("M4.4.13 — P001-P005 THREE-MODAL INFERENCE")
print("M4.4.14 — ANY-ONE / ANY-TWO / ALL-THREE INTERFACE")
print("=" * 100)

ASD MULTIMODAL PROJECT — M4.4.10.9.2
ENVIRONMENTAL PRODUCTION INFERENCE — SAVED PREPROCESSOR

1. LOADING VERIFIED ENVIRONMENTAL PREPROCESSOR
[FOUND] /content/gdrive/MyDrive/ASD_Multimodal_Project/Environmental_Model/02_preprocessing/environmental_preprocessor.joblib
Type: <class 'sklearn.pipeline.Pipeline'>
[PASSED] Saved preprocessing object is transform-capable.

Pipeline steps:
  imputer: <class 'sklearn.impute._base.SimpleImputer'>
  scaler: <class 'sklearn.preprocessing._data.StandardScaler'>

2. LOADING VERIFIED ENVIRONMENTAL ENCODER
[PASSED] Encoder checkpoint loaded.
Input dimension : 31
Embedding dim   : 128
Feature count   : 31
[PASSED] Environmental encoder loaded strictly.
[PASSED] Encoder state-dict exactly matches checkpoint.

3. LOADING VERIFIED ENVIRONMENTAL ALIGNMENT
[PASSED] Environmental alignment loaded strictly.
[PASSED] 128-D -> 32-D verified.

4. LOADING VERIFIED ENVIRONMENTAL EXPERT
[PASSED] Environmental expert loaded strictly.

5. TESTING SAVED PREPROCESSOR ON